# Joint Kinematics Dimensionality Reduction Analysis

This notebook analyzes walking kinematics through dimensionality reduction (PCA/UMAP) to understand how joints coordinate during free-walking behavior.

**Data source**: JARVIS 3D tracking + MuJoCo inverse kinematics

**Reference**: Analysis patterns from Grant's fly_walking_analysis and Pratt et al., 2024

## 1. Configuration

In [ ]:
# Cell 1 -- Imports

import sys
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from scipy import signal
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d
from scipy.stats import circmean, circstd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import cupy
import cuml


# Add 3d_tracking_dataset root to path for utils
_nb_dir = Path().resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
import utils.io_dict_to_hdf5 as ioh5

# Optional: UMAP
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not available. Install with: pip install umap-learn")

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 10

In [ ]:
# Cell 2 -- Configuration
 
# =============================================================================
# CONFIGURATION
# =============================================================================

FPS = 800  # Hz
SMOOTH_SIGMA = 6  # Gaussian σ (frames) for Hilbert phase computation

# Phase reference leg for coordination analysis
REFERENCE_LEG = 'T1_left'

# Bodies to use for heading/velocity computation
HEADING_BODIES = ['thorax']

# PCA / phase-binning parameters
N_PCA_COMPONENTS = 10
N_PHASE_BINS = 24

# Joint sets
JOINT_SETS = {
    #'core': ['coxa_abduct', 'coxa', 'femur', 'tibia'],
    'core': ['coxa_abduct', 'coxa', 'femur', 'tibia'],
    'main': ['coxa_abduct', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia', 'tarsus'],
    'full': ['coxa_abduct', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia',
             'tarsus', 'tarsus2', 'tarsus3', 'tarsus4', 'tarsus5'],
}

ACTIVE_JOINT_SET = 'full'

LEGS = ['T1_left', 'T1_right', 'T2_left', 'T2_right', 'T3_left', 'T3_right']
LEG_LABELS = {'T1_left': 'L1', 'T1_right': 'R1',
              'T2_left': 'L2', 'T2_right': 'R2',
              'T3_left': 'L3', 'T3_right': 'R3'}

JOINT_LABELS = {
    'coxa_abduct': 'Coxa Abd',
    'coxa_twist':   'Coxa Rot',
    'coxa':         'Coxa Flex',
    'femur_twist':  'Femur Rot',
    'femur':        'Femur Flex',
    'tibia':        'Tibia Flex',
    'tarsus':       'Tarsus Flex',
}

# Combined bout-dict HDF5 — UPDATE PATH AS NEEDED
#H5_PATH = Path('/home/user/3D_tracking_paper/IK_outputs/walking_videos/free_walking/Data_analysis/analysis/v1/ik_output_combined_v1_free_walking.h5') just 1_7 flies
H5_PATH = Path('/home/user/3D_tracking_paper/IK_outputs/walking_videos/free_walking/Data_analysis/analysis/v1/ik_output_combined_v1_free_walking.h5') # all flies

OUTPUT_DIR = Path('/home/user/3D_tracking_paper/output/free_walking/joint_kinematics')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Frame rate: {FPS} Hz | Reference leg: {REFERENCE_LEG} | Smooth σ: {SMOOTH_SIGMA} frames")
print(f"Active joint set: '{ACTIVE_JOINT_SET}' "
      f"({len(JOINT_SETS[ACTIVE_JOINT_SET])} joints/leg = "
      f"{len(JOINT_SETS[ACTIVE_JOINT_SET]) * len(LEGS)} total)")
print(f"H5 path: {H5_PATH}")
print(f"Output: {OUTPUT_DIR.absolute()}")

## 2. Data Loading Functions

In [ ]:
# Cell 3 -- Data Loading Functions
def load_ik_bouts(h5_path):
    """Load bout-dict HDF5 via io_dict_to_hdf5.load().

    Returns:
        bout_dict   : full nested dict (info + bout_000 ... bout_N)
        bout_keys   : sorted list of 'bout_NNN' keys
        fly_ids     : array (N_bouts,) of fly identifiers
        clip_lengths: array (N_bouts,) of frames per bout
    """
    d = ioh5.load(h5_path)
    bout_keys = sorted(k for k in d if k.startswith('bout_'))
    fly_ids = np.array(d['info']['fly_ids'])
    clip_lengths = np.array(d['info']['clip_lengths'])
    print(f"Loaded {len(bout_keys)} bouts | {len(np.unique(fly_ids))} unique flies")
    print(f"Frames/bout: min={clip_lengths.min()}, max={clip_lengths.max()}, "
          f"total={clip_lengths.sum():,}")
    return d, bout_keys, fly_ids, clip_lengths


# Mapping from long leg names to egocentric-site short codes
# e.g. T1_left -> T1L,  T2_right -> T2R
_LEG_SHORT = {
    'T1_left':  'T1L', 'T1_right': 'T1R',
    'T2_left':  'T2L', 'T2_right': 'T2R',
    'T3_left':  'T3L', 'T3_right': 'T3R',
}


def find_leg_tip_site_indices(site_names, legs):
    """Find site indices for each leg's distal tip in xpos_egocentric.

    Tries naming conventions in order:
      1. Short-code tip  e.g. 'T1L_TaTip'  (matches 'tracking[T1L_TaTip]_fly')
      2. Long-name tip   e.g. 'T1_left_TaTip'
      3. claw prefix     e.g. 'claw_T1_left'
      4. claw suffix     e.g. 'T1_left_claw'

    Returns dict: leg_name -> site_index (None if not found).
    """
    indices = {}
    for leg in legs:
        short = _LEG_SHORT.get(leg, leg)
        patterns = [
            f"{short}_TaTip",    # tracking[T1L_TaTip]_fly  ← actual format
            f"{leg}_TaTip",      # T1_left_TaTip
            f"claw_{leg}",       # claw_T1_left
            f"{leg}_claw",       # T1_left_claw
        ]
        match = None
        for pattern in patterns:
            match = next((i for i, n in enumerate(site_names) if pattern in n), None)
            if match is not None:
                break
        indices[leg] = match
        if match is None:
            print(f"Warning: no tip site for {leg} (tried: {patterns})")
    return indices


def bouts_to_dataframe(bout_dict, bout_keys, fly_ids, joint_list,
                       names_xpos, heading_bodies,
                       egocentric_site_names, leg_tip_site_indices):
    """Concatenate all bouts into one flat DataFrame.

    Each row = one frame.
    Columns: frame (within-bout), bout_idx, bout_id, fly_id,
             <joint angles>, <body xyz>, <leg_tip_z_ego per leg>.
    """
    frames = []
    for bout_idx, (key, fly_id) in enumerate(zip(bout_keys, fly_ids)):
        b = bout_dict[key]
        qpos = np.array(b['qpos'])
        xpos = np.array(b['xpos'])
        T = len(qpos)

        row = {
            'frame':    np.arange(T),
            'bout_idx': np.full(T, bout_idx, dtype=np.int32),
            'bout_id':  np.full(T, key, dtype=object),
            'fly_id':   np.full(T, fly_id, dtype=object),
        }

        for leg, joint, idx in joint_list:
            row[f"{leg}_{joint}"] = qpos[:, idx]

        for body in heading_bodies:
            if body in names_xpos:
                bidx = names_xpos.index(body)
                row[f"{body}_x"] = xpos[:, bidx, 0]
                row[f"{body}_y"] = xpos[:, bidx, 1]
                row[f"{body}_z"] = xpos[:, bidx, 2]

        if 'xpos_egocentric' in b:
            xpos_ego = np.array(b['xpos_egocentric'])
            for leg, site_idx in leg_tip_site_indices.items():
                if site_idx is not None and site_idx < xpos_ego.shape[1]:
                    row[f"{leg}_tip_x_ego"] = xpos_ego[:, site_idx, 0]
                    row[f"{leg}_tip_y_ego"] = xpos_ego[:, site_idx, 1]
                    row[f"{leg}_tip_z_ego"] = xpos_ego[:, site_idx, 2]


        # World-frame leg tip positions (from MuJoCo xpos)
        _CLAW_NAMES = {
            'T1_left': 'claw_T1_left', 'T1_right': 'claw_T1_right',
            'T2_left': 'claw_T2_left', 'T2_right': 'claw_T2_right',
            'T3_left': 'claw_T3_left', 'T3_right': 'claw_T3_right',
        }
        _names_xpos_list = list(names_xpos)
        for leg, claw_name in _CLAW_NAMES.items():
            if claw_name in _names_xpos_list:
                _cidx = _names_xpos_list.index(claw_name)
                row[f"{leg}_tip_x_world"] = xpos[:, _cidx, 0]
                row[f"{leg}_tip_y_world"] = xpos[:, _cidx, 1]
                row[f"{leg}_tip_z_world"] = xpos[:, _cidx, 2]

        frames.append(pd.DataFrame(row))

    df = pd.concat(frames, ignore_index=True)
    print(f"Built DataFrame: {len(df):,} frames, {len(df.columns)} columns")
    return df


def build_joint_index(names_qpos, joint_set, legs):
    """Build index mapping from joint set to qpos columns.

    Returns:
        joint_index: dict full_name -> index in qpos
        joint_list : list of (leg, joint, index) tuples
    """
    joint_index = {}
    joint_list = []
    for leg in legs:
        for joint in joint_set:
            full_name = f"{joint}_{leg}"
            if full_name in names_qpos:
                idx = names_qpos.index(full_name)
                joint_index[full_name] = idx
                joint_list.append((leg, joint, idx))
            else:
                print(f"Warning: Joint '{full_name}' not found in qpos names")
    print(f"Built index for {len(joint_list)} joints")
    return joint_index, joint_list


## 3. Preprocessing Functions

In [ ]:
# Cell 4 -- Preprocessing Functions
def compute_derivatives(df, joint_columns, fps=800):
    """
    Compute first and second derivatives of joint angles using Savitzky-Golay filter.
    
    Args:
        df: DataFrame with joint angle columns
        joint_columns: list of column names to differentiate
        fps: frame rate
    
    Returns:
        DataFrame with added _d1 (velocity) and _d2 (acceleration) columns
    """
    dt = 1.0 / fps
    s = 1.0 / dt  # Scale for first derivative
    s2 = 1.0 / (dt * dt)  # Scale for second derivative
    
    df = df.copy()
    
    for col in joint_columns:
        arr = df[col].values
        
        # Skip if too short
        if len(arr) < 7:
            df[f"{col}_d1"] = np.nan
            df[f"{col}_d2"] = np.nan
            continue
        
        # Savitzky-Golay filter: window=5, polyorder=2
        # This matches the reference analysis
        df[f"{col}_d1"] = signal.savgol_filter(arr, 5, 2, deriv=1) * s
        df[f"{col}_d2"] = signal.savgol_filter(arr, 5, 2, deriv=2) * s2
    
    return df



def compute_phases_from_swing(df, legs):
    """Step phase by linear interpolation between swing onsets.

    Uses {leg}_swing_stance (from compute_swing_stance_tip_speed).
    Within each step cycle phase advances linearly from -pi at liftoff
    to +pi just before the next liftoff.  Frames outside the first/last
    onset in a bout are NaN.

    Convention [-pi, pi]:
      -pi  swing onset (liftoff)
        0  mid-stride
      +pi  just before next liftoff

    Requires compute_swing_stance_tip_speed to have run first.
    """
    df = df.copy()
    for leg in legs:
        ss_col = f"{leg}_swing_stance"
        if ss_col not in df.columns:
            continue
        phase_arr = np.full(len(df), np.nan)
        for _bid, grp in df.groupby('bout_id', sort=False):
            idx    = grp.index
            onsets = liftoff_from_hilbert(grp[phase_col].values)
            if len(onsets) < 2:
                continue
            phase = np.full(len(grp), np.nan)
            for i in range(len(onsets) - 1):
                t0, t1 = onsets[i], onsets[i + 1]
                phase[t0:t1] = np.linspace(-np.pi, np.pi, t1 - t0, endpoint=False)
            phase_arr[idx] = phase
        df[f"{leg}_phase"] = phase_arr
    return df

def _gaussian_smooth(arr, sigma=4):
    """1-D Gaussian smooth; NaN-safe via weight normalisation."""
    from scipy.ndimage import gaussian_filter1d
    mask = np.isnan(arr)
    out  = gaussian_filter1d(np.where(mask, 0., arr), sigma)
    cnt  = gaussian_filter1d((~mask).astype(float), sigma)
    out[cnt > 0] /= cnt[cnt > 0]
    out[mask]     = np.nan
    return out


def compute_swing_stance_tip_speed(df, legs, fps=800, thresh_frac=0.20):
    """Swing/stance from 3-D world-frame tip speed.

    Speed is ~0 when the foot is stationary on the ground (stance) and
    rises sharply at liftoff (swing onset).
      speed > thresh_frac * per-bout-max -> SWING (0)
      speed <= thresh                    -> STANCE (1)
    Raise thresh_frac if short stance periods are mis-classified as swing.
    """
    df = df.copy()
    for leg in legs:
        xcol = f"{leg}_tip_x_world"
        ycol = f"{leg}_tip_y_world"
        zcol = f"{leg}_tip_z_world"
        if zcol not in df.columns:
            continue
        ss = np.full(len(df), np.nan)
        for _bid, grp in df.groupby('bout_id', sort=False):
            idx = grp.index
            n   = len(grp)
            if n < 5:
                continue
            def _vel(col):
                v = grp[col].values.astype(float) if col in grp.columns \
                    else np.zeros(n)
                return np.gradient(v) * fps * 10  # model-m/frame -> mm/s
            spd = np.sqrt(_vel(xcol)**2 + _vel(ycol)**2 + _vel(zcol)**2)
            spd = _gaussian_smooth(spd, sigma=4)
            thr = thresh_frac * np.nanmax(spd)
            ss[idx] = (spd <= thr).astype(float)
        valid = ~np.isnan(ss)
        df.loc[valid, f"{leg}_swing_stance"] = ss[valid].astype(int)
    return df


def compute_n_legs_stance(df, legs):
    """
    Compute number of legs in stance phase per frame.
    """
    df = df.copy()
    stance_cols = [f"{leg}_swing_stance" for leg in legs if f"{leg}_swing_stance" in df.columns]
    if stance_cols:
        df['n_legs_stance'] = df[stance_cols].sum(axis=1)
    return df

# Model scale: MuJoCo SI positions in meters at ~100× real scale.
# forward_speed = sqrt(dx²+dy²)*fps*10 gives mm/s
# (model-m/s cancels with ÷100 scale factor, then ×10 for cm→mm).
MODEL_SCALE = 100.0   # kept for reference; not used for speed conversion

def compute_heading_and_velocity(df, fps=800, body='thorax', smooth_sigma=2):
    """Compute heading, forward speed, and turning rate from body XY position.

    Positions are in model meters (SI, ~100× real scale). Due to the scale
    factor, sqrt(dx²+dy²)*fps directly gives speed in cm/s.

    Processes each bout independently to avoid cross-bout discontinuities.
    Adds columns: heading (rad), forward_speed (mm/s), turning_rate (rad/s).
    """
    df = df.copy()
    x_col, y_col = f"{body}_x", f"{body}_y"
    if x_col not in df.columns or y_col not in df.columns:
        print(f"Warning: position columns not found for {body}")
        return df

    heading_arr   = np.full(len(df), np.nan)
    speed_arr     = np.full(len(df), np.nan)
    turn_rate_arr = np.full(len(df), np.nan)

    for bout_id, grp in df.groupby('bout_id'):
        idx = grp.index
        x = gaussian_filter1d(grp[x_col].values.astype(float), sigma=smooth_sigma)
        y = gaussian_filter1d(grp[y_col].values.astype(float), sigma=smooth_sigma)
        dx = np.gradient(x) * fps   # m/s
        dy = np.gradient(y) * fps   # m/s
        heading  = np.arctan2(dy, dx)
        heading_u = np.unwrap(heading)
        heading_s = gaussian_filter1d(heading_u, sigma=smooth_sigma)
        heading_s = (heading_s + np.pi) % (2 * np.pi) - np.pi
        speed     = np.sqrt(dx**2 + dy**2) * 10       # cm/s × 10 = mm/s
        turn_rate = np.gradient(heading_s) * fps
        heading_arr[idx]   = heading_s
        speed_arr[idx]     = speed
        turn_rate_arr[idx] = turn_rate

    df['heading']       = heading_arr
    df['forward_speed'] = speed_arr   # mm/s
    df['turning_rate']  = turn_rate_arr
    return df


def compute_relative_phases(df, legs, reference_leg='T1_left'):
    """Compute Hilbert phase of each leg relative to the reference leg.

    Uses {leg}_phase columns (from compute_phases).
    Adds {leg}_phase_rel columns (wrapped to [-pi, pi]) for each non-reference leg.
    """
    df = df.copy()
    ref_col = f"{reference_leg}_phase"
    if ref_col not in df.columns:
        print(f"Warning: reference phase column {ref_col} not found")
        return df
    for leg in legs:
        if leg == reference_leg:
            continue
        leg_col = f"{leg}_phase"
        if leg_col not in df.columns:
            continue
        diff = df[leg_col] - df[ref_col]
        df[f"{leg}_phase_rel"] = np.arctan2(np.sin(diff), np.cos(diff))
    return df


def swing_onsets_from_stance(series):
    """Return frame indices where swing_stance transitions 1->0 (stance->swing)."""
    ss = series.values.astype(int)
    return np.where(np.diff(ss, prepend=ss[0]) == -1)[0]


def compute_step_cycle_speed(df, fps=800, ref_leg='T1_left'):
    """Assign step_cycle_id and step_cycle_mean_speed per frame.

    Step cycles are defined by swing onsets of ref_leg: transitions of
    {ref_leg}_swing_stance from 1 (stance) to 0 (swing). Each step cycle
    runs from one liftoff to the next. Frames before the first or after the
    last swing onset within a bout are left as NaN (partial cycles).

    Adds columns:
      step_cycle_id         int   -- cycle index within each bout (0-based)
      step_cycle_mean_speed float -- mean forward_speed over the cycle (mm/s)
    """
    df = df.copy()
    phase_col = f"{ref_leg}_phase"
    spd_col = 'forward_speed'

    sc_id  = np.full(len(df), np.nan)
    sc_spd = np.full(len(df), np.nan)

    if phase_col not in df.columns:
        print(f"Warning: {phase_col} not found -- run compute_hilbert_phases first.")
        df['step_cycle_id']         = sc_id
        df['step_cycle_mean_speed'] = sc_spd
        return df

    for bout_id, grp in df.groupby('bout_id', sort=False):
        idx    = grp.index
        onsets = liftoff_from_hilbert(grp[phase_col].values)
        if len(onsets) < 2:
            continue
        spd = grp[spd_col].values.astype(float) if spd_col in grp.columns \
              else np.full(len(grp), np.nan)

        for k in range(len(onsets) - 1):
            t0, t1 = onsets[k], onsets[k + 1]
            sc_id[idx[t0:t1]]  = k
            sc_spd[idx[t0:t1]] = np.nanmean(spd[t0:t1])

    df['step_cycle_id']         = sc_id
    df['step_cycle_mean_speed'] = sc_spd
    return df


In [ ]:
# Cell 5 -- Hilbert Phase Functions
# Replaces compute_phases_from_swing (linear ramp) with per-frame Hilbert phase.
# Validated in notebooks/Hilbert_Phase_Tester.ipynb.
#
# Convention:  phase = 0   → mid-swing (peak 3D tip speed)
#              phase = ±π  → mid-stance (speed trough)
# Speed is mean-centered before Hilbert so phase spans the full [-π, π].

def compute_hilbert_phase_bout(positions_xyz, fps, smooth_sigma,
                               mask_stance=False, stance_speed_frac=0.20):
    """Per-frame Hilbert phase from 3D tip speed for one bout.

    Returns
    -------
    phase : (T,) float  — Hilbert phase in [-π, π]; NaN at first/last 2 frames
    speed : (T,) float  — smoothed 3D speed (mm/s), NOT mean-subtracted
    """
    T = len(positions_xyz)
    if T < 10:
        return np.full(T, np.nan), np.zeros(T)
    xyz = positions_xyz.astype(float).copy()
    for dim in range(3):
        col = xyz[:, dim]
        nans = np.isnan(col)
        if nans.any() and (~nans).sum() > 1:
            col[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), col[~nans])
        xyz[:, dim] = col
    vel = np.gradient(xyz, axis=0) * fps
    spd = gaussian_filter1d(np.linalg.norm(vel, axis=1), sigma=smooth_sigma)
    ph  = np.angle(hilbert(spd - np.nanmean(spd)))
    ph[:2] = np.nan
    ph[-2:] = np.nan
    if mask_stance:
        ph[spd <= stance_speed_frac * np.nanmax(spd)] = np.nan
    return ph, spd


def swing_onsets_from_hilbert(phase):
    """Step-cycle boundary indices from 2π increments in unwrapped Hilbert phase."""
    finite_mask = np.isfinite(phase)
    if finite_mask.sum() < 4:
        return np.array([], dtype=int)
    finite_idx = np.where(finite_mask)[0]
    unwrapped  = np.unwrap(phase[finite_mask])
    jumps = np.where(np.diff(np.floor(unwrapped / (2 * np.pi))) > 0)[0]
    return finite_idx[jumps]


def compute_hilbert_phases(df, legs, fps, smooth_sigma):
    """Compute per-frame Hilbert phase for each leg and add {leg}_phase columns.

    Uses world-frame tip XYZ columns ({leg}_tip_x/y/z_world) already in df.
    Processes each bout independently to avoid cross-bout edge effects.
    Output column names are identical to the old compute_phases_from_swing,
    so all downstream code (compute_relative_phases, UMAP, plots) is unaffected.
    """
    df = df.copy()
    for leg in legs:
        df[f'{leg}_phase'] = np.nan
        x_col = f'{leg}_tip_x_world'
        y_col = f'{leg}_tip_y_world'
        z_col = f'{leg}_tip_z_world'
        if z_col not in df.columns:
            print(f'  Warning: {z_col} not found — skipping {leg}')
            continue
        for bid, grp in df.groupby('bout_id', sort=False):
            xyz = grp[[x_col, y_col, z_col]].values
            ph, _ = compute_hilbert_phase_bout(xyz, fps, smooth_sigma, mask_stance=False)
            df.loc[grp.index, f'{leg}_phase'] = ph
    return df



def liftoff_from_hilbert(phase):
    """Upward crossing of phase = -π/2 → liftoff (swing onset).

    Each 2π increment of (unwrapped_phase + π/2) marks one liftoff.
    Returns frame indices relative to the input array.
    """
    finite_mask = np.isfinite(phase)
    if finite_mask.sum() < 4:
        return np.array([], dtype=int)
    finite_idx = np.where(finite_mask)[0]
    unwrapped  = np.unwrap(phase[finite_mask])
    increments = np.diff(np.floor((unwrapped + np.pi / 2) / (2 * np.pi)))
    return finite_idx[np.where(increments > 0)[0]]


def landing_from_hilbert(phase):
    """Upward crossing of phase = +π/2 → landing (stance onset).

    Each 2π increment of (unwrapped_phase - π/2) marks one landing.
    Returns frame indices relative to the input array.
    """
    finite_mask = np.isfinite(phase)
    if finite_mask.sum() < 4:
        return np.array([], dtype=int)
    finite_idx = np.where(finite_mask)[0]
    unwrapped  = np.unwrap(phase[finite_mask])
    increments = np.diff(np.floor((unwrapped - np.pi / 2) / (2 * np.pi)))
    return finite_idx[np.where(increments > 0)[0]]


def compute_swing_stance_from_hilbert(df, legs):
    """Derive binary swing/stance from Hilbert phase.

    swing  (0): |phase| < π/2  — near mid-swing (peak speed)
    stance (1): |phase| >= π/2 — near mid-stance (speed trough)

    Replaces compute_swing_stance_tip_speed. Must be called after
    compute_hilbert_phases. Output column names are identical so all
    downstream code (compute_n_legs_stance, swing_onsets_from_stance,
    gait diagrams) is unaffected.
    """
    df = df.copy()
    for leg in legs:
        phase_col = f'{leg}_phase'
        if phase_col not in df.columns:
            continue
        ph = df[phase_col].values
        ss = np.where(np.isfinite(ph), (np.abs(ph) >= np.pi / 2).astype(float), np.nan)
        df[f'{leg}_swing_stance'] = ss
    return df


print('Hilbert phase functions defined.')

## 4. Dimensionality Reduction Functions

In [ ]:
# Cell 6 -- Feature Matrix Function
def get_feature_matrix(df, joint_list, include_derivatives=True):
    """
    Build feature matrix for dimensionality reduction.
    
    Args:
        df: DataFrame with joint angles (and derivatives if include_derivatives=True)
        joint_list: list of (leg, joint, idx) tuples
        include_derivatives: whether to include _d1 columns
    
    Returns:
        X: (N, features) array
        feature_names: list of feature names
    """
    # Build list of columns to include
    feature_cols = []
    for leg, joint, _ in joint_list:
        col = f"{leg}_{joint}"
        if col in df.columns:
            feature_cols.append(col)
            if include_derivatives and f"{col}_d1" in df.columns:
                feature_cols.append(f"{col}_d1")
    
    # Extract and handle NaNs
    X = df[feature_cols].values
    
    # Remove rows with NaNs
    valid_mask = ~np.any(np.isnan(X), axis=1)
    X = X[valid_mask]
    
    print(f"Feature matrix: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"  Removed {(~valid_mask).sum()} samples with NaNs")
    
    return X, feature_cols, valid_mask


def run_pca(X, n_components=10, standardize=True):
    """
    Run PCA on feature matrix.
    
    Args:
        X: (N, features) array
        n_components: number of PCs to compute
        standardize: whether to standardize features first
    
    Returns:
        pca_result: (N, n_components) transformed data
        pca: fitted PCA object
        scaler: fitted StandardScaler (or None)
        loadings: DataFrame of loadings
    """
    scaler = None
    if standardize:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(X)
    
    print(f"PCA: {n_components} components")
    print(f"  Explained variance: {pca.explained_variance_ratio_[:5].round(3)}...")
    print(f"  Total explained: {pca.explained_variance_ratio_.sum():.1%}")
    
    return pca_result, pca, scaler


def run_umap(X, n_neighbors=15, min_dist=0.1, n_components=2, standardize=True):
    """
    Run UMAP on feature matrix.
    """
    if not HAS_UMAP:
        print("UMAP not available")
        return None, None, None
    
    scaler = None
    if standardize:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=n_components)
    umap_result = reducer.fit_transform(X)
    
    print(f"UMAP: {n_components} components (n_neighbors={n_neighbors}, min_dist={min_dist})")
    
    return umap_result, reducer, scaler

## 5. Visualization Functions

In [ ]:
# Cell 7 -- Visualization Functions

def plot_pca_variance(pca, ax=None):
    """
    Plot explained variance ratio (scree plot).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    
    n_pcs = len(pca.explained_variance_ratio_)
    x = np.arange(1, n_pcs + 1)
    
    # Bar plot for individual variance
    ax.bar(x, pca.explained_variance_ratio_, alpha=0.7, label='Individual')
    
    # Line plot for cumulative variance
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(x, cumvar, 'ro-', label='Cumulative')
    
    ax.set_xlabel('Principal Component')
    ax.set_ylabel('Explained Variance Ratio')
    ax.set_title('PCA Explained Variance')
    ax.legend()
    ax.set_xticks(x)
    ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90%')
    
    return ax


def plot_pca_loadings(pca, feature_names, joint_list, n_pcs=4):
    """
    Plot PCA loadings as heatmap organized by leg and joint.
    """
    loadings = pca.components_[:n_pcs].T
    
    fig, ax = plt.subplots(figsize=(10, max(8, len(feature_names) * 0.3)))
    
    # Create labeled feature names for display
    display_names = []
    for name in feature_names:
        # Parse leg_joint or leg_joint_d1
        parts = name.split('_')
        if parts[-1] == 'd1':
            leg = f"{parts[0]}_{parts[1]}"
            joint = '_'.join(parts[2:-1])
            suffix = ' (vel)'
        else:
            leg = f"{parts[0]}_{parts[1]}"
            joint = '_'.join(parts[2:])
            suffix = ''
        
        leg_label = LEG_LABELS.get(leg, leg)
        joint_label = JOINT_LABELS.get(joint, joint)
        display_names.append(f"{leg_label} {joint_label}{suffix}")
    
    im = ax.imshow(loadings, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
    
    ax.set_yticks(np.arange(len(display_names)))
    ax.set_yticklabels(display_names, fontsize=8)
    ax.set_xticks(np.arange(n_pcs))
    ax.set_xticklabels([f'PC{i+1}' for i in range(n_pcs)])
    ax.set_xlabel('Principal Component')
    ax.set_title('PCA Loadings')
    
    plt.colorbar(im, ax=ax, label='Loading')
    plt.tight_layout()
    
    return fig, ax


def plot_pca_trajectory(pca_result, df_valid, color_by='frame', ax=None, s=5, alpha=0.5):
    """
    Plot PC1 vs PC2 trajectory colored by specified variable.
    
    Args:
        pca_result: (N, n_pcs) array
        df_valid: DataFrame aligned with pca_result (after removing NaNs)
        color_by: column name to use for coloring, or 'frame' for time
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    if color_by == 'frame':
        c = np.arange(len(pca_result))
        cmap = 'viridis'
        label = 'Frame'
    elif color_by in df_valid.columns:
        c = df_valid[color_by].values
        cmap = 'coolwarm' if 'phase' in color_by else 'viridis'
        label = color_by
    else:
        c = 'blue'
        cmap = None
        label = None
    
    scatter = ax.scatter(pca_result[:, 0], pca_result[:, 1], c=c, cmap=cmap, s=s, alpha=alpha)
    
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'PCA Trajectory (colored by {color_by})')
    
    if cmap is not None:
        plt.colorbar(scatter, ax=ax, label=label)
    
    return ax


def plot_joint_vs_phase(df, leg, joint, phase_col=None, n_bins=24, ax=None):
    """
    Plot joint angle vs step phase with binned statistics.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    col = f"{leg}_{joint}"
    if phase_col is None:
        phase_col = f"{leg}_phase"
    
    if col not in df.columns or phase_col not in df.columns:
        print(f"Columns not found: {col}, {phase_col}")
        return ax
    
    # Get valid data
    valid = df[[col, phase_col]].dropna()
    angles = valid[col].values
    phases = valid[phase_col].values
    
    # Bin by phase
    phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
    bin_indices = np.digitize(phases, phase_bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    
    # Compute mean and std per bin
    bin_means = np.array([angles[bin_indices == i].mean() if np.any(bin_indices == i) else np.nan 
                         for i in range(n_bins)])
    bin_stds = np.array([angles[bin_indices == i].std() if np.any(bin_indices == i) else np.nan 
                        for i in range(n_bins)])
    
    # Plot
    ax.scatter(phases, angles, alpha=0.1, s=1, c='gray')
    ax.plot(bin_centers, bin_means, 'b-', lw=2, label='Mean')
    ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.3)
    
    ax.set_xlabel('Step Phase (rad)')
    ax.set_ylabel(f'{JOINT_LABELS.get(joint, joint)} (deg)')
    ax.set_title(f'{LEG_LABELS.get(leg, leg)} {JOINT_LABELS.get(joint, joint)} vs Phase')
    ax.set_xlim(-np.pi, np.pi)
    ax.axvline(0, color='r', linestyle='--', alpha=0.5, label='Swing/Stance')
    ax.legend()
    
    return ax


def plot_leg_coordination(df, legs):
    """
    Plot phase relationships between legs.
    """
    phase_cols = [f"{leg}_phase" for leg in legs if f"{leg}_phase" in df.columns]
    n_legs = len(phase_cols)
    
    fig, axes = plt.subplots(n_legs, n_legs, figsize=(12, 12))
    
    for i, col1 in enumerate(phase_cols):
        for j, col2 in enumerate(phase_cols):
            ax = axes[i, j]
            
            if i == j:
                # Diagonal: histogram of phase
                valid = df[col1].dropna()
                ax.hist(valid, bins=30, density=True, alpha=0.7)
                ax.set_xlim(-np.pi, np.pi)
            else:
                # Off-diagonal: phase vs phase scatter
                valid = df[[col1, col2]].dropna()
                ax.scatter(valid[col2], valid[col1], alpha=0.1, s=1)
                ax.set_xlim(-np.pi, np.pi)
                ax.set_ylim(-np.pi, np.pi)
            
            if i == n_legs - 1:
                leg_label = LEG_LABELS.get(phase_cols[j].replace('_phase', ''), phase_cols[j])
                ax.set_xlabel(leg_label)
            if j == 0:
                leg_label = LEG_LABELS.get(phase_cols[i].replace('_phase', ''), phase_cols[i])
                ax.set_ylabel(leg_label)
    
    plt.suptitle('Inter-leg Phase Relationships')
    plt.tight_layout()
    
    return fig, axes

def plot_embedding(coords, df_valid, color_by='frame', title=None, ax=None,
                   s=5, alpha=0.5):
    """Plot 2-D embedding (PCA or UMAP) with flexible coloring.

    color_by options: 'frame', 'fly_id', 'forward_speed', 'heading',
                      'turning_rate', 'n_legs_stance', '{leg}_phase', '{leg}_phase_rel'
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))

    _circular = {'heading'} | \
        {f"{leg}_phase" for leg in LEGS} | \
        {f"{leg}_phase_rel" for leg in LEGS}

    if color_by in ('fly_id', 'n_legs_stance'):
        vals = df_valid[color_by].values
        unique_vals = sorted(set(vals))
        palette = plt.cm.get_cmap('tab10', max(len(unique_vals), 1))
        color_map = {v: palette(i) for i, v in enumerate(unique_vals)}
        colors = np.array([color_map[v] for v in vals])
        ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=s, alpha=alpha)
        handles = [plt.Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=color_map[v], label=str(v), markersize=8)
                   for v in unique_vals]
        ax.legend(handles=handles, fontsize=8, title=color_by)
    elif color_by in _circular and color_by in df_valid.columns:
        sc = ax.scatter(coords[:, 0], coords[:, 1],
                        c=df_valid[color_by].values, cmap='hsv',
                        vmin=-np.pi, vmax=np.pi, s=s, alpha=alpha)
        plt.colorbar(sc, ax=ax, label=color_by)
    elif color_by == 'turning_rate' and color_by in df_valid.columns:
        c = df_valid[color_by].values
        p95 = np.nanpercentile(np.abs(c), 95)
        sc  = ax.scatter(coords[:, 0], coords[:, 1], c=c, cmap='RdBu_r',
                         vmin=-p95, vmax=p95, s=s, alpha=alpha)
        plt.colorbar(sc, ax=ax, label='Turning rate (rad/s)')
    elif color_by == 'frame':
        sc = ax.scatter(coords[:, 0], coords[:, 1],
                        c=np.arange(len(coords)), cmap='viridis', s=s, alpha=alpha)
        plt.colorbar(sc, ax=ax, label='Frame')
    elif color_by in df_valid.columns:
        sc = ax.scatter(coords[:, 0], coords[:, 1],
                        c=df_valid[color_by].values, cmap='viridis', s=s, alpha=alpha)
        plt.colorbar(sc, ax=ax, label=color_by)
    else:
        ax.scatter(coords[:, 0], coords[:, 1], c='steelblue', s=s, alpha=alpha)

    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')
    ax.set_title(title or f'Embedding — {color_by}')
    return ax


def plot_phase_averaged_pca(df_valid, pca_result, reference_leg='T1_left',
                             n_bins=24, ax=None):
    """Phase-averaged PCA trajectory (gait cycle mapped to PC space).

    Bins all frames by reference_leg Hilbert phase into n_bins.
    One light line per fly; thick coloured line = grand mean.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))

    phase_col = f"{reference_leg}_phase"
    if phase_col not in df_valid.columns:
        print(f"Phase column {phase_col} not found")
        return ax

    bin_edges   = np.linspace(-np.pi, np.pi, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    cmap = plt.cm.hsv

    all_m1, all_m2 = [], []

    for fly_id, grp in df_valid.groupby('fly_id'):
        phases = grp[phase_col].values
        pc1    = pca_result[grp.index, 0]
        pc2    = pca_result[grp.index, 1]
        bins   = np.clip(np.digitize(phases, bin_edges) - 1, 0, n_bins - 1)
        m1 = np.array([pc1[bins == k].mean() if np.any(bins == k) else np.nan
                        for k in range(n_bins)])
        m2 = np.array([pc2[bins == k].mean() if np.any(bins == k) else np.nan
                        for k in range(n_bins)])
        m1c = np.append(m1, m1[0]);  m2c = np.append(m2, m2[0])
        bc  = np.append(bin_centers, bin_centers[0])
        for k in range(n_bins):
            if not (np.isnan(m1c[k]) or np.isnan(m1c[k + 1])):
                col = cmap((bc[k] + np.pi) / (2 * np.pi))
                ax.plot([m1c[k], m1c[k + 1]], [m2c[k], m2c[k + 1]],
                        color=col, lw=1.5, alpha=0.5)
        all_m1.append(m1);  all_m2.append(m2)

    if all_m1:
        gm1c = np.append(np.nanmean(all_m1, axis=0), np.nanmean(all_m1, axis=0)[0])
        gm2c = np.append(np.nanmean(all_m2, axis=0), np.nanmean(all_m2, axis=0)[0])
        bc   = np.append(bin_centers, bin_centers[0])
        for k in range(n_bins):
            if not (np.isnan(gm1c[k]) or np.isnan(gm1c[k + 1])):
                col = cmap((bc[k] + np.pi) / (2 * np.pi))
                ax.plot([gm1c[k], gm1c[k + 1]], [gm2c[k], gm2c[k + 1]],
                        color=col, lw=3.5, alpha=0.95)

    ax.set_xlabel('PC1');  ax.set_ylabel('PC2')
    ax.set_title(f'Phase-averaged PCA trajectory\n'
                 f'(binned by {reference_leg} phase, {n_bins} bins)')
    sm = plt.cm.ScalarMappable(cmap='hsv', norm=plt.Normalize(-np.pi, np.pi))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Phase (rad)')
    return ax


## 6. Coordination Metrics


In [ ]:
# Cell 8 -- Coordination Functions


def compute_tripod_coordination_strength(df, legs):
    """Compute Tripod Coordination Strength (TCS).

    Tripod gait: alternating tripods
    - Left tripod: L1, R2, L3 (T1_left, T2_right, T3_left)
    - Right tripod: R1, L2, R3 (T1_right, T2_left, T3_right)

    TCS = overlap duration of tripod swing / total tripod duration
    """
    df = df.copy()

    left_tripod  = ['T1_left',  'T2_right', 'T3_left']
    right_tripod = ['T1_right', 'T2_left',  'T3_right']

    ss_cols_left  = [f"{leg}_swing_stance" for leg in left_tripod]
    ss_cols_right = [f"{leg}_swing_stance" for leg in right_tripod]

    if not all(col in df.columns for col in ss_cols_left + ss_cols_right):
        print("Warning: swing_stance columns missing")
        return df, {}

    left_all_swing  = (df[ss_cols_left]  == 0).all(axis=1)
    left_any_swing  = (df[ss_cols_left]  == 0).any(axis=1)
    right_all_swing = (df[ss_cols_right] == 0).all(axis=1)
    right_any_swing = (df[ss_cols_right] == 0).any(axis=1)

    df['left_tripod_sync']  = left_all_swing.astype(int)
    df['right_tripod_sync'] = right_all_swing.astype(int)

    left_tcs  = (left_all_swing.sum()  / left_any_swing.sum()
                 if left_any_swing.sum()  > 0 else 0)
    right_tcs = (right_all_swing.sum() / right_any_swing.sum()
                 if right_any_swing.sum() > 0 else 0)

    print("Tripod Coordination Strength:")
    print(f"  Left  tripod (L1-R2-L3): {left_tcs:.2%}")
    print(f"  Right tripod (R1-L2-R3): {right_tcs:.2%}")

    return df, {'left_tcs': left_tcs, 'right_tcs': right_tcs}


def compute_phase_offset(swing_onsets_a, swing_onsets_b, fps):
    """Phase of each onset of leg A relative to leg B's stride cycle.

    For each onset of A, finds the enclosing B stride cycle and expresses
    A's timing as a fraction of B's period in radians [0, 2pi).

    Returns:
        1-D float array of phase offsets in [0, 2pi). Empty if insufficient data.
    """
    if len(swing_onsets_a) < 1 or len(swing_onsets_b) < 2:
        return np.array([])
    phases = []
    for t_a in swing_onsets_a:
        preceding = swing_onsets_b[swing_onsets_b < t_a]
        if len(preceding) == 0:
            continue
        t_b_prev = preceding[-1]
        following = swing_onsets_b[swing_onsets_b > t_b_prev]
        if len(following) == 0:
            continue
        period = following[0] - t_b_prev
        if period <= 0:
            continue
        phases.append((t_a - t_b_prev) / period * 2 * np.pi)
    return np.array(phases)


def compute_phase_offset_by_cycle(ref_onsets, leg_onsets, cycle_speeds, q_lo, q_hi):
    """Phase offsets (0-2pi) for T1_left step cycles in a given speed range.

    For each T1_left cycle i (ref_onsets[i] to ref_onsets[i+1]):
      - If cycle_speeds[i] is in [q_lo, q_hi): include leg onsets from that cycle.
      - Phase = (leg_onset - t0) / period * 2*pi.

    Args:
        ref_onsets:   sorted 1-D int array of T1_left swing onset indices
        leg_onsets:   sorted 1-D int array of other-leg swing onset indices
        cycle_speeds: 1-D float array, one value per cycle (len = len(ref_onsets)-1)
        q_lo, q_hi:   speed range [q_lo, q_hi)

    Returns:
        1-D float array of phase offsets in [0, 2pi)
    """
    if len(ref_onsets) < 2 or len(leg_onsets) < 1:
        return np.array([])
    phases = []
    for i in range(len(ref_onsets) - 1):
        spd = cycle_speeds[i]
        if np.isnan(spd) or not (q_lo <= spd < q_hi):
            continue
        t0, t1 = ref_onsets[i], ref_onsets[i + 1]
        period  = t1 - t0
        in_cyc  = leg_onsets[(leg_onsets >= t0) & (leg_onsets < t1)]
        for t_a in in_cyc:
            phases.append((t_a - t0) / period * 2 * np.pi)
    return np.array(phases)


def mean_resultant_length(phases):
    """Mean resultant vector length R in [0, 1] (circular synchrony measure).

    R = 1: perfect synchrony; R = 0: uniform distribution.
    """
    phases = np.asarray(phases)
    if len(phases) == 0:
        return np.nan
    return float(np.abs(np.mean(np.exp(1j * phases))))



## 7. Main Analysis Pipeline

In [ ]:
# Cell 9 -- Load Data
# Load bout-dict HDF5
bout_dict, bout_keys, fly_ids, clip_lengths = load_ik_bouts(H5_PATH)

# Metadata from info dict
names_qpos            = list(bout_dict['info']['names_qpos'])
names_xpos            = list(bout_dict['info']['names_xpos'])
egocentric_site_names = list(bout_dict['info']['site_names_egocentric'])

print(f"\nUnique fly IDs: {np.unique(fly_ids).tolist()}")
print("Bouts per fly:")
for fid, cnt in zip(*np.unique(fly_ids, return_counts=True)):
    print(f"  {fid}: {cnt} bouts")

# ── Diagnostics: coordinate frame and units ──────────────────────────────────
_first_key = bout_keys[0]
_thorax_candidates = [n for n in names_xpos if 'thorax' in n.lower()]
print(f"\nnames_qpos[:8]: {names_qpos[:8]}")
print(f"names_xpos[:6]: {names_xpos[:6]}")
print(f"Thorax-containing names in names_xpos: {_thorax_candidates}")

if _thorax_candidates:
    _th_name = _thorax_candidates[0]
    _th_idx  = list(names_xpos).index(_th_name)
    _full_xpos = np.array(bout_dict[_first_key]['xpos'])
    _full_qpos = np.array(bout_dict[_first_key]['qpos'])
    print(f"\nThorax '{_th_name}' xpos (first 3 frames):")
    print(_full_xpos[:3, _th_idx, :])
    print(f"Thorax x range: [{_full_xpos[:, _th_idx, 0].min():.6f}, "
          f"{_full_xpos[:, _th_idx, 0].max():.6f}]")
    print(f"Thorax y range: [{_full_xpos[:, _th_idx, 1].min():.6f}, "
          f"{_full_xpos[:, _th_idx, 1].max():.6f}]")
    print(f"qpos[0:3, 0:7] (root free joint — translation + quaternion):")
    print(_full_qpos[:3, :7])
    print(f"qpos root x range: [{_full_qpos[:, 0].min():.6f}, {_full_qpos[:, 0].max():.6f}]")
    print(f"qpos root y range: [{_full_qpos[:, 1].min():.6f}, {_full_qpos[:, 1].max():.6f}]")
else:
    print("WARNING: no 'thorax' body found in names_xpos — forward_speed will be NaN")
    print(f"All names_xpos: {list(names_xpos)}")


In [ ]:
# Cell 10 -- Build Joint Index
# Build joint index for current joint set
joint_index, joint_list = build_joint_index(
    names_qpos,
    JOINT_SETS[ACTIVE_JOINT_SET],
    LEGS
)

# Find egocentric leg-tip site indices.
# xpos_egocentric has shape (T, 50, 3) and is indexed by site_names_egocentric.
leg_tip_site_indices = find_leg_tip_site_indices(egocentric_site_names, LEGS)
print(f"\nLeg tip site indices: {leg_tip_site_indices}")

print("\nJoint index (first 6):")
for leg, joint, idx in joint_list[:6]:
    print(f"  {leg}_{joint} -> qpos[{idx}]")


In [ ]:
# Cell 11 -- Build DataFrame
# Build flat multi-bout DataFrame
df = bouts_to_dataframe(
    bout_dict, bout_keys, fly_ids, joint_list,
    names_xpos=names_xpos,
    heading_bodies=HEADING_BODIES,
    egocentric_site_names=egocentric_site_names,
    leg_tip_site_indices=leg_tip_site_indices
)

print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)[:15]}...")
print("\nBouts x fly summary (n_bouts | total_frames):")
print(df.groupby(['fly_id', 'bout_id']).size()
      .groupby('fly_id').agg(['count', 'sum'])
      .rename(columns={'count': 'n_bouts', 'sum': 'total_frames'}))


In [ ]:
# Cell 12 -- Preprocessing Pipeline
joint_cols = [f"{leg}_{joint}" for leg, joint, _ in joint_list]

# 1. Joint angle derivatives
df = compute_derivatives(df, joint_cols, fps=FPS)

# 2. Per-frame Hilbert phase from 3D tip speed
#    Convention: phase=0 → mid-swing (peak speed), phase=±π → mid-stance
df = compute_hilbert_phases(df, LEGS, fps=FPS, smooth_sigma=SMOOTH_SIGMA)

# 3. Swing/stance from Hilbert phase (replaces speed-threshold approach)
#    swing (0): |phase| < π/2;  stance (1): |phase| >= π/2
df = compute_swing_stance_from_hilbert(df, LEGS)
df = compute_n_legs_stance(df, LEGS)

# 4. Heading and velocity from body position (per bout)
df = compute_heading_and_velocity(df, fps=FPS, body=HEADING_BODIES[0])

# 5. Relative phase vs reference leg
df = compute_relative_phases(df, LEGS, reference_leg=REFERENCE_LEG)

phase_rel_cols = [c for c in df.columns if c.endswith('_phase_rel')]
print(f"Preprocessed: {len(df.columns)} columns")
print(f"Relative phase columns: {phase_rel_cols}")
spd = df['forward_speed'].dropna()
print(f"Forward speed (mm/s): "
      f"mean={spd.mean():.2f}  median={spd.median():.2f}")

# 6. Mean absolute joint velocity (locomotion activity proxy)
_d1_cols = [c for c in df.columns if c.endswith('_d1')]
_mav_raw  = df[_d1_cols].abs().mean(axis=1)
_mav_arr  = np.full(len(df), np.nan)
for _, _grp in df.groupby('bout_id', sort=False):
    _mav_arr[_grp.index] = gaussian_filter1d(
        _mav_raw.values[_grp.index], sigma=10)
df['mean_abs_vel'] = _mav_arr
_mav = df['mean_abs_vel'].dropna()
print(f"mean_abs_vel (rad/s): p25={_mav.quantile(0.25):.2f}  "
      f"p50={_mav.quantile(0.5):.2f}  p75={_mav.quantile(0.75):.2f}  "
      f"p90={_mav.quantile(0.9):.2f}")

# 7. Step-cycle ID and mean speed (cycles defined by T1_left Hilbert phase onsets)
if 'T1_left_phase' in df.columns:
    df = compute_step_cycle_speed(df, fps=FPS)
    _sc = df['step_cycle_mean_speed'].dropna()
    print(f"Step-cycle mean speed (mm/s): "
          f"median={_sc.median():.2f}  p25={_sc.quantile(0.25):.2f}  "
          f"p75={_sc.quantile(0.75):.2f}  n_frames={len(_sc):,}")
else:
    print("Warning: T1_left_phase not found -- re-run Cell 11 from the top.")

In [ ]:
# Cell 13 -- Build Feature Matrix
# Build feature matrix
X, feature_names, valid_mask = get_feature_matrix(df, joint_list, include_derivatives=True)

# Get aligned DataFrame for plotting
df_valid = df.loc[valid_mask].reset_index(drop=True)

In [ ]:
# Mean-center joint angle columns → df_mc
# Subtracts per-column grand mean so resting position = 0°.
# Used by paper figure cells; df_valid is unchanged.
_jangle_cols = [c for c in df_valid.columns
                if any(c == f"{leg}_{jnt}"
                       for leg in LEGS
                       for jnt in JOINT_SETS['full'])]
df_mc = df_valid.copy()
_jangle_means = {}
for _col in _jangle_cols:
    _jangle_means[_col] = float(df_mc[_col].mean())
    df_mc[_col] = df_mc[_col] - _jangle_means[_col]
print(f'Mean-centered {len(_jangle_cols)} joint columns → df_mc')


In [ ]:
# Cell 14 -- Run PCA
# Run PCA
pca_result, pca, scaler = run_pca(X, n_components=5)

In [ ]:
# Cell 15 -- Add PC Scores
# Add PC values to DataFrame
for i in range(pca_result.shape[1]):
    df_valid[f'PC{i+1}'] = pca_result[:, i]

In [ ]:
# Cell 16 -- Overview Distributions
# ── Overview distributions: speed, heading, height, leg spread ───────────────────
# All quantities per fly (overlaid histograms) + per-bout strip plot.
# forward_speed in mm/s; thorax height in real mm; leg spread in real mm².

_fly_order = sorted(df_valid['fly_id'].unique())

# ── Speed bin boundaries (used in phase-portrait cells) ──────────────────────────
_spd = df_valid['forward_speed']
_spd_q33 = _spd.quantile(0.33)
_spd_q67 = _spd.quantile(0.67)
print(f"Speed percentiles — p33: {_spd_q33:.2f} mm/s  |  p67: {_spd_q67:.2f} mm/s")

# ── Leg-spread: vectorised shoelace on egocentric XY of 6 leg tips ───────────────
# Vertex order (CCW viewed from above): L1 → L2 → L3 → R3 → R2 → R1
_TIP_ORDER = ['T1_left', 'T2_left', 'T3_left', 'T3_right', 'T2_right', 'T1_right']
_x_tip_cols = [f'{l}_tip_x_ego' for l in _TIP_ORDER]
_y_tip_cols = [f'{l}_tip_y_ego' for l in _TIP_ORDER]
_has_spread = all(c in df_valid.columns for c in _x_tip_cols + _y_tip_cols)
if _has_spread:
    _xs = df_valid[_x_tip_cols].values   # (N, 6) model-metres
    _ys = df_valid[_y_tip_cols].values
    _xs_next = np.roll(_xs, -1, axis=1)
    _ys_next = np.roll(_ys, -1, axis=1)
    _area_m2  = 0.5 * np.abs(np.sum(_xs * _ys_next - _xs_next * _ys, axis=1))
    # 1 model-m = 10 real mm  →  1 model-m² = 100 real mm²
    df_valid['leg_spread_mm2'] = _area_m2 * 100
    print(f"Leg spread (mm²): median={df_valid['leg_spread_mm2'].median():.2f}  "
          f"p5={df_valid['leg_spread_mm2'].quantile(0.05):.2f}  "
          f"p95={df_valid['leg_spread_mm2'].quantile(0.95):.2f}")
else:
    print("WARNING: tip_x_ego / tip_y_ego columns missing — re-run cells 5 and 17.")

# ── Figure 1: 2×2 KDE overview ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Panel (0,0): Forward speed ────────────────────────────────────────────────────
ax = axes[0, 0]
for fly in _fly_order:
    s = _spd[df_valid['fly_id'] == fly].dropna()
    s = s[s <= s.quantile(0.99)]
    ax.hist(s, bins=60, density=True, alpha=0.45, label=fly.split('/')[-1])
ax.axvline(_spd_q33, color='k', ls='--', lw=1.5, label=f'p33 = {_spd_q33:.2f} mm/s')
ax.axvline(_spd_q67, color='k', ls=':',  lw=1.5, label=f'p67 = {_spd_q67:.2f} mm/s')
ax.set_xlabel('Forward speed (mm/s)')
ax.set_ylabel('Density')
ax.set_title('Speed distribution by fly')
ax.legend(fontsize=8)
sns.despine(ax=ax)

# ── Panel (0,1): Heading deviation from corridor axis ───────────────────────────
# Fold raw heading [-180°, 180°] to corridor-axis deviation [0°, 90°]:
#   0° = going straight along corridor (either direction)
#  90° = going perfectly perpendicular to the corridor
# Transform: h = heading mod 180  (collapses ±180 → 0);  dev = min(h, 180-h)  (→ [0,90])
ax = axes[0, 1]
_h_mod = np.mod(np.degrees(df_valid['heading']), 180)   # [0, 180)
_heading_dev = np.minimum(_h_mod, 180 - _h_mod)          # [0, 90]
df_valid['heading_dev'] = _heading_dev
for fly in _fly_order:
    h = _heading_dev[df_valid['fly_id'] == fly].dropna()
    ax.hist(h, bins=45, density=True, alpha=0.45, label=fly.split('/')[-1])
ax.set_xlabel('Heading deviation from corridor axis (°)')
ax.set_ylabel('Density')
ax.set_title('Heading distribution by fly\n(0° = straight along corridor, 90° = perpendicular)')
ax.legend(fontsize=8)
sns.despine(ax=ax)

# ── Panel (1,0): Thorax height ───────────────────────────────────────────────────
ax = axes[1, 0]
# thorax_z is in model-metres; × 10 → real mm (1 model-m = 1/100 real-m = 10 mm)
_height_mm = df_valid['thorax_z'] * 10
for fly in _fly_order:
    h = _height_mm[df_valid['fly_id'] == fly].dropna()
    h = h[h.between(h.quantile(0.01), h.quantile(0.99))]
    ax.hist(h, bins=60, density=True, alpha=0.45, label=fly.split('/')[-1])
ax.set_xlabel('Thorax height (mm)')
ax.set_ylabel('Density')
ax.set_title('Height distribution by fly')
ax.legend(fontsize=8)
sns.despine(ax=ax)

# ── Panel (1,1): Leg spread ───────────────────────────────────────────────────────
ax = axes[1, 1]
if _has_spread:
    _spread = df_valid['leg_spread_mm2']
    for fly in _fly_order:
        s = _spread[df_valid['fly_id'] == fly].dropna()
        s = s[s <= s.quantile(0.99)]
        ax.hist(s, bins=60, density=True, alpha=0.45, label=fly.split('/')[-1])
    ax.set_xlabel('Leg-spread area (mm²)')
    ax.set_ylabel('Density')
    ax.set_title('Leg-spread distribution by fly')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'leg_spread_mm2 not available\n(re-run cells 5 and 17)',
            ha='center', va='center', transform=ax.transAxes, color='red')
    ax.set_title('Leg-spread distribution (unavailable)')
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distributions_overview.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'distributions_overview.pdf'}")

# ── Figure 2: Per-bout median speed strip plot ────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 5))
_bout_spd = (df_valid
             .groupby(['fly_id', 'bout_id'])['forward_speed']
             .median()
             .reset_index()
             .rename(columns={'forward_speed': 'speed_mms'}))
sns.stripplot(data=_bout_spd, x='fly_id', y='speed_mms',
              order=_fly_order, jitter=True, size=4, ax=ax2)
ax2.axhline(_spd_q33, color='k', ls='--', lw=1, label=f'p33 = {_spd_q33:.2f}')
ax2.axhline(_spd_q67, color='k', ls=':',  lw=1, label=f'p67 = {_spd_q67:.2f}')
ax2.set_xlabel('Fly ID')
ax2.set_ylabel('Median speed per bout (mm/s)')
ax2.set_title('Per-bout speed by fly  (each dot = one bout)')
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', rotation_mode='anchor')
ax2.legend(fontsize=8)
sns.despine(ax=ax2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_distribution_perbout.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'speed_distribution_perbout.pdf'}")


---

## Section 3 — Phase Coordination

Relative phase of each leg vs **T1_left** (front left), using two complementary methods:
- **Cell 24 — Hilbert (continuous)**: Circular histograms of `{leg}_phase_rel` columns
- **Cell 25 — Onset-based (discrete)**: Swing-onset timing → polar plots + R matrix per fly


In [ ]:
# Cell 17 -- Step-Cycle Diagnostic
# --- Step-cycle diagnostic: 10 short + 10 long bouts, 4-panel per bout ------
from matplotlib.backends.backend_pdf import PdfPages

_LEG_ORDER  = ['T1_left', 'T2_left', 'T3_left', 'T1_right', 'T2_right', 'T3_right']
_LEG_COLORS = {leg: ('#888888', 0.8, 0.6) for leg in _LEG_ORDER}
_LEG_COLORS['T1_left'] = ('#e03030', 2.0, 1.0)   # bold red for reference leg

# Bout selection: 10 shortest + 10 longest
_bout_lengths = df.groupby('bout_id').size().sort_values()
_n_diag      = 5
_long_bouts  = list(_bout_lengths.index[-_n_diag:])
_short_bouts = list(_bout_lengths.index[:_n_diag])
_diag_bouts  = _short_bouts + _long_bouts
print(f"Diagnostic: {len(_short_bouts)} short + {len(_long_bouts)} long bouts "
      f"= {len(_diag_bouts)} pages")

_phase_col = 'T1_left_phase'
_spd_col  = 'forward_speed'
_sc_col   = 'step_cycle_mean_speed'
_pdf_path = OUTPUT_DIR / 'step_cycle_diagnostic.pdf'
_has_sc   = _sc_col in df.columns
if not _has_sc:
    print(f"Warning: {_sc_col} not found -- run cell 18 steps 8-9 first.")

with PdfPages(_pdf_path) as _pdf:
    for _bid in _diag_bouts:
        _grp = df[df['bout_id'] == _bid].reset_index(drop=True)
        T    = len(_grp)
        _fx  = np.arange(T)

        # Swing onsets (1->0 transitions of T1_left_swing_stance)
        _onsets = (liftoff_from_hilbert(_grp[_phase_col].values)
                   if _phase_col in _grp.columns else np.array([], dtype=int))

        _fig, _axes = plt.subplots(
            4, 1, figsize=(14, 10), sharex=True,
            gridspec_kw={'height_ratios': [2, 1, 1.5, 1.5]})
        _fig.suptitle(
            f'Bout: {_bid}  |  {T} frames  ({T / FPS:.2f} s)  '
            f'|  {len(_onsets)} swing onsets',
            fontsize=11, fontweight='bold')

        # -- Panel 1: all 6 leg tip Z (real mm) --------------------------------
        _ax1 = _axes[0]
        for _leg in _LEG_ORDER:
            _col = f'{_leg}_tip_z_world'
            if _col not in _grp.columns:
                continue
            _z_mm = _grp[_col].values * 10   # model-m -> real mm
            _c, _lw, _al = _LEG_COLORS[_leg]
            _ax1.plot(_fx, _z_mm, color=_c, lw=_lw, alpha=_al,
                      label=LEG_LABELS.get(_leg, _leg))
        _ax1.set_ylabel('Tip Z (real mm)', fontsize=9)
        _ax1.set_title('All leg tips -- world-frame Z', fontsize=9)
        _ax1.legend(loc='upper right', fontsize=7, ncol=3, frameon=False)

        # -- Panel 2: T1_left swing_stance step fn + onset markers -------------
        _ax2 = _axes[1]
        _ss_col = 'T1_left_swing_stance'
        if _ss_col in _grp.columns:
            _ax2.step(_fx, _grp[_ss_col].values, where='post',
                      color='steelblue', lw=1.2)
        for _on in _onsets:
            _ax2.axvline(_on, color='red', lw=1.0, linestyle='--', alpha=0.7)
        _ax2.set_yticks([0, 1])
        _ax2.set_yticklabels(['swing', 'stance'], fontsize=8)
        _ax2.set_ylabel('T1_left', fontsize=9)
        _ax2.set_title(
            f'T1_left swing_stance  '
            f'(red dashes = swing onsets i.e. liftoffs; {len(_onsets)} detected)',
            fontsize=9)

        # -- Panel 3: instantaneous forward speed ------------------------------
        _ax3 = _axes[2]
        _spd_vals = (_grp[_spd_col].values if _spd_col in _grp.columns
                     else np.full(T, np.nan))
        _ax3.plot(_fx, _spd_vals, color='#2a6e2a', lw=0.8)
        for _on in _onsets:
            _ax3.axvline(_on, color='#aaaaaa', lw=0.6, linestyle=':', alpha=0.7)
        _ax3.set_ylabel('mm/s', fontsize=9)
        _ax3.set_title('Instantaneous forward speed', fontsize=9)

        # -- Panel 4: step-cycle mean speed (step function) --------------------
        _ax4 = _axes[3]
        if _has_sc:
            _sc_vals = _grp[_sc_col].values.astype(float)
            _ax4.step(_fx, _sc_vals, where='post', color='#994400', lw=1.2)
        else:
            _ax4.text(0.5, 0.5, f'{_sc_col} not computed',
                      ha='center', va='center', transform=_ax4.transAxes, fontsize=9)
        for _on in _onsets:
            _ax4.axvline(_on, color='#aaaaaa', lw=0.6, linestyle=':', alpha=0.7)
        _ax4.set_ylabel('mm/s', fontsize=9)
        _ax4.set_xlabel('Frame', fontsize=9)
        _ax4.set_title('Step-cycle mean speed  (NaN gaps = partial cycles)', fontsize=9)

        plt.tight_layout()
        plt.show()
        _pdf.savefig(_fig, bbox_inches='tight')
        plt.close(_fig)
        

print(f"Saved {len(_diag_bouts)}-page PDF: {_pdf_path}")


In [ ]:
# Cell 18 -- Leg Coordination Polar Plots
# --- 3b: Onset-based polar plots + per-fly R matrix -------------------------
# Coordination analysis split by STEP-CYCLE mean speed quartile.
# Each step cycle (between consecutive T1_left swing onsets) contributes
# its phase offsets to whichever speed quartile its step_cycle_mean_speed
# falls in. One bout can span multiple quartiles.
import math
import matplotlib.gridspec as gridspec

_non_ref    = [l for l in LEGS if l != REFERENCE_LEG]
_fly_ids_u  = sorted(df['fly_id'].unique())
_ref_phase_col = f"{REFERENCE_LEG}_phase"
_sc_spd_col = 'step_cycle_mean_speed'

# Global quartile thresholds from all step cycles
_sc_spd_all = df[_sc_spd_col].dropna()
_q33, _q67, _q75 = _sc_spd_all.quantile([0.33, 0.67, 0.75]).values
_SPEED_QUARTILES = [
    ('Q1 (slowest)', 0,     _q33,  'q1'),
    ('Q2',           _q33,  _q67,  'q2'),
    ('Q3',           _q67,  np.inf,  'q3'), #,  _q75,
    # ('Q4 (fastest)', _q75,  np.inf,'q4'),
]
print(f"Step-cycle speed quartile boundaries (mm/s):  "
    #   f"Q1 < {_q25:.1f}  |  Q2 < {_q50:.1f}  |  Q3 < {_q75:.1f}  |  Q4 >= {_q75:.1f}")
      f"Q1 < {_q33:.1f}  |  Q2 < {_q67:.1f}  | Q3 >= {_q67:.1f}")
print()

# Pre-allocate accumulators: _fly_phases_q[q_tag][fly_id][leg] = list of phases
_fly_phases_q = {
    q_tag: {fid: {leg: [] for leg in _non_ref} for fid in _fly_ids_u}
    for _, _, _, q_tag in _SPEED_QUARTILES
}

if _ref_phase_col not in df.columns:
    print(f"WARNING: {_ref_phase_col} not found — re-run Cell 11.")
else:
    for _bid, _grp in df.groupby('bout_id', sort=False):
        _fid  = _grp['fly_id'].iloc[0]
        _spd  = _grp[_sc_spd_col].values         # step_cycle_mean_speed per frame

        _ref_onsets = liftoff_from_hilbert(_grp[f'{REFERENCE_LEG}_phase'].values)
        if len(_ref_onsets) < 2:
            continue

        # Speed of each T1_left cycle (constant within cycle; sample at cycle start)
        _cycle_speeds = _spd[_ref_onsets[:-1]]

        for _leg in _non_ref:
            if f"{_leg}_phase" not in _grp.columns:
                continue
            _leg_onsets = liftoff_from_hilbert(_grp[f'{_leg}_phase'].values)

            for _, _q_lo, _q_hi, _q_tag in _SPEED_QUARTILES:
                _ph = compute_phase_offset_by_cycle(
                    _ref_onsets, _leg_onsets, _cycle_speeds, _q_lo, _q_hi)
                if len(_ph) > 0:
                    _fly_phases_q[_q_tag][_fid][_leg].extend(_ph.tolist())

# Convert accumulated lists to arrays
for _, _, _, _q_tag in _SPEED_QUARTILES:
    for _fid in _fly_phases_q[_q_tag]:
        for _leg in _fly_phases_q[_q_tag][_fid]:
            _fly_phases_q[_q_tag][_fid][_leg] = np.array(
                _fly_phases_q[_q_tag][_fid][_leg])

# ── Loop for plotting — one iteration per quartile ────────────────────────────────────────────────────────────────────────────
for _q_label, _q_lo, _q_hi, _q_tag in _SPEED_QUARTILES:
    _fly_phases = _fly_phases_q[_q_tag]
    _q_hi_str   = f'{_q_hi:.1f}' if np.isfinite(_q_hi) else '∞'
    _spd_range  = f'[{_q_lo:.1f}–{_q_hi_str} mm/s]'
    _n_cycles = sum(
        len(_fly_phases[_fid][_non_ref[0]])
        for _fid in _fly_ids_u
        if len(_fly_phases[_fid][_non_ref[0]]) > 0
    )
    print(f"{'='*62}")
    print(f"{_q_label}  {_spd_range}: {_n_cycles} step-cycle onsets")
    if _n_cycles == 0:
        print("  No data — skipping.")
        continue
    # ── Polar histograms ─────────────────────────────────────────────────
    _n_bins    = N_PHASE_BINS
    _bin_edges = np.linspace(0, 2 * np.pi, _n_bins + 1)
    _bin_cents = (_bin_edges[:-1] + _bin_edges[1:]) / 2
    _fly_pal   = plt.cm.get_cmap('tab10', max(len(_fly_ids_u), 1))
    _fly_clr   = {fid: _fly_pal(i) for i, fid in enumerate(_fly_ids_u)}

    # Pass 1: grand-mean histograms + per-quartile rmax
    _hist_ob = {}
    _rmax_ob = 0.0
    for _leg in _non_ref:
        _all = np.concatenate([_fly_phases[f][_leg] for f in _fly_ids_u
                               if len(_fly_phases[f][_leg]) > 0], axis=0) \
               if any(len(_fly_phases[f][_leg]) > 0 for f in _fly_ids_u) \
               else np.array([])
        if len(_all) > 0:
            _cnt, _ = np.histogram(_all, bins=_bin_edges)
            _cn     = _cnt / _cnt.sum()
            _hist_ob[_leg] = (_cn, mean_resultant_length(_all))
            _rmax_ob = max(_rmax_ob, _cn.max())
    _rmax_ob *= 1.15

    # Pass 2: draw
    _n_pairs = len(_non_ref)
    _n_cols  = 3
    _n_rows  = math.ceil((_n_pairs + 1) / _n_cols)

    _fig = plt.figure(figsize=(5.5 * _n_cols, 4.8 * _n_rows))
    _gs  = gridspec.GridSpec(_n_rows, _n_cols, figure=_fig,
                             hspace=0.6, wspace=0.4)
    _axes_polar = [
        _fig.add_subplot(_gs[_i // _n_cols, _i % _n_cols], projection='polar')
        for _i in range(_n_pairs)
    ]
    _ax_leg = _fig.add_subplot(_gs[_n_pairs // _n_cols, _n_pairs % _n_cols])
    _ax_leg.set_axis_off()

    _tick_pos    = np.linspace(0, 2 * np.pi, 5)[:-1]   # 0, π/2, π, 3π/2
    _tick_labels = ['0', 'π/2', 'π', '3π/2']

    for _ax, _leg in zip(_axes_polar, _non_ref):
        _R_vals = []
        for _fid in _fly_ids_u:
            _ph = _fly_phases[_fid][_leg]
            if len(_ph) < 3:
                continue
            _cnt, _ = np.histogram(_ph, bins=_bin_edges)
            _cn     = _cnt / _cnt.sum()
            _R_vals.append(mean_resultant_length(_ph))
            _bc_cl  = np.r_[_bin_cents, _bin_cents[:1]]
            _cn_cl  = np.r_[_cn, _cn[:1]]
            _ax.plot(_bc_cl, _cn_cl, color=_fly_clr[_fid], lw=1.2, alpha=0.6)
            _ax.fill(_bc_cl, _cn_cl, color=_fly_clr[_fid], alpha=0.12)

        _R_all = np.nan
        if _leg in _hist_ob:
            _cn_all, _R_all = _hist_ob[_leg]
            _ax.plot(np.r_[_bin_cents, _bin_cents[:1]],
                     np.r_[_cn_all, _cn_all[:1]],
                     color='k', lw=3.0, zorder=10, solid_capstyle='round')

        _ax.set_ylim(0, max(_rmax_ob, 0.01))
        _ax.yaxis.set_visible(False)
        _ax.set_xticks(_tick_pos)
        _ax.set_xticklabels(_tick_labels, fontsize=11)
        _ax.tick_params(axis='x', pad=5)
        _ax.grid(True, linewidth=0.5, linestyle=':', alpha=0.6)
        _ax.spines['polar'].set_linewidth(1.2)

        _lbl = LEG_LABELS.get(_leg, _leg)
        _ref = LEG_LABELS.get(REFERENCE_LEG, REFERENCE_LEG)
        _ax.set_title(
            f'{_lbl} vs {_ref}\n'
            f'R={_R_all:.2f}   \u27e8R\u27e9={np.nanmean(_R_vals):.2f}',
            fontsize=12, pad=16, fontweight='bold'
        )

    _handles = [
        plt.Line2D([0], [0], color='k', lw=3.0, label='Grand mean'),
    ] + [
        plt.Line2D([0], [0], color=_fly_clr[_fid], lw=2.0, label=str(_fid))
        for _fid in _fly_ids_u
    ]
    _ax_leg.legend(handles=_handles, loc='center', fontsize=11,
                   frameon=False, title='Fly ID', title_fontsize=12)

    _fig.suptitle(
        f'Onset-based inter-leg phase offsets\u2003{_q_label}\u2003{_spd_range}\n'
        f'(reference: {LEG_LABELS.get(REFERENCE_LEG, REFERENCE_LEG)})',
        fontsize=13, fontweight='bold'
    )
    _fname = OUTPUT_DIR / f'phase_coordination_onsets_polar_{_q_tag}.pdf'
    plt.savefig(_fname, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {_fname}")

    # ── R matrix ─────────────────────────────────────────────────────────
    _R_matrix = np.full((len(_fly_ids_u), len(_non_ref)), np.nan)
    for _i, _fid in enumerate(_fly_ids_u):
        for _j, _leg in enumerate(_non_ref):
            _R_matrix[_i, _j] = mean_resultant_length(_fly_phases[_fid][_leg])

    _R_plot  = np.vstack([_R_matrix, np.nanmean(_R_matrix, axis=0)])
    _row_lbl = [str(f)[:15] for f in _fly_ids_u] + ['MEAN']
    _col_lbl = [LEG_LABELS.get(l, l) for l in _non_ref]

    _fig2, _ax2 = plt.subplots(figsize=(max(5, len(_non_ref) * 1.4),
                                         max(3, len(_fly_ids_u) * 0.7 + 1.5)))
    _im = _ax2.imshow(_R_plot, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
    _ax2.set_xticks(range(len(_col_lbl)));  _ax2.set_xticklabels(_col_lbl, fontsize=10)
    _ax2.set_yticks(range(len(_row_lbl)));  _ax2.set_yticklabels(_row_lbl, fontsize=9)
    _ax2.axhline(len(_fly_ids_u) - 0.5, color='k', lw=2)
    for _i in range(_R_plot.shape[0]):
        for _j in range(_R_plot.shape[1]):
            if not np.isnan(_R_plot[_i, _j]):
                _ax2.text(_j, _i, f'{_R_plot[_i, _j]:.2f}', ha='center', va='center',
                          fontsize=9, color='k' if _R_plot[_i, _j] < 0.7 else 'w')
    plt.colorbar(_im, ax=_ax2, label='R (coordination strength)')
    _ax2.set_title(
        f'Coordination R\u2003{_q_label}\u2003{_spd_range}\n'
        f'(rows=flies, cols=pair vs {LEG_LABELS.get(REFERENCE_LEG, REFERENCE_LEG)})',
        fontsize=11
    )
    plt.tight_layout()
    _fname2 = OUTPUT_DIR / f'phase_coordination_R_matrix_{_q_tag}.pdf'
    plt.savefig(_fname2, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {_fname2}")

# ── Restore full-df state for downstream cells (Section 5) ───────────────────
_fly_ids_u  = sorted(df['fly_id'].unique())
_fly_phases_full = {fid: {leg: [] for leg in _non_ref} for fid in _fly_ids_u}
_ref_ss_col = f"{REFERENCE_LEG}_swing_stance"
if _ref_ss_col in df.columns:
    for _bid, _grp in df.groupby('bout_id', sort=False):
        _fid        = _grp['fly_id'].iloc[0]
        _ref_onsets = liftoff_from_hilbert(_grp[f'{REFERENCE_LEG}_phase'].values)
        if len(_ref_onsets) < 2:
            continue
        for _leg in _non_ref:
            if f"{_leg}_phase" not in _grp.columns:
                continue
            _leg_onsets = liftoff_from_hilbert(_grp[f'{_leg}_phase'].values)
            _ph = compute_phase_offset(_leg_onsets, _ref_onsets, FPS)
            if len(_ph) > 0:
                _fly_phases_full[_fid][_leg].extend(_ph.tolist())
# Export as expected by downstream cells
fly_phases   = {fid: {leg: np.array(v) for leg, v in d.items()}
                for fid, d in _fly_phases_full.items()}
non_ref_legs = _non_ref
R_matrix     = np.full((len(_fly_ids_u), len(_non_ref)), np.nan)
for _i, _fid in enumerate(_fly_ids_u):
    for _j, _leg in enumerate(_non_ref):
        R_matrix[_i, _j] = mean_resultant_length(fly_phases[_fid][_leg])
fly_ids_u_ob = _fly_ids_u
print(f"\nDone. Full-data state restored for downstream cells.")
print(f"Saved coordination plots to {OUTPUT_DIR}")


---

## 8. Visualizations

In [ ]:
# Cell 19 -- Single-Bout Coordination Diagnostic
# --- Single-bout coordination diagnostic ------------------------------------
# Set DIAG_BOUT to any bout key shown by the print below, then re-run.
# Figure 1: world-frame Z traces + swing onset markers for all 6 legs.
# Figure 2: per-T1_left-cycle polar plots showing each leg's phase offset.
import math as _math

DIAG_BOUT = 'bout_358'   # ← CHANGE THIS

# ── Available bouts ──────────────────────────────────────────────────────────
_bout_lens = df.groupby('bout_id').size().sort_values(ascending=False)
print(f"Available bouts sorted by length ({len(_bout_lens)} total):")
_col_w = max(len(k) for k in _bout_lens.index)
for _bk, _bl in _bout_lens.items():
    print(f"  {_bk:{_col_w}s}  {_bl:5d} frames  ({_bl / FPS:.2f} s)")
print()

# ── Leg color map (consistent across both figures) ───────────────────────────
_LEG_CLR = {
    'T1_left':  '#d62728',
    'T1_right': '#ff7f0e',
    'T2_left':  '#1f77b4',
    'T2_right': '#9467bd',
    'T3_left':  '#2ca02c',
    'T3_right': '#8c564b',
}

# ── Extract bout ─────────────────────────────────────────────────────────────
if DIAG_BOUT not in df['bout_id'].values:
    print(f"ERROR: {DIAG_BOUT!r} not found. Choose from the list above.")
else:
    _grp = df[df['bout_id'] == DIAG_BOUT].reset_index(drop=True)
    _fid = _grp['fly_id'].iloc[0]
    T    = len(_grp)
    _fx  = np.arange(T)

    _ref_ss_col = f"{REFERENCE_LEG}_swing_stance"
    _non_ref    = [l for l in LEGS if l != REFERENCE_LEG]

    # Swing onsets per leg (same function + same column used in Cell 18)
    _all_onsets = {}
    for _leg in LEGS:
        _phase_col = f"{_leg}_phase"
        _all_onsets[_leg] = (liftoff_from_hilbert(_grp[_phase_col].values)
                             if _phase_col in _grp.columns else np.array([], int))

    _ref_onsets = _all_onsets[REFERENCE_LEG]
    _n_cycles   = max(0, len(_ref_onsets) - 1)
    _sc_spd     = _grp['step_cycle_mean_speed'].values

    print(f"Bout: {DIAG_BOUT}  |  fly: {_fid}  |  {T} frames "
          f"({T / FPS:.2f} s)  |  {_n_cycles} {REFERENCE_LEG} cycles")
    for _leg in LEGS:
        print(f"  {LEG_LABELS.get(_leg, _leg):3s} ({_leg}): "
              f"{len(_all_onsets[_leg])} swing onsets")
    print()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 1 — Z traces + onset markers
    # ════════════════════════════════════════════════════════════════════════
    _n_rows    = 1 + len(LEGS)
    _h_ratios  = [3.0] + [0.65] * len(LEGS)
    _fig1, _axes1 = plt.subplots(
        _n_rows, 1, figsize=(16, 3.5 + 0.75 * len(LEGS)),
        sharex=True, gridspec_kw={'height_ratios': _h_ratios}
    )
    _fig1.suptitle(
        f"Bout {DIAG_BOUT}  |  fly {_fid}  |  {T} frames ({T / FPS:.2f} s)"
        f"  |  {_n_cycles} {REFERENCE_LEG} cycles",
        fontsize=11, fontweight='bold'
    )

    # Row 0 — all Z traces overlaid
    _ax_z = _axes1[0]
    for _leg in LEGS:
        _z_col = f"{_leg}_tip_z_world"
        if _z_col not in _grp.columns:
            continue
        _z_mm = _grp[_z_col].values * 10
        _ax_z.plot(_fx, _z_mm, color=_LEG_CLR[_leg], lw=1.0,
                   label=LEG_LABELS.get(_leg, _leg), alpha=0.9)
        # Onset markers for this leg
        for _on in _all_onsets[_leg]:
            _ax_z.axvline(_on, color=_LEG_CLR[_leg], lw=0.8,
                          linestyle='--', alpha=0.55, ymin=0, ymax=1)
    # T1_left cycle boundaries as light gray
    for _on in _ref_onsets:
        _ax_z.axvline(_on, color='#bbbbbb', lw=0.7, linestyle='-', alpha=0.5,
                      zorder=0)
    _ax_z.set_ylabel('Tip Z world (mm)', fontsize=9)
    _ax_z.legend(loc='upper right', fontsize=8, ncol=3, frameon=False)
    _ax_z.set_title('World-frame claw Z  (dashed = swing onset per leg)', fontsize=9)

    # Rows 1–6 — per-leg swing_stance step function + onsets
    for _ri, _leg in enumerate(LEGS):
        _ax = _axes1[_ri + 1]
        _ss_col = f"{_leg}_swing_stance"
        if _ss_col in _grp.columns:
            _ax.step(_fx, _grp[_ss_col].values, where='post',
                     color=_LEG_CLR[_leg], lw=1.1)
        for _on in _all_onsets[_leg]:
            _ax.axvline(_on, color=_LEG_CLR[_leg], lw=0.9,
                        linestyle='--', alpha=0.7)
        _ax.set_yticks([0, 1])
        _ax.set_yticklabels(['sw', 'st'], fontsize=7)
        _ax.set_ylabel(LEG_LABELS.get(_leg, _leg), fontsize=9, rotation=0,
                       labelpad=22)
        _ax.tick_params(axis='x', labelbottom=False)

    _axes1[-1].tick_params(axis='x', labelbottom=True)
    _axes1[-1].set_xlabel(f'Frame  ({FPS} fps)', fontsize=9)
    plt.tight_layout()
    _fig1_path = OUTPUT_DIR / f'diag_bout_{DIAG_BOUT}_timeseries.pdf'
    plt.savefig(_fig1_path, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_fig1_path}")

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 2 — Per-cycle polar grid
    # ════════════════════════════════════════════════════════════════════════
    if _n_cycles < 1:
        print("No complete T1_left step cycles — skipping polar figure.")
    else:
        _n_cols2  = min(_n_cycles, 5)
        _n_rows2  = _math.ceil(_n_cycles / _n_cols2)
        _fig2, _axes2 = plt.subplots(
            _n_rows2, _n_cols2,
            figsize=(_n_cols2 * 2.6, _n_rows2 * 2.9),
            subplot_kw={'projection': 'polar'}
        )
        # Flatten axes array for uniform iteration
        _axes2_flat = np.array(_axes2).flatten() if _n_cycles > 1 else [_axes2]

        _tick_pos = [0, np.pi / 2, np.pi, 3 * np.pi / 2]
        _tick_lbl = ['0', 'π/2', 'π', '3π/2']

        for _ci in range(_n_cycles):
            _ax = _axes2_flat[_ci]
            _ax.set_theta_zero_location('N')   # 0 = top = start of cycle
            _ax.set_theta_direction(-1)         # clockwise = forward in time
            _ax.set_ylim(0, 1.18)
            _ax.set_yticks([])
            _ax.set_xticks(_tick_pos)
            _ax.set_xticklabels(_tick_lbl, fontsize=7)

            _t0     = _ref_onsets[_ci]
            _t1     = _ref_onsets[_ci + 1]
            _period = _t1 - _t0
            _cyc_spd = _sc_spd[_t0] if not np.isnan(_sc_spd[_t0]) else np.nan

            # Unit circle reference
            _theta_ring = np.linspace(0, 2 * np.pi, 200)
            _ax.plot(_theta_ring, np.ones(200), color='#cccccc', lw=0.7, zorder=1)

            for _leg in _non_ref:
                _leg_ons = _all_onsets[_leg]
                _in_cyc  = _leg_ons[(_leg_ons >= _t0) & (_leg_ons < _t1)]
                if len(_in_cyc) == 0:
                    # Mark absence with × at a fixed angle for this leg
                    _slot = _non_ref.index(_leg) * (2 * np.pi / len(_non_ref))
                    _ax.scatter(_slot, 0.75, s=50, color=_LEG_CLR[_leg],
                                marker='x', linewidths=1.5, zorder=4, alpha=0.6)
                else:
                    for _t_a in _in_cyc:
                        _ph = (_t_a - _t0) / _period * 2 * np.pi
                        _ax.scatter(_ph, 1.0, s=65, color=_LEG_CLR[_leg],
                                    zorder=5, edgecolors='none')

            _spd_str = f"{_cyc_spd:.0f}" if not np.isnan(_cyc_spd) else 'n/a'
            _ax.set_title(
                f"#{_ci}  {_spd_str}mm/s\n{_period}fr",
                fontsize=8, pad=4
            )

        # Hide unused polar axes
        for _ci in range(_n_cycles, len(_axes2_flat)):
            _axes2_flat[_ci].set_visible(False)

        # Shared legend
        _leg_handles = [
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=_LEG_CLR[_leg], markersize=9,
                       label=LEG_LABELS.get(_leg, _leg))
            for _leg in _non_ref
        ]
        _fig2.legend(handles=_leg_handles, loc='lower center',
                     ncol=len(_non_ref), fontsize=9, frameon=False,
                     bbox_to_anchor=(0.5, -0.02))
        _fig2.suptitle(
            f"Per-cycle phase offsets  |  {DIAG_BOUT}  |  {_n_cycles} cycles"
            f"\n(0 = top, clockwise. Ref = {REFERENCE_LEG}.  × = no onset in cycle)",
            fontsize=10, fontweight='bold'
        )
        plt.tight_layout()
        _fig2_path = OUTPUT_DIR / f'diag_bout_{DIAG_BOUT}_per_cycle_polar.pdf'
        plt.savefig(_fig2_path, bbox_inches='tight')
        plt.show()
        print(f"Saved: {_fig2_path}")


In [ ]:
# Cell 20 -- PCA Variance Plot
# PCA Variance plot (Scree plot)
fig, ax = plt.subplots(figsize=(8, 5))
plot_pca_variance(pca, ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_variance.pdf')
plt.show()

In [ ]:
# Cell 21 -- PCA Loadings Heatmap
# PCA Loadings heatmap
fig, ax = plot_pca_loadings(pca, feature_names, joint_list, n_pcs=4)
plt.savefig(OUTPUT_DIR / 'pca_loadings.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 22 -- Multi-Variable PCA Embeddings
# Multi-panel PCA embeddings coloured by 6 variables
_color_vars = [
    ('fly_id',       'Fly ID'),
    ('n_legs_stance', 'N legs stance'),
    ('forward_speed', 'Forward speed (mm/s)'),
    ('heading',       'Heading (rad)'),
    ('turning_rate',  'Turning rate (rad/s)'),
    (f'{REFERENCE_LEG}_phase', f'{REFERENCE_LEG} phase (rad)'),
]
_cv_avail = [(cv, lbl) for cv, lbl in _color_vars if cv in df_valid.columns]

_ncols = min(3, len(_cv_avail))
_nrows = int(np.ceil(len(_cv_avail) / _ncols))
_fig, _axes = plt.subplots(_nrows, _ncols, figsize=(7 * _ncols, 6 * _nrows))
_axes_flat  = np.array(_axes).reshape(-1) if len(_cv_avail) > 1 else [_axes]

for _ax, (_cv, _lbl) in zip(_axes_flat, _cv_avail):
    plot_embedding(pca_result, df_valid, color_by=_cv, title=_lbl, ax=_ax)
for _ax in _axes_flat[len(_cv_avail):]:
    _ax.set_visible(False)

_fig.suptitle('PCA embedding — multi-variable coloring', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_multicolor.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'pca_multicolor.pdf'}")


In [ ]:
# Cell 23 -- Phase-Averaged PCA Trajectory
# Phase-averaged PCA trajectory (gait cycle in PC space)
fig_pa, ax_pa = plt.subplots(figsize=(8, 7))
plot_phase_averaged_pca(df_valid, pca_result, reference_leg=REFERENCE_LEG,
                         n_bins=N_PHASE_BINS, ax=ax_pa)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_phase_averaged.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'pca_phase_averaged.pdf'}")


In [ ]:
# Cell 24 -- PCA Trajectory by Stance Count
# PCA trajectory colored by n_legs_stance
if 'n_legs_stance' in df_valid.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    plot_pca_trajectory(pca_result, df_valid, color_by='n_legs_stance', ax=ax)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'pca_trajectory_nlegs.pdf')
    plt.show()

# ANGLE CHECKING

In [ ]:
### Diagnostic — all bouts: joint angle traces (clipping check), one PNG per bout
### Saved to OUTPUT_DIR/joint_clipping/

import re
from matplotlib.lines import Line2D

# ── Toggle: "full", "main", "core", or None (all) ──────────────────────────────
CLIPPING_JOINT_SET = "core"   # set to None to show every joint

# ── Parse joint names ─────────────────────────────────────────────────────────
LEG_PAT    = re.compile(r"^(.+)_(T[123])_(left|right)$")
leg_joints = []
for i, name in enumerate(names_qpos):
    m = LEG_PAT.match(name)
    if m:
        leg_joints.append({"idx": i, "jtype": m.group(1),
                           "seg": m.group(2), "side": m.group(3)})

_seen = set()
joint_types = [lj["jtype"] for lj in leg_joints
               if not (lj["jtype"] in _seen or _seen.add(lj["jtype"]))]

if CLIPPING_JOINT_SET is not None:
    _keep = set(JOINT_SETS[CLIPPING_JOINT_SET])
    joint_types = [jt for jt in joint_types if jt in _keep]

print(f"Joint types ({len(joint_types)}): {joint_types}")
print(f"Total bouts: {len(bout_keys)}")

segs       = ["T1", "T2", "T3"]
seg_title  = {"T1": "Front (T1)", "T2": "Mid (T2)", "T3": "Hind (T3)"}
side_color = {"left": "#4878CF", "right": "#D65F5F"}
n_rows     = len(joint_types)

_clip_dir = OUTPUT_DIR / "joint_clipping"
_clip_dir.mkdir(exist_ok=True)

for bkey in bout_keys:
    qpos   = np.array(bout_dict[bkey]["qpos"])
    n_fr   = len(qpos)
    t      = np.arange(n_fr) / FPS

    _bout_df = df[df["bout_id"] == bkey].sort_values("frame").reset_index(drop=True)
    _swing   = _bout_df["T1_left_swing_stance"].values if "T1_left_swing_stance" in _bout_df else None

    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 2.2 * n_rows), squeeze=False)

    for ci, seg in enumerate(segs):
        for ri, jtype in enumerate(joint_types):
            ax = axes[ri, ci]

            entries = {lj["side"]: lj["idx"] for lj in leg_joints
                       if lj["seg"] == seg and lj["jtype"] == jtype}
            if not entries:
                ax.axis("off")
                continue

            if _swing is not None and len(_swing) == n_fr:
                in_swing = False
                sw_start = 0
                for fi, sw in enumerate(_swing):
                    if sw == 0 and not in_swing:
                        sw_start = fi / FPS
                        in_swing = True
                    elif sw != 0 and in_swing:
                        ax.axvspan(sw_start, fi / FPS, color="#aaaaaa", alpha=0.18, lw=0)
                        in_swing = False
                if in_swing:
                    ax.axvspan(sw_start, n_fr / FPS, color="#aaaaaa", alpha=0.18, lw=0)

            for side, qi in entries.items():
                ax.plot(t, np.degrees(qpos[:, qi]),
                        color=side_color[side], alpha=0.7, lw=0.7)

            if ri == 0:
                ax.set_title(seg_title[seg], fontsize=10, fontweight="bold")
            if ci == 0:
                ax.set_ylabel(jtype.replace("_", "\n"), fontsize=7)
            if ri == n_rows - 1:
                ax.set_xlabel("Time (s)", fontsize=7)
            ax.tick_params(labelsize=6)

    handles = [Line2D([0], [0], color=c, lw=2, label=s) for s, c in side_color.items()]
    handles.append(Line2D([0], [0], color="#aaaaaa", lw=8, alpha=0.4, label="T1_left swing"))
    axes[0, -1].legend(handles=handles, fontsize=8, loc="upper right")

    fi_bout = bout_keys.index(bkey)
    fig.suptitle(
        f"{bkey}  |  fly={fly_ids[fi_bout]}  |  {n_fr} frames ({n_fr/FPS:.1f} s)"
        "flat lines at constant value = clipping at joint limit",
        fontsize=10, fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(_clip_dir / f"{bkey}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

print(f"Saved {len(bout_keys)} PNGs to {_clip_dir}")

In [ ]:
                                                                                                                                                                                           
key = bout_keys[150
                ]                                                                                                                                                                          
xpos_ego = np.array(bout_dict[key]['xpos_egocentric'])                                                                                                                                      
bout_df   = df[df['bout_id'] == key].sort_values('frame')                                                                                                                                   
T_bout    = min(len(bout_df), xpos_ego.shape[0])                                                                                                                                            
                                                                                                                                                                                            
ph_L = bout_df['T1_left_phase'].values[:T_bout]  if 'T1_left_phase'  in bout_df.columns else None                                                                                           
ph_R = bout_df['T1_right_phase'].values[:T_bout] if 'T1_right_phase' in bout_df.columns else None                                                                                             
                                                                                                                                                                                              
z_L = xpos_ego[:T_bout, 18, 2] * 10   # T1L_TaTip Z                                                                                                                                           
z_R = xpos_ego[:T_bout, 25, 2] * 10   # T1R_TaTip Z                                                                                                                                           
x_L = xpos_ego[:T_bout, 18, 0] * 10                                                                                                                                                           
x_R = xpos_ego[:T_bout, 25, 0] * 10                                                                                                                                                           
                                                                                                                                                                                              
t = np.arange(T_bout)                                                                                                                                                                         
                                                                                                                                                                                              
fig, axes = plt.subplots(2, 2, figsize=(14, 8))                                                                                                                                               
                                                                                                                                                                                              
# Z vs time                                                                                                                                                                                   
axes[0, 0].plot(t, z_L, alpha=0.7, label='T1L Z')                                                                                                                                             
axes[0, 0].plot(t, z_R, alpha=0.7, label='T1R Z', ls='--')                                                                                                                                    
axes[0, 0].set_title('Z ego vs time'); axes[0, 0].legend()                                                                                                                                    
                                                                                                                                                                                              
# Phase vs time                                                                                                                                                                               
axes[0, 1].plot(t, ph_L % (2*np.pi), alpha=0.7, label='T1L phase')                                                                                                                          
axes[0, 1].plot(t, ph_R % (2*np.pi), alpha=0.7, label='T1R phase', ls='--')                                                                                                                   
axes[0, 1].set_title('Phase vs time'); axes[0, 1].legend()                                                                                                                                  
                                                                                                                                                                                              
# Z vs phase (this IS the arc)                                                                                                                                                              
axes[1, 0].scatter(ph_L % (2*np.pi), z_L, s=1, alpha=0.3, c='steelblue', label='T1L')                                                                                                         
axes[1, 0].scatter(ph_R % (2*np.pi), z_R, s=1, alpha=0.3, c='tomato',    label='T1R')                                                                                                         
axes[1, 0].set_xlabel('phase'); axes[1, 0].set_ylabel('Z ego (mm)')                                                                                                                           
axes[1, 0].set_title('Z vs phase — raw pairing'); axes[1, 0].legend(markerscale=5)                                                                                                            
                                                                                                                                                                                              
# X vs phase                                                                                                                                                                                  
axes[1, 1].scatter(ph_L % (2*np.pi), x_L, s=1, alpha=0.3, c='steelblue', label='T1L')                                                                                                         
axes[1, 1].scatter(ph_R % (2*np.pi), x_R, s=1, alpha=0.3, c='tomato',    label='T1R')                                                                                                         
axes[1, 1].set_xlabel('phase'); axes[1, 1].set_ylabel('X ego (mm)')                                                                                                                           
axes[1, 1].set_title('X vs phase — raw pairing'); axes[1, 1].legend(markerscale=5)                                                                                                            
                                                                                                                                                                                              
plt.tight_layout(); plt.show()                                                                                                                                                                
                                                                                                                                                                                              
# Key numbers                                                                                                                                                                                 
print("Phase L range (mod 2pi):", np.nanmin(ph_L % (2*np.pi)), "to", np.nanmax(ph_L % (2*np.pi)))                                                                                             
print("Phase R range (mod 2pi):", np.nanmin(ph_R % (2*np.pi)), "to", np.nanmax(ph_R % (2*np.pi)))                                                                                             
print("Z_L at phase≈0  :", np.nanmean(z_L[np.abs(ph_L % (2*np.pi)) < 0.3]))                                                                                                                   
print("Z_R at phase≈0  :", np.nanmean(z_R[np.abs(ph_R % (2*np.pi)) < 0.3]))                                                                                                                   
print("Z_L at phase≈pi :", np.nanmean(z_L[np.abs((ph_L % (2*np.pi)) - np.pi) < 0.3]))                                                                                                         
print("Z_R at phase≈pi :", np.nanmean(z_R[np.abs((ph_R % (2*np.pi)) - np.pi) < 0.3]))    

In [ ]:
# Cell 27 -- Joint angles vs phase: mean ± std + raw points, per leg pair
 # Layout: n_joints rows × 3 cols (T1, T2, T3)
 # Each panel: left + right legs overlaid, line=mean, shade=std, dots=all frames.                                                                                                              

# ── Colors ────────────────────────────────────────────────────────────────────
FIG2_TIER_COLORS = {'T1': '#d62728', 'T2': '#1f77b4', 'T3': '#2ca02c'}
FIG2_LEG_COLORS  = {
    'T1_left':  '#d62728', 'T1_right': '#ff7f0e',
    'T2_left':  '#1f77b4', 'T2_right': '#9467bd',
    'T3_left':  '#2ca02c', 'T3_right': '#8c564b',
}
                                                                                                                                                                                             
_N_BINS_P = 24
_JOINTS_P = JOINT_SETS['main']
_PAIRS_P  = [('T1', 'T1_left', 'T1_right'),
             ('T2', 'T2_left', 'T2_right'),
             ('T3', 'T3_left', 'T3_right')]
_PH_BINS_P = np.linspace(-np.pi, np.pi, _N_BINS_P + 1)
_PH_CTRS_P = (_PH_BINS_P[:-1] + _PH_BINS_P[1:]) / 2
fig_p, axes_p = plt.subplots(
    len(_JOINTS_P), len(_PAIRS_P),
    figsize=(10, 2.2 * len(_JOINTS_P)),
    sharex=True,
)
for ci, (tier, leg_L, leg_R) in enumerate(_PAIRS_P):
    for ri, joint in enumerate(_JOINTS_P):
        ax = axes_p[ri, ci]
        for leg, clr in [(leg_L, FIG2_LEG_COLORS[leg_L]),
                         (leg_R, FIG2_LEG_COLORS[leg_R])]:
            col   = f"{leg}_{joint}"
            ph_col = f"{leg}_phase"
            if col not in df_valid.columns or ph_col not in df_valid.columns:
                continue
            valid  = df_valid[[col, ph_col]].dropna()
            angles = np.degrees(valid[col].values.astype(float))
            phases = ((valid[ph_col].values.astype(float) + np.pi) % (2 * np.pi)) - np.pi
            # Raw points
            ax.scatter(phases, angles, s=1, color=clr, alpha=0.15,
                       linewidths=0, rasterized=True, zorder=1)
            # Binned mean ± std
            bi    = np.clip(np.digitize(phases, _PH_BINS_P) - 1, 0, _N_BINS_P - 1)
            means = np.array([angles[bi == k].mean() if np.any(bi == k) else np.nan
                              for k in range(_N_BINS_P)])
            stds  = np.array([angles[bi == k].std()  if np.any(bi == k) else np.nan
                              for k in range(_N_BINS_P)])
            ax.fill_between(_PH_CTRS_P, means - stds, means + stds,
                            color=clr, alpha=0.25, linewidth=0, zorder=2)
            ax.plot(_PH_CTRS_P, means, color=clr, lw=1.5, zorder=3)
        ax.axvline(0, color='#ccc', lw=0.6, zorder=0)
        ax.set_xlim(-np.pi, np.pi)
        ax.tick_params(labelsize=6)
        sns.despine(ax=ax)
        if ri == 0:
            ax.set_title(tier, fontsize=10, fontweight='bold')
        if ci == 0:
            ax.set_ylabel(JOINT_LABELS.get(joint, joint) + '\n(°)', fontsize=7)
        if ri == len(_JOINTS_P) - 1:
            ax.set_xlabel('Phase (rad)', fontsize=7)
# Legend
_lh_p = [plt.Line2D([0], [0], color=FIG2_LEG_COLORS[f'{t}_left'],  lw=2, label=f'{t} L')
         for t, _, _ in _PAIRS_P] + \
        [plt.Line2D([0], [0], color=FIG2_LEG_COLORS[f'{t}_right'], lw=2, label=f'{t} R')
         for t, _, _ in _PAIRS_P]
fig_p.legend(handles=_lh_p, fontsize=7, frameon=False, ncol=6,
             loc='upper center', bbox_to_anchor=(0.5, 1.02))
fig_p.suptitle('Joint angles vs phase  ·  mean ± std + all points  ·  per leg pair',
               fontsize=10, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'joint_angles_vs_phase_points.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Cell: Joint Angle Distributions per Joint per Contralateral Leg Group
# Layout: n_joints rows × 3 cols (T1, T2, T3)
# Each panel: overlaid histograms for left + right legs.
# Vertical lines per leg (same color):
#   --   liftoff    (phase = -π/2)
#   :    landing    (phase = +π/2)
#   -.   mid-swing  (phase ≈  0,  peak speed)
#   —    mid-stance (phase ≈ ±π, speed trough)

plt.rcParams['pdf.fonttype'] = 42   # TrueType fonts → editable text in Illustrator

_PAIRS_HIST  = [('T1', 'T1_left', 'T1_right'),
                ('T2', 'T2_left', 'T2_right'),
                ('T3', 'T3_left', 'T3_right')]
_JOINTS_HIST = JOINT_SETS['main']   # 7 joints
_N_BINS_HIST = 60
_PHASE_WINDOW = np.pi / 8   # ±22.5° window for mid-swing / mid-stance sampling

fig_hist, axes_hist = plt.subplots(
    len(_JOINTS_HIST), len(_PAIRS_HIST),
    figsize=(9, 2.5 * len(_JOINTS_HIST)),
    sharey='row',
)

for ci, (tier, leg_L, leg_R) in enumerate(_PAIRS_HIST):
    for ri, joint in enumerate(_JOINTS_HIST):
        ax = axes_hist[ri, ci]
        for leg, clr in [(leg_L, FIG2_LEG_COLORS[leg_L]),
                         (leg_R, FIG2_LEG_COLORS[leg_R])]:
            col    = f"{leg}_{joint}"
            ph_col = f"{leg}_phase"
            if col not in df_valid.columns or ph_col not in df_valid.columns:
                continue
            valid  = df_valid[[col, ph_col, 'bout_id']].dropna(subset=[col, ph_col])
            angles = np.degrees(valid[col].values.astype(float))

            ax.hist(angles, bins=_N_BINS_HIST, color=clr, alpha=0.55,
                    density=True, linewidth=0,
                    label=LEG_LABELS.get(leg, leg))

            # Collect event angles per bout
            lo_ang, td_ang, ms_ang, mst_ang = [], [], [], []
            for bid, grp in valid.groupby('bout_id', sort=False):
                ph  = grp[ph_col].values
                ang = np.degrees(grp[col].values.astype(float))

                lo = liftoff_from_hilbert(ph);  lo = lo[lo < len(ang)]
                td = landing_from_hilbert(ph);  td = td[td < len(ang)]
                if len(lo): lo_ang.extend(ang[lo])
                if len(td): td_ang.extend(ang[td])

                ms_mask = np.abs(ph) < _PHASE_WINDOW
                if ms_mask.any(): ms_ang.extend(ang[ms_mask])

                mst_mask = np.abs(ph) > np.pi - _PHASE_WINDOW
                if mst_mask.any(): mst_ang.extend(ang[mst_mask])

            if lo_ang:  ax.axvline(np.nanmean(lo_ang),  color=clr, lw=1.5, ls='--', alpha=0.9)
            if td_ang:  ax.axvline(np.nanmean(td_ang),  color=clr, lw=1.5, ls=':',  alpha=0.9)
            if ms_ang:  ax.axvline(np.nanmean(ms_ang),  color=clr, lw=1.5, ls='-.', alpha=0.9)
            if mst_ang: ax.axvline(np.nanmean(mst_ang), color=clr, lw=1.0, ls='-',  alpha=0.9)

        sns.despine(ax=ax)
        ax.tick_params(labelsize=6)
        if ri == 0:
            ax.set_title(tier, fontsize=10, fontweight='bold')
        if ci == 0:
            # joint name as row label; y-axis value is density
            ax.set_ylabel('Density', fontsize=7)
            ax.text(-0.32, 0.5, JOINT_LABELS.get(joint, joint),
                    transform=ax.transAxes, fontsize=8, fontweight='bold',
                    va='center', ha='right', rotation=90)
        if ri == len(_JOINTS_HIST) - 1:
            ax.set_xlabel('Angle (°)', fontsize=7)
        if ri == 0 and ci == 0:
            ax.legend(fontsize=5, loc='upper right')

# Legend for line styles (top-right panel)
_ax_leg = axes_hist[0, -1]
for ls, lbl in [('--', 'liftoff'), (':', 'landing'), ('-.', 'mid-swing'), ('-', 'mid-stance')]:
    _ax_leg.plot([], [], color='k', lw=1.5, ls=ls, label=lbl)
_ax_leg.legend(fontsize=5, loc='upper right', frameon=False)

fig_hist.suptitle('Joint Angle Distributions by Tier',
                  fontsize=11, y=1.01)
fig_hist.tight_layout()
_out = OUTPUT_DIR / 'joint_angle_distributions_by_tier.pdf'
fig_hist.savefig(_out, bbox_inches='tight')
plt.show()
print(f"Saved: {_out}")

In [ ]:
# Cell: Joint Angle Distributions — Publication Figure
# 22 mm × 60 mm, Arial 6 pt
# Color by joint type (flex=#856500, abduct=#d67116, rot=#d62728)
# Opacity by tier: T1=1.0, T2=0.75, T3=0.50
# Histograms: per-bout density averaged across bouts (left+right pooled)
# Vertical lines: averaged phase-event angles (liftoff --, landing :, mid-swing -., mid-stance -)

import matplotlib as _mpl
import matplotlib.font_manager as _fm
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold.ttf")
_rc_orig = {k: _mpl.rcParams[k] for k in ('font.family', 'font.size',
                                            'pdf.fonttype', 'ps.fonttype')}
_mpl.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                      'pdf.fonttype': 42, 'ps.fonttype': 42})

_JOINTS_PUB  = JOINT_SETS['main']          # 7 joints
_PAIRS_PUB   = [('T1', 'T1_left', 'T1_right'),
                ('T2', 'T2_left', 'T2_right'),
                ('T3', 'T3_left', 'T3_right')]
_N_BINS_PUB  = 60
_PHASE_WIN   = np.pi / 8
SHARED_X     = True   # True = all panels share global angle range

_JTYPE_COLOR = {'flex': '#856500', 'abduct': '#d67116', 'rot': '#d62728'}
_TIER_ALPHA  = {'T1': 1.0, 'T2': 0.50, 'T3': 0.25}

def _jcat(jname):
    if 'abduct' in jname: return 'abduct'
    if 'twist'  in jname: return 'rot'
    return 'flex'

# Precompute global x range if needed
_global_xlo, _global_xhi = np.inf, -np.inf
if SHARED_X:
    for _tier, _lL, _lR in _PAIRS_PUB:
        for _jnt in _JOINTS_PUB:
            for _col in [f"{_lL}_{_jnt}", f"{_lR}_{_jnt}"]:
                if _col in df_mc.columns:
                    _v = np.degrees(df_mc[_col].dropna().values.astype(float))
                    if len(_v):
                        _global_xlo = min(_global_xlo, np.percentile(_v, 1))
                        _global_xhi = max(_global_xhi, np.percentile(_v, 99))

fig_pub, axes_pub = plt.subplots(
    len(_JOINTS_PUB), 3,
    figsize=(32 / 25.4, 60 / 25.4),
    sharey=True,
)

for ci, (tier, leg_L, leg_R) in enumerate(_PAIRS_PUB):
    alpha = _TIER_ALPHA[tier]

    for ri, joint in enumerate(_JOINTS_PUB):
        ax    = axes_pub[ri, ci]
        color = _JTYPE_COLOR[_jcat(joint)]

        col_L  = f"{leg_L}_{joint}"
        col_R  = f"{leg_R}_{joint}"
        ph_L   = f"{leg_L}_phase"
        ph_R   = f"{leg_R}_phase"

        # collect all angle values to fix histogram edges
        all_ang = []
        for col in (col_L, col_R):
            if col in df_mc.columns:
                all_ang.extend(np.degrees(df_mc[col].dropna().values.astype(float)))
        if not all_ang:
            ax.axis('off')
            continue

        _edges   = np.linspace(np.nanmin(all_ang), np.nanmax(all_ang), _N_BINS_PUB + 1)
        _centers = (_edges[:-1] + _edges[1:]) / 2

        # per-bout averaged histogram (pool left+right)
        _densities = []
        for bid, grp in df_mc.groupby('bout_id', sort=False):
            _bout_ang = []
            for col in (col_L, col_R):
                if col in grp.columns:
                    _v = grp[col].dropna().values.astype(float)
                    if len(_v): _bout_ang.extend(np.degrees(_v))
            if len(_bout_ang) < 2: continue
            _d, _ = np.histogram(_bout_ang, bins=_edges, density=True)
            _densities.append(_d)

        if not _densities:
            ax.axis('off')
            continue

        _mean_d = np.nanmean(np.vstack(_densities), axis=0)
        import matplotlib.ticker as _ticker
        _p5, _p50, _p95 = np.percentile(all_ang, [5, 50, 95])
        _hw = max(_p50 - _p5, _p95 - _p50)
        _xlo = _global_xlo if SHARED_X else _p50 - _hw * 1.2
        _xhi = _global_xhi if SHARED_X else _p50 + _hw * 1.2
        ax.set_xlim(_xlo, _xhi)
        _cmask = (_centers >= _xlo) & (_centers <= _xhi)
        ax.fill_between(_centers[_cmask], _mean_d[_cmask], alpha=alpha, color=color, lw=0, step=None)

        _r5  = int(round(_p5  / 5) * 5)
        _r95 = int(round(_p95 / 5) * 5)
        ax.set_xticks([_r5, _r95])
        ax.set_xticklabels([f'{_r5}°', f'{_r95}°'])
        ax.set_yticks([0, 0.06])
        ax.set_ylim(0, max(_mean_d.max() * 1.75, 0.068))
        ax.tick_params(labelsize=5, pad=2, length=2)
        import seaborn as _sns
        _sns.despine(ax=ax, left=(ci != 0))

        if ri == 0:
            ax.set_title(tier, fontsize=6, fontweight='bold', pad=2)
        if ci == 0:
            _jlabel = JOINT_LABELS.get(joint, joint)
            ax.set_ylabel('', fontsize=0)   # suppress default
            ax.text(-0.55, 0.7, _jlabel,
                    transform=ax.transAxes, fontsize=5,
                    va='center', ha='right', rotation=90)
        if ri == len(_JOINTS_PUB) - 1 and ci == 1:
            ax.set_xlabel('Angle (°)', fontsize=5, labelpad=2)

plt.subplots_adjust(left=0.32, right=0.99, top=0.93, bottom=0.07, hspace=0.55, wspace=0.35)
_out_pub = OUTPUT_DIR / 'joint_angle_distributions_paper.pdf'
fig_pub.savefig(_out_pub, bbox_inches='tight')
plt.show()
print(f'Saved: {_out_pub}')
print(f'Size: {np.array(fig_pub.get_size_inches()) * 25.4} mm')

# restore rcParams
_mpl.rcParams.update(_rc_orig)


In [ ]:
# Cell: Joint Angle Distributions by Leg Group — Publication Figure
# Layout: 7 joints × 1 column; all 3 leg groups (T1/T2/T3) overlaid per panel
# Angles: ORIGINAL (not mean-centered) — uses df_valid
# Color by joint type; opacity by tier (T1=1.0, T2=0.5, T3=0.25)
# Toggle: SHARED_X = True/False

import matplotlib as _mpl
import matplotlib.font_manager as _fm
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold.ttf")
_rc_orig2 = {k: _mpl.rcParams[k] for k in ('font.family', 'font.size',
                                              'pdf.fonttype', 'ps.fonttype')}
_mpl.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                      'pdf.fonttype': 42, 'ps.fonttype': 42})

# ── Config ────────────────────────────────────────────────────────────────────
SHARED_X2    = True    # True = all panels share one x range; False = per-joint range
_N_BINS2     = 60
_JOINTS2     = JOINT_SETS['main']   # 7 joints
_PAIRS2      = [('T1', 'T1_left', 'T1_right'),
                ('T2', 'T2_left', 'T2_right'),
                ('T3', 'T3_left', 'T3_right')]
_JTYPE_COLOR2 = {'flex': '#856500', 'abduct': '#d67116', 'rot': '#d62728'}
_TIER_ALPHA2  = {'T1': 1.0, 'T2': 0.50, 'T3': 0.25}
_XPAD2       = 0.05   # fractional padding on each side of x range

def _jcat2(jname):
    if 'abduct' in jname: return 'abduct'
    if 'twist'  in jname: return 'rot'
    return 'flex'

# ── Global x range (across all joints and tiers) ─────────────────────────────
_gxlo2, _gxhi2 = np.inf, -np.inf
for _tier2, _lL2, _lR2 in _PAIRS2:
    for _jnt2 in _JOINTS2:
        for _col2 in [f"{_lL2}_{_jnt2}", f"{_lR2}_{_jnt2}"]:
            if _col2 in df_valid.columns:
                _v2 = np.degrees(df_valid[_col2].dropna().values.astype(float))
                if len(_v2):
                    _gxlo2 = min(_gxlo2, np.percentile(_v2, 1))
                    _gxhi2 = max(_gxhi2, np.percentile(_v2, 99))

# ── Figure ────────────────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(
    len(_JOINTS2), 1,
    figsize=(22 / 25.4, 60 / 25.4),
    sharey=True,
    sharex=SHARED_X2,
)

_ymax_global2 = 0.0

for ri2, joint2 in enumerate(_JOINTS2):
    ax2   = axes2[ri2]
    color2 = _JTYPE_COLOR2[_jcat2(joint2)]

    # per-tier overlay
    for tier2, leg_L2, leg_R2 in _PAIRS2:
        col_L2 = f"{leg_L2}_{joint2}"
        col_R2 = f"{leg_R2}_{joint2}"

        all_ang2 = []
        for col2 in (col_L2, col_R2):
            if col2 in df_valid.columns:
                all_ang2.extend(np.degrees(df_valid[col2].dropna().values.astype(float)))
        if not all_ang2:
            continue

        _edges2   = np.linspace(np.nanmin(all_ang2), np.nanmax(all_ang2), _N_BINS2 + 1)
        _centers2 = (_edges2[:-1] + _edges2[1:]) / 2

        # per-bout averaged histogram
        _dens2 = []
        for _bid2, _grp2 in df_valid.groupby('bout_id', sort=False):
            _ba2 = []
            for col2 in (col_L2, col_R2):
                if col2 in _grp2.columns:
                    _vv2 = _grp2[col2].dropna().values.astype(float)
                    if len(_vv2): _ba2.extend(np.degrees(_vv2))
            if len(_ba2) < 2: continue
            _d2, _ = np.histogram(_ba2, bins=_edges2, density=True)
            _dens2.append(_d2)

        if not _dens2:
            continue

        _mean_d2 = np.nanmean(np.vstack(_dens2), axis=0)
        _ymax_global2 = max(_ymax_global2, _mean_d2.max())

        # x limits for this panel
        _p5_2, _p50_2, _p95_2 = np.percentile(all_ang2, [5, 50, 95])
        if SHARED_X2:
            _xlo2, _xhi2 = _gxlo2, _gxhi2
        else:
            _hw2 = max(_p50_2 - _p5_2, _p95_2 - _p50_2)
            _xlo2 = _p50_2 - _hw2 * 1.2
            _xhi2 = _p50_2 + _hw2 * 1.2

        _cmask2 = (_centers2 >= _xlo2) & (_centers2 <= _xhi2)
        ax2.fill_between(_centers2[_cmask2], _mean_d2[_cmask2],
                         alpha=_TIER_ALPHA2[tier2], color=color2, lw=0)

    # x limits (set once per panel when not shared)
    if not SHARED_X2:
        all_ang_joint = []
        for _, _lL2, _lR2 in _PAIRS2:
            for _c2 in [f"{_lL2}_{joint2}", f"{_lR2}_{joint2}"]:
                if _c2 in df_valid.columns:
                    all_ang_joint.extend(
                        np.degrees(df_valid[_c2].dropna().values.astype(float)))
        if all_ang_joint:
            _p5j, _p50j, _p95j = np.percentile(all_ang_joint, [5, 50, 95])
            _hwj = max(_p50j - _p5j, _p95j - _p50j)
            _xpad2j = _hwj * 1.2 * _XPAD2
            ax2.set_xlim(_p50j - _hwj * 1.2 - _xpad2j, _p50j + _hwj * 1.2 + _xpad2j)
    else:
        _xpad2 = (_gxhi2 - _gxlo2) * _XPAD2
        ax2.set_xlim(_gxlo2 - _xpad2, _gxhi2 + _xpad2)

    ax2.xaxis.set_major_locator(
        _mpl.ticker.MaxNLocator(nbins=4, integer=True, prune='both'))
    ax2.xaxis.set_major_formatter(
        _mpl.ticker.FuncFormatter(lambda x, _: f'{int(x)}°'))
    ax2.tick_params(labelsize=5, pad=2, length=2)

    import seaborn as _sns2
    _sns2.despine(ax=ax2, left=False)

    _jlabel2 = JOINT_LABELS.get(joint2, joint2)
    ax2.set_ylabel('', fontsize=0)
    ax2.text(-0.55, 0.7, _jlabel2,
             transform=ax2.transAxes, fontsize=5,
             va='center', ha='right', rotation=90)

    if ri2 == len(_JOINTS2) - 1:
        ax2.set_xlabel('Angle (°)', fontsize=5, labelpad=2)

# shared y: ticks at 0 and 0.06 but don't clip data above
_ylim2 = max(_ymax_global2 * 1.15, 0.062)
for ax2 in axes2:
    ax2.set_ylim(0, _ylim2)
    ax2.set_yticks([0, 0.06])

# tier legend
from matplotlib.patches import Patch as _Patch
axes2[0].legend(
    handles=[_Patch(facecolor='grey', alpha=_TIER_ALPHA2[t], label=t)
             for t in ('T1', 'T2', 'T3')],
    frameon=False, fontsize=4.5, loc='upper right',
    handlelength=0.8, handleheight=0.8, borderpad=0.2, labelspacing=0.2)

plt.subplots_adjust(left=0.32, right=0.99, top=0.97, bottom=0.07, hspace=0.55)
_out2 = OUTPUT_DIR / 'joint_angle_distributions_by_leggroup.pdf'
fig2.savefig(_out2, bbox_inches='tight')
plt.show()
print(f'Saved: {_out2}')

_mpl.rcParams.update(_rc_orig2)


In [ ]:
# Cell 28 -- All Joints vs Phase Grid
# ALL JOINTS vs PHASE for ALL LEGS (7 joints × 6 legs grid)
# This shows how each joint angle varies with step phase across all legs

MAIN_JOINTS = ['coxa_abduction', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia', 'tarsus']
n_joints = len(MAIN_JOINTS)
n_legs = len(LEGS)
n_bins = 24

fig, axes = plt.subplots(n_joints, n_legs, figsize=(16, 18), sharex=True)

for i, joint in enumerate(MAIN_JOINTS):
    for j, leg in enumerate(LEGS):
        ax = axes[i, j]
        
        col = f"{leg}_{joint}"
        phase_col = f"{leg}_phase"
        
        if col not in df_mc.columns or phase_col not in df_mc.columns:
            ax.set_visible(False)
            continue
        
        # Get valid data
        valid = df_mc[[col, phase_col]].dropna()
        angles = valid[col].values
        phases = valid[phase_col].values
        
        if len(angles) < 10:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue
        
        # Bin by phase
        phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
        bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
        bin_indices = np.digitize(phases, phase_bins) - 1
        bin_indices = np.clip(bin_indices, 0, n_bins - 1)
        
        # Compute mean and std per bin
        bin_means = np.array([angles[bin_indices == k].mean() if np.any(bin_indices == k) else np.nan 
                             for k in range(n_bins)])
        bin_stds = np.array([angles[bin_indices == k].std() if np.any(bin_indices == k) else np.nan 
                            for k in range(n_bins)])
        
        # Plot
        ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.3, color='blue')
        ax.plot(bin_centers, bin_means, 'b-', lw=1.5)
        ax.axvline(0, color='r', linestyle='--', alpha=0.5, lw=0.5)
        
        ax.set_xlim(-np.pi, np.pi)
        
        # Labels
        if i == n_joints - 1:
            ax.set_xlabel(LEG_LABELS.get(leg, leg))
        if j == 0:
            ax.set_ylabel(JOINT_LABELS.get(joint, joint), fontsize=9)
        
        # Column titles (leg names) only on top row
        if i == 0:
            ax.set_title(LEG_LABELS.get(leg, leg), fontsize=10)

plt.suptitle('All Joints vs Step Phase (mean ± std)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'all_joints_vs_phase_full.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'all_joints_vs_phase_full.pdf'}")

---

## 9. Optional: UMAP

In [ ]:
# Cell 29 -- GPU Library Imports

# Build colour variable: mean absolute joint angular velocity, Gaussian-smoothed
# Uses the _d1 (first-derivative) columns already added during preprocessing
d1_cols = [c for c in df_valid.columns if c.endswith('_d1')]
mean_abs_vel = df_valid[d1_cols].abs().mean(axis=1).values
vel_smooth = gaussian_filter1d(mean_abs_vel, sigma=10)  # ~12 ms window at 800 Hz

# Scale feature matrix (fresh scaler — independent of the PCA scaler)
scaler_umap = StandardScaler()
X_scaled = scaler_umap.fit_transform(X)

# GPU-accelerated UMAP (cuML / RAPIDS)
reducer_cuml = cuml.manifold.UMAP(
    n_components=3, n_neighbors=15, min_dist=0.5,
    metric='euclidean', random_state=70
)
umap_result = reducer_cuml.fit_transform(X_scaled)
print(f'UMAP embedding: {umap_result.shape}')


In [ ]:
# Cell 30 -- 3-D UMAP
# ── Colour toggle ─────────────────────────────────────────────────────────────
COLOR_BY      = 'phase'  # 'instant_vel' | 'cycle_speed' | 'n_legs_stance'
                                  # 'phase_rel'   | 'phase'
PHASE_REL_LEG = 'T1_right'       # used when COLOR_BY == 'phase_rel'
PHASE_LEG     = 'T1_left'        # used when COLOR_BY == 'phase'
                                  # any of: 'T1_left','T1_right','T2_left','T2_right','T3_left','T3_right'

if COLOR_BY == 'cycle_speed':
    _cvals  = df_valid['step_cycle_mean_speed'].values.astype(float)
    _clabel = 'Step-cycle mean speed (mm/s)'
    _ctitle = 'UMAP of Leg Joint Angles — colored by step-cycle mean speed'
    _cfname = 'umap_3d_cycle_speed.pdf'
    _cmap   = 'turbo'
    _vmin, _vmax = float(np.nanmin(_cvals)), float(np.nanmax(_cvals))
elif COLOR_BY == 'n_legs_stance':
    _cvals  = df_valid['n_legs_stance'].values.astype(float)
    _clabel = 'N legs in stance'
    _ctitle = 'UMAP of Leg Joint Angles — colored by N legs in stance'
    _cfname = 'umap_3d_n_legs_stance.pdf'
    _cmap   = 'RdYlGn'
    _vmin, _vmax = 0, 6
elif COLOR_BY == 'phase_rel':
    _phase_col = f"{PHASE_REL_LEG}_phase_rel"
    assert _phase_col in df_valid.columns, f"{_phase_col} not found — run Cell 11 first"
    _cvals  = df_valid[_phase_col].values.astype(float)
    _clabel = f'{PHASE_REL_LEG} phase relative to L1 (rad)'
    _ctitle = f'UMAP of Leg Joint Angles — colored by {PHASE_REL_LEG} phase rel. to L1'
    _cfname = f'umap_3d_phase_rel_{PHASE_REL_LEG}.pdf'
    _cmap   = 'twilight'
    _vmin, _vmax = -np.pi, np.pi
elif COLOR_BY == 'phase':
    _phase_col = f"{PHASE_LEG}_phase"
    assert _phase_col in df_valid.columns, f"{_phase_col} not found — run Cell 11 first"
    _cvals  = df_valid[_phase_col].values.astype(float)
    _clabel = f'{PHASE_LEG} step-cycle phase (rad)'
    _ctitle = f'UMAP of Leg Joint Angles — colored by {PHASE_LEG} phase'
    _cfname = f'umap_3d_phase_{PHASE_LEG}.pdf'
    _cmap   = 'twilight'
    _vmin, _vmax = -np.pi, np.pi
else:
    _cvals  = vel_smooth
    _clabel = 'Mean |joint velocity| (smoothed)'
    _ctitle = 'UMAP of Leg Joint Angles — colored by mean |angular velocity|'
    _cfname = 'umap_3d_velocity.pdf'
    _cmap   = 'turbo'
    _vmin, _vmax = float(np.nanmin(_cvals)), float(np.nanmax(_cvals))

fig = plt.figure(figsize=(20, 25))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    umap_result[:, 0], umap_result[:, 1], umap_result[:, 2],
    c=_cvals, cmap=_cmap, s=3, alpha=0.5, vmin=_vmin, vmax=_vmax
)
cb = plt.colorbar(sc, label=_clabel)
if COLOR_BY == 'n_legs_stance':
    cb.set_ticks(range(7))
elif COLOR_BY in ('phase_rel', 'phase'):
    cb.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    cb.set_ticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_zlabel('UMAP 3')
ax.set_title(_ctitle)
ax.view_init(elev=20, azim=70)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / _cfname)
plt.show()


In [ ]:
# Cell 31 -- Fly ID Integer Encoding
# Step 1: convert fly_id strings → integer codes
fly_ids   = df_valid['fly_id'].values
codes, categories = pd.factorize(fly_ids)   # codes: array of ints, categories: unique labels
# Step 2: pick a discrete colormap with enough colours
n_flies = len(categories)
cmap    = plt.cm.get_cmap('tab10', n_flies)   # tab10 has 10 distinct colours
fig = plt.figure(figsize=(20, 25))
ax  = fig.add_subplot(111, projection='3d')
sc  = ax.scatter(
    umap_result[:, 0], umap_result[:, 1], umap_result[:, 2],
    c=codes,            # <-- integer codes, not strings
    cmap=cmap,
    vmin=-0.5, vmax=n_flies - 0.5,   # centres each colour band on its integer
    s=3, alpha=0.5
)
# Step 3: legend (not colorbar) — shows actual fly_id labels
handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=cmap(i), markersize=8, label=str(cat))
    for i, cat in enumerate(categories)
]
ax.legend(handles=handles, title='Fly ID', loc='upper left',
          fontsize=9, framealpha=0.7)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_zlabel('UMAP 3')
ax.set_title('UMAP of Leg Joint Angles — colored by Fly ID')
ax.view_init(elev=0, azim=100)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_3d_fly_id.pdf')
plt.show()

In [ ]:
# Cell 32 -- UMAP Color Variables
color_vars = [
    ('n_legs_stance', 'N legs stance',     'viridis'),
    ('forward_speed', 'Forward speed (mm/s)', 'turbo'),
    ('bout_idx',      'Bout',               'tab20'),
    ('fly_id',        'Fly ID',              'tab10'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (col, label, cmap) in zip(axes, color_vars):
    if col not in df_valid.columns:
        ax.axis('off')
        continue
    c = df_valid[col]
    if c.dtype == object or str(c.dtype) == 'category':
        codes = pd.Categorical(c).codes
        sc = ax.scatter(umap_result[:, 1], umap_result[:, 2],
                        c=codes, cmap=cmap, s=3, alpha=0.4)
    else:
        sc = ax.scatter(umap_result[:, 1], umap_result[:, 2],
                        c=c, cmap=cmap, s=3, alpha=0.4)
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'UMAP — {label}')
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_2d_panels.pdf')
plt.show()


In [ ]:
# Cell 33 -- Multi-Bout UMAP Plot
n_bouts_show = min(189, len(bout_keys))
n_cols = 5
n_rows = (n_bouts_show + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()

frame_start = 0
for i, key in enumerate(bout_keys[:n_bouts_show]):
    n_frames = int((df_valid['bout_id'] == key).sum())
    frame_end = frame_start + n_frames
    axes[i].scatter(
        umap_result[frame_start:frame_end, 0],
        umap_result[frame_start:frame_end, 1],
        c=np.arange(n_frames), cmap='turbo', s=5, alpha=0.7
    )
    axes[i].set_title(key)
    axes[i].set_xlabel('UMAP 1')
    axes[i].set_ylabel('UMAP 2')
    sns.despine(ax=axes[i])
    frame_start = frame_end

for j in range(n_bouts_show, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_per_bout.pdf')
plt.show()


---

## Section 5 — Per-fly Summaries

Aggregate metrics per fly: mean forward speed, onset-based coordination R, and plots.


In [ ]:
# Cell 34 -- Per-Fly Summaries
# ─── Per-fly summaries ────────────────────────────────────────────────────
_fly_ids_s = sorted(df['fly_id'].unique())
_fly_pal_s = plt.cm.get_cmap('tab10', max(len(_fly_ids_s), 1))
_fly_clr_s = {fid: _fly_pal_s(i) for i, fid in enumerate(_fly_ids_s)}

# Compute per-fly stats
_summary = []
for _fid in _fly_ids_s:
    _mask = df['fly_id'] == _fid
    _mean_spd = df.loc[_mask, 'forward_speed'].mean()

    _Rs = []
    if 'fly_phases' in dir():
        for _leg in non_ref_legs:
            _R = mean_resultant_length(fly_phases[_fid][_leg])
            if not np.isnan(_R):
                _Rs.append(_R)
    _summary.append({'fly_id': _fid,
                     'mean_speed': _mean_spd,
                     'mean_R':     np.nanmean(_Rs) if _Rs else np.nan})

_df_summ = pd.DataFrame(_summary)
print(_df_summ.to_string(index=False))

fig_s, axes_s = plt.subplots(1, 3, figsize=(16, 5))

# 1. Strip plot: R per fly, one point per leg pair
_ax = axes_s[0]
if 'fly_phases' in dir():
    for _j, _leg in enumerate(non_ref_legs):
        for _i, _fid in enumerate(_fly_ids_s):
            _R = mean_resultant_length(fly_phases[_fid][_leg])
            if not np.isnan(_R):
                _ax.scatter(_i + _j * 0.1, _R, color=_fly_clr_s[_fid], s=80, alpha=0.85, zorder=3)
_ax.set_xticks(range(len(_fly_ids_s)))
_ax.set_xticklabels([str(f)[:10] for f in _fly_ids_s], rotation=30, ha='right', fontsize=8)
_ax.set_ylabel('R (coordination strength)')
_ax.set_title('Onset-based R per fly\n(each point = one leg pair)')
_ax.set_ylim(0, 1.05)

# 2. Scatter: mean R vs mean speed
_ax = axes_s[1]
for _, _row in _df_summ.iterrows():
    _ax.scatter(_row['mean_speed'], _row['mean_R'],
                color=_fly_clr_s[_row['fly_id']], s=120, zorder=3,
                label=str(_row['fly_id'])[:12])
_ax.set_xlabel('Mean forward speed (mm/s)')
_ax.set_ylabel('Mean coordination R')
_ax.set_title('Coordination vs speed')
_ax.set_ylim(0, 1.05)
_ax.legend(fontsize=7, framealpha=0.8)

# 3. Mean R per leg pair (bar chart)
_ax = axes_s[2]
if 'R_matrix' in dir() and 'non_ref_legs' in dir():
    _R_bar  = np.nanmean(R_matrix, axis=0)
    _col_lb = [LEG_LABELS.get(l, l) for l in non_ref_legs]
    _ax.bar(_col_lb, _R_bar, color=[_fly_pal_s(0.5)] * len(non_ref_legs), edgecolor='k')
    _ax.set_ylim(0, 1.05)
    for _ii, _r in enumerate(_R_bar):
        if not np.isnan(_r):
            _ax.text(_ii, _r + 0.02, f'{_r:.2f}', ha='center', va='bottom', fontsize=10)
_ax.set_ylabel('Mean R across flies')
_ax.set_title(f'Grand mean R vs {REFERENCE_LEG}')

fig_s.suptitle('Per-fly Summary Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_fly_summaries.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'per_fly_summaries.pdf'}")


---

## 11. Paper Figures — PCA Phase Portrait, Speed-Binned Trajectory, Range of Motion

Reproduces three figures from the original reference analysis:
- **PCA phase portrait**: 2×3 LineCollection grid, each panel colored by that leg's step phase relative to L1
- **Speed-binned PCA trajectory**: phase-averaged mean trajectory per speed bin, colored by n_legs_stance
- **Range of motion**: bar chart per joint + per-fly strip/point plots by leg tier


In [ ]:
# Cell 35 -- Paper Figure: Speed-Binned PCA
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

# ── Phase display mode ──────────────────────────────────────────────────────────
# 'signed'  : [-π, π]  shows whether each leg LEADS (+) or LAGS (-) L1
#             +π/2 = leg is quarter-cycle ahead of L1
#             -π/2 = leg is quarter-cycle behind L1
#             ±π   = antiphase (half-cycle offset, same physical state)
# 'absolute': [0, π]   shows magnitude of offset only (in-phase vs antiphase)
#             0 = moves together with L1
#             π = alternates with L1
PHASE_MODE = 'absolute'   # change to 'absolute' to collapse lead/lag information

if PHASE_MODE == 'absolute':
    # PHASE_CMAP = 'RdYlGn_r'          # green = in phase, red = antiphase
    PHASE_CMAP = 'plasma'          # green = in phase, red = antiphase  ########## Modified by Elliott
    _norm      = Normalize(0, np.pi)
    _cb_ticks  = [0, np.pi / 2, np.pi]
    _cb_labels = ['0\n(in phase)', r'$\pi/2$', r'$\pi$' + '\n(antiphase)']
    _cb_title  = '|Phase offset from L1|'
else:  # signed
    PHASE_CMAP = 'twilight'               # circular: red at ±π (antiphase), green at 0
    _norm      = Normalize(-np.pi, np.pi)
    _cb_ticks  = [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi]
    _cb_labels = [r'$-\pi$' + '\n(antiphase)', r'$-\pi/2$' + '\n(leg lags L1)',
                  '0\n(in phase)', r'$\pi/2$' + '\n(leg leads L1)',
                  r'$\pi$' + '\n(antiphase)']
    _cb_title  = 'Phase rel. to L1  (+ = leads, − = lags)'

# ── Activity mask ───────────────────────────────────────────────────────────────
# True  → keep only top 60% of frames by mean joint angular velocity
#          (removes slow/stationary frames but also discards slow locomotion)
# False → keep all frames with valid speed data (recommended for full comparison)
USE_ACTIVITY_MASK = False   # <- toggle here

# ── Speed binning column ────────────────────────────────────────────────────────
# 'forward_speed'         -- per-frame instantaneous body velocity (mm/s)
# 'step_cycle_mean_speed' -- mean speed over each T1_left step cycle (mm/s)
#                            (NaN frames at bout edges are excluded automatically)
SPEED_BIN_COL = 'step_cycle_mean_speed'   # <- toggle here

# ── Activity mask (computed first so quantiles reflect the plotted pool) ────────
_spd_real_all = df_valid[SPEED_BIN_COL]
if USE_ACTIVITY_MASK and 'mean_abs_vel' in df_valid.columns:
    _MAV_THRESH    = df_valid['mean_abs_vel'].quantile(0.40)
    _activity_mask = df_valid['mean_abs_vel'] > _MAV_THRESH
else:
    _activity_mask = pd.Series(True, index=df_valid.index)

# ── Equal-count speed bins from the valid pool ──────────────────────────────────
# Each group contains ~1/3 of the frames that will actually be plotted.
_valid_pool = _activity_mask & _spd_real_all.notna()
_spd_valid  = _spd_real_all[_valid_pool]
_q33, _q67  = _spd_valid.quantile([0.33, 0.67]).values
SPEED_GROUPS = [
    ('slow',   0,     _q33),
    ('medium', _q33,  _q67),
    ('fast',   _q67,  np.inf),
]

# ── Compute phase of each leg relative to L1 (T1_left) ──────────────────────────
_ref_phase = df_valid['T1_left_phase'].values
for _leg in LEGS:
    if _leg == 'T1_left':
        continue
    _diff = df_valid[f'{_leg}_phase'].values - _ref_phase
    df_valid[f'{_leg}_phase_rel_L1'] = np.arctan2(np.sin(_diff), np.cos(_diff))

# ── Build bout-safe LineCollection segments ──────────────────────────────────────
def make_lc_segments(df, mask, x_col, y_col):
    """(N, 2, 2) segment array skipping across-bout joins."""
    all_segs = []
    for _, grp in df[mask].groupby('bout_id', sort=False):
        pts = grp[[x_col, y_col]].values
        if len(pts) < 2:
            continue
        all_segs.append(np.concatenate([pts[:-1, np.newaxis],
                                         pts[1:,  np.newaxis]], axis=1))
    return np.concatenate(all_segs, axis=0) if all_segs else np.empty((0, 2, 2))



leg_mat  = [['T1_left', 'T2_left', 'T3_left'],
            ['T1_right','T2_right','T3_right']]

# ── One figure per speed bin ─────────────────────────────────────────────────────
for _label, _spd_lo, _spd_hi in SPEED_GROUPS:
    _spd_mask = (_spd_real_all >= _spd_lo) & (_spd_real_all < _spd_hi)
    _n_frames = (_spd_mask & _activity_mask).sum()
    print(f"Speed bin '{_label}': [{_spd_lo:.2f}, {_spd_hi:.2f}) mm/s  "
          f"→ {_n_frames:,} frames after activity filter")

    fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharex=True, sharey=True)
    _last_lc = None

    for i, row_legs in enumerate(leg_mat):
        for j, leg in enumerate(row_legs):
            ax = axes[i, j]
            ax.set_title(LEG_LABELS[leg], fontsize=20)
            ax.set_axis_off()

            if leg == 'T1_left':          # reference leg
                ax.text(0.5, 0.5, 'L1\n(reference)', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12, color='gray')
                continue

            phase_col = f'{leg}_phase_rel_L1'
            if phase_col not in df_valid.columns:
                continue

            _mask = _activity_mask & _spd_mask & df_valid[phase_col].notna()
            segs  = make_lc_segments(df_valid, _mask, 'PC1', 'PC2')
            if len(segs) == 0:
                continue

            seg_colors = []
            for _, grp in df_valid[_mask].groupby('bout_id', sort=False):
                c = grp[phase_col].values
                if len(c) < 2:
                    continue
                _c = np.abs(c[:-1]) if PHASE_MODE == 'absolute' else c[:-1]
                seg_colors.extend(_c)
            seg_colors = np.array(seg_colors)

            lc = LineCollection(segs, cmap=PHASE_CMAP, norm=_norm, linewidth=1, alpha=0.3, rasterized=True) ########## Modified by Elliott
            lc.set_array(seg_colors)
            ax.add_collection(lc)
            ax.autoscale()
            _last_lc = lc

    if _last_lc is not None:
        cbar_ax = fig.add_axes([1.02, 0.15, 0.02, 0.7])
        cbar    = fig.colorbar(_last_lc, cax=cbar_ax)
        cbar.set_ticks(_cb_ticks)
        cbar.set_ticklabels(_cb_labels, fontsize=11)
        cbar.set_label(_cb_title, rotation=270, labelpad=26, fontsize=13)
        cbar.solids.set_alpha(1)


    _spd_hi_str = f'{_spd_hi:.2f}' if np.isfinite(_spd_hi) else '∞'
    fig.suptitle(f'Phase portrait — {_label} speed  '
                 f'[{_spd_lo:.2f}--{_spd_hi_str} mm/s]  ({SPEED_BIN_COL})  ({_n_frames:,} frames)',
                 fontsize=13)
    fig.supxlabel('PC1', fontsize=13)
    fig.supylabel('PC2', fontsize=13)
    plt.tight_layout()
    _fname = OUTPUT_DIR / f'pca_phase_portrait_{_label}.svg'
    plt.savefig(_fname, bbox_inches='tight',dpi=300)
    # Save each leg panel as a separate transparent PNG for Inkscape
    _renderer = fig.canvas.get_renderer()
    for _i, _row_legs in enumerate(leg_mat):
        for _j, _leg in enumerate(_row_legs):
            _ax = axes[_i, _j]
            _bbox = _ax.get_tightbbox(_renderer)
            if _bbox is None:
                continue
            _bbox_inch = _bbox.transformed(fig.dpi_scale_trans.inverted())
            _png_fname = OUTPUT_DIR / f'pca_phase_portrait_{_label}_{_leg}.png'
            fig.savefig(_png_fname, bbox_inches=_bbox_inch.expanded(1.08, 1.08),
                        dpi=150, transparent=True)
            print(f"Saved: {_png_fname}")
    plt.show()
    print(f"Saved: {_fname}")

In [ ]:
# Cell 36 -- Paper Figure: Speed-Binned Trajectory
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

# ── Config ──────────────────────────────────────────────────────────────────────
PHASE_CMAP2 = 'twilight'

# ── Activity mask ───────────────────────────────────────────────────────────────
# True  → keep only top 60% of frames by mean joint angular velocity
# False → keep all frames with valid speed data (recommended for full comparison)
USE_ACTIVITY_MASK = False   # <- toggle here

# ── Speed binning column ────────────────────────────────────────────────────────
# 'forward_speed'         -- per-frame instantaneous body velocity (mm/s)
# 'step_cycle_mean_speed' -- mean speed over each T1_left step cycle (mm/s)
SPEED_BIN_COL = 'step_cycle_mean_speed'   # <- toggle here

# ── Activity mask (computed first so quantiles reflect the plotted pool) ────────
_spd_real_all2 = df_valid[SPEED_BIN_COL]
if USE_ACTIVITY_MASK and 'mean_abs_vel' in df_valid.columns:
    _activity_mask2 = df_valid['mean_abs_vel'] > df_valid['mean_abs_vel'].quantile(0.40)
else:
    _activity_mask2 = pd.Series(True, index=df_valid.index)

# ── Equal-count speed bins from the valid pool ──────────────────────────────────
_valid_pool2 = _activity_mask2 & _spd_real_all2.notna()
_spd_valid2  = _spd_real_all2[_valid_pool2]
_q33_2, _q67_2 = _spd_valid2.quantile([0.33, 0.67]).values
SPEED_GROUPS2 = [
    ('slow',   0,       _q33_2),
    ('medium', _q33_2,  _q67_2),
    ('fast',   _q67_2,  np.inf),
]

# ── Step phase column per leg ───────────────────────────────────────────────────
_leg_phase_cols = {leg: f'{leg}_phase' for leg in LEGS}

_norm2   = Normalize(-np.pi, np.pi)
leg_mat2 = [['T1_left', 'T2_left', 'T3_left'],
            ['T1_right','T2_right','T3_right']]

# ── One figure per speed bin ─────────────────────────────────────────────────────
for _label, _spd_lo, _spd_hi in SPEED_GROUPS2:
    _spd_mask2 = (_spd_real_all2 >= _spd_lo) & (_spd_real_all2 < _spd_hi)
    _base_mask = _activity_mask2 & _spd_mask2
    _n_frames  = _base_mask.sum()
    print(f"Speed bin '{_label}': [{_spd_lo:.1f}, "
          f"{'∞' if not np.isfinite(_spd_hi) else f'{_spd_hi:.1f}'}) mm/s  "
          f"→ {_n_frames:,} frames after activity filter")

    # Build segments + colour arrays in one groupby pass (bout-safe)
    _seg_list   = []
    _color_dict = {leg: [] for leg in LEGS}

    for _, grp in df_valid[_base_mask].groupby('bout_id', sort=False):
        pts = grp[['PC1', 'PC2']].values
        if len(pts) < 2:
            continue
        _seg_list.append(
            np.concatenate([pts[:-1, np.newaxis], pts[1:, np.newaxis]], axis=1)
        )
        for leg in LEGS:
            pcol = _leg_phase_cols[leg]
            vals = grp[pcol].values if pcol in grp.columns else np.full(len(grp), np.nan)
            _color_dict[leg].extend(vals[:-1])

    if not _seg_list:
        print(f"  No data for bin '{_label}', skipping.")
        continue

    _segments_bin  = np.concatenate(_seg_list, axis=0)
    _color_arrays2 = {leg: np.array(_color_dict[leg]) for leg in LEGS}

    fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharex=True, sharey=True)
    _last_lc = None

    for i, row_legs in enumerate(leg_mat2):
        for j, leg in enumerate(row_legs):
            ax = axes[i, j]
            ax.set_title(LEG_LABELS[leg], fontsize=20)

            c_arr = _color_arrays2[leg]
            if len(_segments_bin) == 0 or np.all(np.isnan(c_arr)):
                ax.set_axis_off()
                continue

            lc = LineCollection(_segments_bin, cmap=PHASE_CMAP2, norm=_norm2,
                                linewidth=1)
            lc.set_array(c_arr)
            ax.add_collection(lc)
            ax.autoscale()
            ax.set_axis_off()
            _last_lc = lc

    if _last_lc is not None:
        cbar_ax = fig.add_axes([1.02, 0.15, 0.02, 0.7])
        cbar    = fig.colorbar(_last_lc, cax=cbar_ax)
        cbar.set_ticks([-np.pi, 0, np.pi])
        cbar.set_ticklabels([r'$-\pi$', '0', r'$\pi$'], fontsize=20)
        cbar.set_label('Step phase', rotation=270, labelpad=18, fontsize=20)

    _spd_hi_str = f'{_spd_hi:.2f}' if np.isfinite(_spd_hi) else '∞'
    fig.suptitle(f'Own-phase portrait — {_label} speed  '
                 f'[{_spd_lo:.2f}–{_spd_hi_str} mm/s]  ({SPEED_BIN_COL})  ({_n_frames:,} frames)',
                 fontsize=13)
    plt.tight_layout()
    _fname = OUTPUT_DIR / f'pca_own_phase_portrait_{_label}.pdf'
    plt.savefig(_fname, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_fname}")


In [ ]:
# Cell 37 -- Paper Figure: 3-D Speed-Binned PCA
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize

assert 'PC3' in df_valid.columns, "PC3 not found -- re-run Cells 13-14 with N_PCA_COMPONENTS >= 3"

# ── Phase display mode (same as Cell 34) ─────────────────────────────────────
PHASE_MODE_3D = 'absolute'   # 'absolute' or 'signed'

# ── Activity mask ─────────────────────────────────────────────────────────────
# True  → keep only top 60% of frames by mean joint angular velocity
# False → keep all frames with valid speed data (recommended for full comparison)
USE_ACTIVITY_MASK = False   # <- toggle here

# ── Speed binning column ──────────────────────────────────────────────────────
# 'forward_speed'         -- per-frame instantaneous body velocity (mm/s)
# 'step_cycle_mean_speed' -- mean speed over each T1_left step cycle (mm/s)
SPEED_BIN_COL = 'step_cycle_mean_speed'   # <- toggle here

if PHASE_MODE_3D == 'absolute':
    _cmap_3d = 'RdYlGn_r'
    _norm_3d = Normalize(0, np.pi)
    _cb_label_3d = '|Phase offset from L1|'
else:
    _cmap_3d = 'twilight'
    _norm_3d = Normalize(-np.pi, np.pi)
    _cb_label_3d = 'Phase rel. to L1  (+ leads, - lags)'

# ── Activity mask + equal-count speed bins ────────────────────────────────────
_spd_3d = df_valid[SPEED_BIN_COL]
if USE_ACTIVITY_MASK and 'mean_abs_vel' in df_valid.columns:
    _act_3d = df_valid['mean_abs_vel'] > df_valid['mean_abs_vel'].quantile(0.40)
else:
    _act_3d = pd.Series(True, index=df_valid.index)

_valid_pool_3d = _act_3d & _spd_3d.notna()
_spd_valid_3d  = _spd_3d[_valid_pool_3d]
_q33_3d, _q67_3d = _spd_valid_3d.quantile([0.33, 0.67]).values
SPEED_GROUPS_3D = [
    ('slow',   0,        _q33_3d),
    ('medium', _q33_3d,  _q67_3d),
    ('fast',   _q67_3d,  np.inf),
]

# ── Ensure relative-phase columns exist (same as Cell 34) ────────────────────
_ref_ph_3d = df_valid['T1_left_phase'].values
for _leg in LEGS:
    if _leg == 'T1_left':
        continue
    _d = df_valid[f'{_leg}_phase'].values - _ref_ph_3d
    df_valid[f'{_leg}_phase_rel_L1'] = np.arctan2(np.sin(_d), np.cos(_d))

leg_mat_3d = [['T1_left', 'T2_left', 'T3_left'],
              ['T1_right', 'T2_right', 'T3_right']]

# ── One figure per speed bin ──────────────────────────────────────────────────
for _label, _spd_lo, _spd_hi in SPEED_GROUPS_3D:
    _spd_mask_3d = (_spd_3d >= _spd_lo) & (_spd_3d < _spd_hi)
    _base_3d = _act_3d & _spd_mask_3d
    _n = int(_base_3d.sum())
    _spd_hi_str = f'{_spd_hi:.1f}' if np.isfinite(_spd_hi) else 'inf'
    print(f"Speed bin '{_label}' [{_spd_lo:.1f}-{_spd_hi_str}]: {_n:,} frames")
    if _n < 10:
        continue

    fig = plt.figure(figsize=(16, 10))
    _last_sc = None

    for _i, row_legs in enumerate(leg_mat_3d):
        for _j, _leg in enumerate(row_legs):
            ax = fig.add_subplot(2, 3, _i * 3 + _j + 1, projection='3d')
            ax.set_title(LEG_LABELS[_leg], fontsize=13)

            if _leg == 'T1_left':
                ax.text2D(0.5, 0.5, 'L1\n(reference)', ha='center', va='center',
                          transform=ax.transAxes, fontsize=11, color='gray')
                ax.set_axis_off()
                continue

            _ph_col = f'{_leg}_phase_rel_L1'
            _m = _base_3d & df_valid[_ph_col].notna()
            sub = df_valid[_m]
            if len(sub) < 2:
                continue

            _c = np.abs(sub[_ph_col].values) if PHASE_MODE_3D == 'absolute'                  else sub[_ph_col].values
            sc = ax.scatter(sub['PC1'].values, sub['PC2'].values, sub['PC3'].values,
                            c=_c, cmap=_cmap_3d, norm=_norm_3d,
                            s=5, alpha=0.35, linewidths=2)
            ax.set_xlabel('PC1', fontsize=8, labelpad=0)
            ax.set_ylabel('PC2', fontsize=8, labelpad=0)
            ax.set_zlabel('PC3', fontsize=8, labelpad=0)
            ax.tick_params(labelsize=7)
            _last_sc = sc

    if _last_sc is not None:
        cbar_ax = fig.add_axes([1.02, 0.15, 0.02, 0.70])
        cbar = fig.colorbar(_last_sc, cax=cbar_ax)
        cbar.set_label(_cb_label_3d, rotation=270, labelpad=20, fontsize=11)

    fig.suptitle(f'3-D phase portrait -- {_label} speed  '
                 f'[{_spd_lo:.1f}-{_spd_hi_str} mm/s]  ({SPEED_BIN_COL})  ({_n:,} frames)',
                 fontsize=12)
    plt.tight_layout()
    _fname = OUTPUT_DIR / f'pca3d_phase_portrait_{_label}.pdf'
    plt.savefig(_fname, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_fname}")


In [ ]:
# Cell 38 -- Paper Figure: 3-D Speed-Binned Trajectory
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize

assert 'PC3' in df_valid.columns, "PC3 not found -- re-run Cells 13-14 with N_PCA_COMPONENTS >= 3"

# ── Activity mask ─────────────────────────────────────────────────────────────
# True  → keep only top 60% of frames by mean joint angular velocity
# False → keep all frames with valid speed data (recommended for full comparison)
USE_ACTIVITY_MASK = False   # <- toggle here

# ── Speed binning column ──────────────────────────────────────────────────────
# 'forward_speed'         -- per-frame instantaneous body velocity (mm/s)
# 'step_cycle_mean_speed' -- mean speed over each T1_left step cycle (mm/s)
SPEED_BIN_COL = 'step_cycle_mean_speed'   # <- toggle here

# ── Activity mask + equal-count speed bins ────────────────────────────────────
_spd_3db = df_valid[SPEED_BIN_COL]
if USE_ACTIVITY_MASK and 'mean_abs_vel' in df_valid.columns:
    _act_3db = df_valid['mean_abs_vel'] > df_valid['mean_abs_vel'].quantile(0.40)
else:
    _act_3db = pd.Series(True, index=df_valid.index)

_valid_pool_3db = _act_3db & _spd_3db.notna()
_spd_valid_3db  = _spd_3db[_valid_pool_3db]
_q33_3db, _q67_3db = _spd_valid_3db.quantile([0.33, 0.67]).values
SPEED_GROUPS_3DB = [
    ('slow',   0,         _q33_3db),
    ('medium', _q33_3db,  _q67_3db),
    ('fast',   _q67_3db,  np.inf),
]

_norm_3db = Normalize(-np.pi, np.pi)
leg_mat_3db = [['T1_left', 'T2_left', 'T3_left'],
               ['T1_right', 'T2_right', 'T3_right']]

# ── One figure per speed bin ──────────────────────────────────────────────────
for _label, _spd_lo, _spd_hi in SPEED_GROUPS_3DB:
    _spd_mask_3db = (_spd_3db >= _spd_lo) & (_spd_3db < _spd_hi)
    _base_3db = _act_3db & _spd_mask_3db
    _n = int(_base_3db.sum())
    _spd_hi_str = f'{_spd_hi:.1f}' if np.isfinite(_spd_hi) else 'inf'
    print(f"Speed bin '{_label}' [{_spd_lo:.1f}-{_spd_hi_str}]: {_n:,} frames")
    if _n < 10:
        continue

    fig = plt.figure(figsize=(16, 10))
    _last_sc2 = None

    for _i, row_legs in enumerate(leg_mat_3db):
        for _j, _leg in enumerate(row_legs):
            ax = fig.add_subplot(2, 3, _i * 3 + _j + 1, projection='3d')
            ax.set_title(LEG_LABELS[_leg], fontsize=13)

            _ph_col = f'{_leg}_phase'
            _m = _base_3db & df_valid[_ph_col].notna()
            sub = df_valid[_m]
            if len(sub) < 2:
                ax.set_axis_off()
                continue

            sc2 = ax.scatter(sub['PC1'].values, sub['PC2'].values, sub['PC3'].values,
                             c=sub[_ph_col].values, cmap='twilight',
                             norm=_norm_3db, s=1, alpha=0.35, linewidths=1)
            ax.set_xlabel('PC1', fontsize=8, labelpad=0)
            ax.set_ylabel('PC2', fontsize=8, labelpad=0)
            ax.set_zlabel('PC3', fontsize=8, labelpad=0)
            ax.tick_params(labelsize=7)
            _last_sc2 = sc2

    if _last_sc2 is not None:
        cbar_ax = fig.add_axes([1.02, 0.15, 0.02, 0.70])
        cbar = fig.colorbar(_last_sc2, cax=cbar_ax)
        cbar.set_ticks([-np.pi, 0, np.pi])
        cbar.set_ticklabels([r'$-\pi$' + '\n(liftoff)', '0\n(mid-stride)',
                             r'$+\pi$' + '\n(pre-liftoff)'], fontsize=10)
        cbar.set_label('Step phase', rotation=270, labelpad=20, fontsize=11)

    fig.suptitle(f'3-D own-phase portrait -- {_label} speed  '
                 f'[{_spd_lo:.1f}-{_spd_hi_str} mm/s]  ({SPEED_BIN_COL})  ({_n:,} frames)',
                 fontsize=12)
    plt.tight_layout()
    _fname = OUTPUT_DIR / f'pca3d_own_phase_portrait_{_label}.pdf'
    plt.savefig(_fname, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_fname}")


In [ ]:
# Cell 39 -- Paper Figure: Range of Motion
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap

# ── Config ────────────────────────────────────────────────────────────────────
N_PHASE_BINS_TRJ = 24
STANCE_CMAP      = ListedColormap([
    'tab:purple', 'tab:blue', 'tab:cyan', 'tab:green', 'y', 'tab:orange', 'tab:red'
])
REF_PHASE_COL    = 'T1_left_phase'   # L1 phase used for binning

# ── Activity mask ─────────────────────────────────────────────────────────────
# True  → keep only top 60% of frames by mean joint angular velocity
# False → keep all frames with valid speed data (recommended)
USE_ACTIVITY_MASK = False   # <- toggle here

# ── Speed binning column ──────────────────────────────────────────────────────
# 'forward_speed'         -- per-frame instantaneous body velocity (mm/s)
# 'step_cycle_mean_speed' -- mean speed over each T1_left step cycle (mm/s)
SPEED_BIN_COL = 'step_cycle_mean_speed'   # <- toggle here

# ── Equal-count speed bins from the valid pool (consistent with Cells 34-37) ──
_spd_38 = df_valid[SPEED_BIN_COL]
if USE_ACTIVITY_MASK and 'mean_abs_vel' in df_valid.columns:
    _act_38 = df_valid['mean_abs_vel'] > df_valid['mean_abs_vel'].quantile(0.40)
else:
    _act_38 = pd.Series(True, index=df_valid.index)

_valid_38      = _act_38 & _spd_38.notna() & df_valid[REF_PHASE_COL].notna()
_spd_v38       = _spd_38[_valid_38]
_q33_38, _q67_38 = _spd_v38.quantile([0.33, 0.67]).values
SPEED_GROUPS_38 = [
    ('slow',   0,        _q33_38,  ':'),   # (label, lo, hi, linestyle)
    ('medium', _q33_38,  _q67_38,  '--'),
    ('fast',   _q67_38,  np.inf,   '-'),
]
print(f"Speed bins (mm/s):  slow < {_q33_38:.1f}  |  "
      f"medium < {_q67_38:.1f}  |  fast >= {_q67_38:.1f}")

# ── Bin and average ───────────────────────────────────────────────────────────
phase_edges = np.linspace(-np.pi, np.pi, N_PHASE_BINS_TRJ + 1)

fig, ax = plt.subplots(figsize=(8, 8))
_last_lc    = None
_line_labels = []

for _grp_label, _spd_lo, _spd_hi, _ls in SPEED_GROUPS_38:
    pc1_avg, pc2_avg, stance_avg = [], [], []

    for k in range(len(phase_edges) - 1):
        mask = (
            _act_38 &
            (_spd_38 >= _spd_lo) & (_spd_38 < _spd_hi) &
            (df_valid[REF_PHASE_COL] > phase_edges[k]) &
            (df_valid[REF_PHASE_COL] <= phase_edges[k + 1])
        )
        bin_df = df_valid[mask]
        if len(bin_df) == 0:
            continue
        pc1_avg.append(bin_df['PC1'].mean())
        pc2_avg.append(bin_df['PC2'].mean())
        stance_avg.append(round(bin_df['n_legs_stance'].mean()))

    if len(pc1_avg) < 2:
        print(f'Speed bin "{_grp_label}": not enough data, skipping')
        continue

    pts  = np.array([pc1_avg, pc2_avg]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    norm = plt.Normalize(0, 6)
    lc   = LineCollection(segs, cmap=STANCE_CMAP, norm=norm,
                          linestyle=_ls, linewidth=2.5, alpha=1.0)
    lc.set_array(np.array(stance_avg[:-1]))
    ax.add_collection(lc)
    _last_lc = lc
    _line_labels.append((_grp_label, _ls))

ax.autoscale()
ax.set_xlabel('PC1', fontsize=13)
ax.set_ylabel('PC2', fontsize=13)
ax.set_title('Phase-averaged PCA trajectory by speed group', fontsize=13)
sns.despine(ax=ax)

if _last_lc is not None:
    cbar = fig.colorbar(_last_lc, ax=ax, fraction=0.025, pad=0.15)
    cbar.set_label('Number of legs in stance', fontsize=12)
    cbar.set_ticks(range(7))

if _line_labels:
    from matplotlib.lines import Line2D
    _handles = [
        Line2D([0], [0], color='#333333', linewidth=2.5, linestyle=ls, label=lbl)
        for lbl, ls in _line_labels
    ]
    ax.legend(handles=_handles, title='Speed group', fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_speed_binned_trajectory_averaged.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'pca_speed_binned_trajectory_averaged.pdf'}")


# ═══════════════════════════════════════════════════════════════════════════════
# Figure 2 — Per-leg swing/stance fraction mapped onto the averaged PCA loop
# ═══════════════════════════════════════════════════════════════════════════════
# Each subplot: same 3 speed-group loops (slow/medium/fast), colored by the
# fraction of frames where THAT leg is in stance at each L1 phase bin.
# Blue = swing (0), Red = stance (1).

_cmap_ss = plt.cm.coolwarm
_norm_ss = plt.Normalize(0, 1)

_leg_layout2 = [
    ['T1_left',  'T2_left',  'T3_left'],
    ['T1_right', 'T2_right', 'T3_right'],
]

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)
_last_lc2 = None

for _ri, row_legs in enumerate(_leg_layout2):
    for _ci, _leg in enumerate(row_legs):
        ax2 = axes2[_ri, _ci]
        _ss_col = f'{_leg}_swing_stance'
        ax2.set_title(LEG_LABELS.get(_leg, _leg), fontsize=13)
        ax2.set_axis_off()

        for _grp_label, _spd_lo, _spd_hi, _ls in SPEED_GROUPS_38:
            pc1_avg2, pc2_avg2, ss_avg2 = [], [], []
            for k in range(len(phase_edges) - 1):
                mask2 = (
                    _act_38 &
                    (_spd_38 >= _spd_lo) & (_spd_38 < _spd_hi) &
                    (df_valid[REF_PHASE_COL] > phase_edges[k]) &
                    (df_valid[REF_PHASE_COL] <= phase_edges[k + 1])
                )
                bd = df_valid[mask2]
                if len(bd) == 0:
                    continue
                pc1_avg2.append(bd['PC1'].mean())
                pc2_avg2.append(bd['PC2'].mean())
                ss_avg2.append(bd[_ss_col].mean() if _ss_col in bd.columns else np.nan)

            if len(pc1_avg2) < 2:
                continue
            pts2  = np.array([pc1_avg2, pc2_avg2]).T.reshape(-1, 1, 2)
            segs2 = np.concatenate([pts2[:-1], pts2[1:]], axis=1)
            lc2 = LineCollection(segs2, cmap=_cmap_ss, norm=_norm_ss,
                                 linestyle=_ls, linewidth=2.5, alpha=1.0)
            lc2.set_array(np.array(ss_avg2[:-1]))
            ax2.add_collection(lc2)
            ax2.set_axis_on()
            _last_lc2 = lc2

        ax2.autoscale()
        sns.despine(ax=ax2)
        if _ri == 1:
            ax2.set_xlabel('PC1', fontsize=11)
        if _ci == 0:
            ax2.set_ylabel('PC2', fontsize=11)

        # Speed group legend — top-left panel only
        if _ri == 0 and _ci == 0:
            from matplotlib.lines import Line2D as _LD2
            _ss_leg_handles = [
                _LD2([0],[0], color='#333333', linewidth=2.5, linestyle=ls, label=lbl)
                for lbl, _, _, ls in SPEED_GROUPS_38
            ]
            ax2.legend(handles=_ss_leg_handles, title='Speed group',
                       fontsize=9, loc='upper right')

if _last_lc2 is not None:
    cbar2 = fig2.colorbar(_last_lc2, ax=axes2[:, -1].tolist(),
                          fraction=0.03, pad=0.04)
    cbar2.set_label('P(stance)', fontsize=12)
    cbar2.set_ticks([0, 0.5, 1])
    cbar2.set_ticklabels(['0\n(swing)', '0.5', '1\n(stance)'], fontsize=10)

fig2.suptitle('Phase-averaged PCA trajectory — per-leg stance fraction\n'
              '(Blue = swing, Red = stance  |  linestyle = speed group)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_per_leg_stance_trajectory.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'pca_per_leg_stance_trajectory.pdf'}")


In [ ]:
# Cell 40 -- Paper Figure: 3-D Per-Leg Stance Trajectory
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import matplotlib as _mpl39

assert 'PC3' in df_valid.columns, "PC3 not found -- re-run Cells 13-14 with N_PCA_COMPONENTS >= 3"

# ── Config (mirrors Cell 38) ──────────────────────────────────────────────────
N_PHASE_BINS_3D   = 24
REF_PHASE_COL_3D  = 'T1_left_phase'
USE_ACTIVITY_MASK = False           # <- toggle here
SPEED_BIN_COL     = 'step_cycle_mean_speed'   # <- toggle here

_spd_39 = df_valid[SPEED_BIN_COL]
if USE_ACTIVITY_MASK and 'mean_abs_vel' in df_valid.columns:
    _act_39 = df_valid['mean_abs_vel'] > df_valid['mean_abs_vel'].quantile(0.40)
else:
    _act_39 = pd.Series(True, index=df_valid.index)

_valid_39      = _act_39 & _spd_39.notna() & df_valid[REF_PHASE_COL_3D].notna()
_q33_39, _q67_39 = _spd_39[_valid_39].quantile([0.33, 0.67]).values
SPEED_GROUPS_39 = [
    ('slow',   0,        _q33_39, ':'),
    ('medium', _q33_39,  _q67_39, '--'),
    ('fast',   _q67_39,  np.inf,  '-'),
]
print(f"Speed bins (mm/s):  slow < {_q33_39:.1f}  |  "
      f"medium < {_q67_39:.1f}  |  fast >= {_q67_39:.1f}")

phase_edges_39 = np.linspace(-np.pi, np.pi, N_PHASE_BINS_3D + 1)
_cmap_ss3d = plt.cm.coolwarm
_norm_ss3d = Normalize(0, 1)

_leg_layout_3d = [
    ['T1_left',  'T2_left',  'T3_left'],
    ['T1_right', 'T2_right', 'T3_right'],
]

fig3d, axes3d = plt.subplots(2, 3, figsize=(16, 10),
                              subplot_kw={'projection': '3d'})

for _ri, row_legs in enumerate(_leg_layout_3d):
    for _ci, _leg in enumerate(row_legs):
        ax3 = axes3d[_ri, _ci]
        _ss_col = f'{_leg}_swing_stance'
        ax3.set_title(LEG_LABELS.get(_leg, _leg), fontsize=11)

        for _grp_label, _spd_lo, _spd_hi, _ls in SPEED_GROUPS_39:
            pc1_3d, pc2_3d, pc3_3d, ss_3d = [], [], [], []
            for k in range(len(phase_edges_39) - 1):
                m3 = (
                    _act_39 &
                    (_spd_39 >= _spd_lo) & (_spd_39 < _spd_hi) &
                    (df_valid[REF_PHASE_COL_3D] > phase_edges_39[k]) &
                    (df_valid[REF_PHASE_COL_3D] <= phase_edges_39[k + 1])
                )
                bd3 = df_valid[m3]
                if len(bd3) == 0:
                    continue
                pc1_3d.append(bd3['PC1'].mean())
                pc2_3d.append(bd3['PC2'].mean())
                pc3_3d.append(bd3['PC3'].mean())
                ss_3d.append(bd3[_ss_col].mean() if _ss_col in bd3.columns else np.nan)

            if len(pc1_3d) < 2:
                continue

            # Draw each segment individually with the colour from the stance fraction
            for k in range(len(pc1_3d) - 1):
                sv = ss_3d[k]
                col3 = _cmap_ss3d(_norm_ss3d(sv)) if not np.isnan(sv) else (0.5, 0.5, 0.5, 1.)
                ax3.plot([pc1_3d[k], pc1_3d[k+1]],
                         [pc2_3d[k], pc2_3d[k+1]],
                         [pc3_3d[k], pc3_3d[k+1]],
                         color=col3, linewidth=2.0, linestyle=_ls, alpha=0.9)

        ax3.set_xlabel('PC1', fontsize=8, labelpad=0)
        ax3.set_ylabel('PC2', fontsize=8, labelpad=0)
        ax3.set_zlabel('PC3', fontsize=8, labelpad=0)
        ax3.view_init(elev=15, azim=30)
        ax3.tick_params(labelsize=7)
# Shared colorbar (using ScalarMappable since no LineCollection)
_sm3d = _mpl39.cm.ScalarMappable(cmap=_cmap_ss3d, norm=_norm_ss3d)
_sm3d.set_array([])
cbar3d = fig3d.colorbar(_sm3d, ax=axes3d[:, -1].tolist(),
                         fraction=0.03, pad=0.04)
cbar3d.set_label('P(stance)', fontsize=12)
cbar3d.set_ticks([0, 0.5, 1])
cbar3d.set_ticklabels(['0\n(swing)', '0.5', '1\n(stance)'], fontsize=10)

# Speed group legend
_handles39 = [
    Line2D([0],[0], color='#333333', linewidth=2.0, linestyle=ls, label=lbl)
    for lbl, _, _, ls in SPEED_GROUPS_39
]
fig3d.legend(handles=_handles39, title='Speed group', fontsize=9,
             loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))

fig3d.suptitle('3-D phase-averaged PCA trajectory — per-leg stance fraction\n'
               '(Blue = swing, Red = stance  |  linestyle = speed group)',
               fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca3d_per_leg_stance_trajectory.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'pca3d_per_leg_stance_trajectory.pdf'}")


In [ ]:
# Cell 41 -- ROM Joints Configuration
# ── Config ────────────────────────────────────────────────────────────────────
ROM_JOINTS = ['coxa_abduct', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia', 'tarsus']                                                                                              
ROM_LABELS = ['Coxa\nabd', 'Coxa\nrot', 'Coxa\nflex', 'Femur\nrot',                                                                                                                           
              'Femur\nflex', 'Tibia\nflex', 'Tarsus\nflex']                                                                                                                                   
LEG_TIERS  = [('T1', 'T1_left', 'T1_right'),                                                                                                                                                  
              ('T2', 'T2_left', 'T2_right'),                                                                                                                                                  
              ('T3', 'T3_left', 'T3_right')]                                                                                                                                                  
                                                                                                                                                                                              
def rom_per_cycle(data, col):                                                                                                                                                                 
    """Mean per-cycle ROM (max-min per step cycle) in degrees.                                                                                                                                
                                                                                                                                                                                              
    Cycles are defined by T1_left swing onsets (step_cycle_id).                                                                                                                               
    Returns mean across all complete cycles with >= 2 frames.                                                                                                                                 
    """                                                                                                                                                                                       
    sub = data.dropna(subset=[col, 'step_cycle_id'])                                                                                                                                          
    if sub.empty:                                                                                                                                                                             
        return np.nan                                                                                                                                                                         
    ranges = (                                                                                                                                                                                
        sub.groupby(['bout_id', 'step_cycle_id'])[col]                                                                                                                                        
           .agg(lambda x: x.max() - x.min() if len(x) >= 2 else np.nan)                                                                                                                       
           .dropna()                                                                                                                                                                          
    )                                                                                                                                                                                         
    return np.degrees(ranges.mean()) if len(ranges) else np.nan
# ── Part 1: Bar chart — T1 joint ROM (L+R averaged, all cycles) ───────────────
roms_bar = []
for joint in ROM_JOINTS:
    col_L = f'T1_left_{joint}'
    col_R = f'T1_right_{joint}'
    rom_L = rom_per_cycle(df_valid, col_L)
    rom_R = rom_per_cycle(df_valid, col_R)
    roms_bar.append(np.nanmean([rom_L, rom_R]))
plt.figure(figsize=(8, 3))
plt.bar(range(len(roms_bar)), roms_bar, tick_label=ROM_LABELS, align='center',
        color=sns.color_palette('viridis', len(roms_bar)))
plt.xlabel('Joint angle')
plt.ylabel('Range of motion (°)')
sns.despine()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rom_bar_T1.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'rom_bar_T1.pdf'}")
# ── Part 2: Per-fly strip+point plot (3 rows × n_joints) ──────────────────────
rom_records = {}
for fly_id in df_valid['fly_id'].unique():
    fly_df = df_valid[df_valid['fly_id'] == fly_id]
    row = {}
    for tier, leg_L, leg_R in LEG_TIERS:
        for joint in ROM_JOINTS:
            col_L = f'{leg_L}_{joint}'
            col_R = f'{leg_R}_{joint}'
            if col_L in fly_df.columns and col_R in fly_df.columns:
                row[f'{tier}_{joint}'] = np.nanmean([
                    rom_per_cycle(fly_df, col_L),
                    rom_per_cycle(fly_df, col_R),
                ])
    rom_records[fly_id] = row
rom_by_fly = pd.DataFrame(rom_records).T
fig, axes = plt.subplots(3, 1, figsize=(7.5, 6), sharex=True, sharey=True)
for ax, (tier, _L, _R) in zip(axes, LEG_TIERS):
    cols   = [f'{tier}_{j}' for j in ROM_JOINTS]
    avail  = [c for c in cols if c in rom_by_fly.columns]
    labels = [ROM_LABELS[ROM_JOINTS.index(c.replace(tier + '_', ''))]
              for c in avail]
    data   = rom_by_fly[avail].copy()
    data.columns = labels
    sns.stripplot(data=data, ax=ax, palette='tab10', size=5, alpha=0.8)
    sns.pointplot(data=data, ax=ax, join=False, errorbar=None,
                  estimator='mean', color='k', markers='_', scale=1.5)
    ax.set_ylabel(tier, fontsize=11)
    sns.despine(ax=ax)
axes[-1].tick_params(axis='x', rotation=25)
fig.supylabel('Range of motion (°)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rom_per_fly_strips.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'rom_per_fly_strips.pdf'}")

---

## 10. Segment UMAP (DeAngelis et al., eLife 46409)

Reproduces the segment-windowed UMAP from DeAngelis et al.:
- **Full-body**: 6 leg-tip egocentric (x, y) positions, windowed segments, two-stage standardisation → UMAP 3D
- **Per-(leg × joint)**: one 2D UMAP per joint per leg — displayed as a 6 × N_joints grid
- **Per-joint-type**: one 3D UMAP per joint type using all 6 legs simultaneously

All three analyses share the same randomly-sampled segment centre frames for direct comparability.

**Key difference from paper**: our data is 800 Hz vs 150 Hz.  `HALF_WIN_MS=100` preserves the same 200 ms biological window; dimensionality scales to 161×12 = 1 932 (paper: 31×12 = 372).

In [ ]:
# Cell 42 -- Segment UMAP Configuration
# ============================================================
# SEGMENT UMAP CONFIGURATION  (DeAngelis et al., eLife 46409)
# ============================================================

FPS                  = 800
HALF_WIN_MS          = 100                                    # ms — paper: 100 ms
HALF_WIN_FRAMES      = int(round(HALF_WIN_MS * 1e-3 * FPS))   # 80 at 800 Hz
WIN_LENGTH           = 2 * HALF_WIN_FRAMES + 1                # 161 frames
N_SEGMENTS_TARGET    = 100_000                                 # paper: 10^5
SEGMENT_SEED         = 42
SEG_UMAP_COMPONENTS  = 6    # full-body & per-joint-type; per-(leg×joint) uses 2
SEG_UMAP_N_NEIGHBORS = 200
SEG_UMAP_MIN_DIST    = 0.8

print(f'Window : {WIN_LENGTH} frames  ({HALF_WIN_MS} ms half-width at {FPS} Hz)')
print(f'Full-body dims  : {WIN_LENGTH} × 12 = {WIN_LENGTH * 12}')
print(f'Per-joint dims  : {WIN_LENGTH} × 1  = {WIN_LENGTH}')
print(f'Per-type dims   : {WIN_LENGTH} × 6  = {WIN_LENGTH * 6}')
print(f'Target segments : {N_SEGMENTS_TARGET:,}')
print(f'Memory estimate (full-body raw float32) : '
      f'~{N_SEGMENTS_TARGET * WIN_LENGTH * 12 * 4 / 1e9:.2f} GB')
print('Reduce N_SEGMENTS_TARGET or HALF_WIN_MS if OOM.')

# ── Coloring toggle (applies to Cells 44, 46, 47, 48) ─────────────────────────
# Change here to set the default; or override in the 'Recompute Colors' cell.
COLOR_BY_JT      = 'n_legs_stance'  # 'instant_vel' | 'cycle_speed' | 'n_legs_stance'
                                   # 'phase_rel'   | 'phase'
PHASE_REL_LEG_JT = 'T1_right'     # used when COLOR_BY_JT == 'phase_rel'
PHASE_LEG_JT     = 'T1_left'      # used when COLOR_BY_JT == 'phase'


In [ ]:
# Cell 43 -- Sample Segments Function
def sample_segments(bout_dict, bout_keys, fly_ids_per_bout,
                     half_win, n_target, leg_site_indices, legs, seed=42):
    """Randomly sample fixed-length egocentric-position segments.

    Uses x, y of each leg-tip site from xpos_egocentric.
    Returns:
        segments : (N, win_length, n_legs*2)  float32
        meta     : dict with fly_id, bout_id, bout_idx, center_frame arrays
    """
    rng = np.random.default_rng(seed)
    win = 2 * half_win + 1
    valid_legs = [(leg, leg_site_indices[leg])
                  for leg in legs if leg_site_indices.get(leg) is not None]
    n_vars = len(valid_legs) * 2

    # Collect all valid (bout_idx, centre_frame) candidates
    all_bidx, all_ctr = [], []
    for b_idx, key in enumerate(bout_keys):
        T = np.asarray(bout_dict[key]['xpos_egocentric']).shape[0]
        if T <= 2 * half_win:
            continue
        ctrs = np.arange(half_win, T - half_win)
        all_bidx.extend([b_idx] * len(ctrs))
        all_ctr.extend(ctrs.tolist())

    all_bidx = np.array(all_bidx, dtype=np.int32)
    all_ctr  = np.array(all_ctr,  dtype=np.int32)
    n_avail  = len(all_bidx)
    print(f'Valid segment positions: {n_avail:,}')

    if n_avail <= n_target:
        print(f'Using all {n_avail:,} (fewer than target {n_target:,})')
        chosen = np.arange(n_avail)
    else:
        chosen = np.sort(rng.choice(n_avail, size=n_target, replace=False))

    sel_bidx = all_bidx[chosen]
    sel_ctr  = all_ctr[chosen]
    N = len(chosen)

    segments = np.empty((N, win, n_vars), dtype=np.float32)
    _key, _xpos = None, None
    for i, (b_idx, ctr) in enumerate(zip(sel_bidx, sel_ctr)):
        key = bout_keys[b_idx]
        if key != _key:
            _key  = key
            _xpos = np.asarray(bout_dict[key]['xpos_egocentric'], dtype=np.float32)
        win_data = _xpos[ctr - half_win: ctr + half_win + 1]  # (win, n_sites, 3)
        col = 0
        for leg, site_idx in valid_legs:
            segments[i, :, col]     = win_data[:, site_idx, 0]  # x
            segments[i, :, col + 1] = win_data[:, site_idx, 1]  # y
            col += 2

    meta = {
        'fly_id':       np.array([fly_ids_per_bout[b] for b in sel_bidx]),
        'bout_id':      np.array([bout_keys[b]         for b in sel_bidx]),
        'bout_idx':     sel_bidx,
        'center_frame': sel_ctr,
    }
    print(f'Extracted {N:,} segments | shape {segments.shape}')
    return segments, meta


def standardize_segments(segs):
    """Two-stage standardisation (DeAngelis et al.).

    Stage 1 — per-segment temporal mean subtraction:
        Removes absolute position; retains relative motion within the window.
    Stage 2 — cross-segment z-score per (timepoint, variable):
        Zero mean, unit variance at every (t, v) position across all segments.

    Args:
        segs: (N, T, V) float array
    Returns:
        standardised (N, T, V) float64
        stage1_means (N, 1, V), stage2_mean (1, T, V), stage2_std (1, T, V)
    """
    s = segs.astype(np.float64)
    s1_means = s.mean(axis=1, keepdims=True)   # subtract temporal mean per segment
    s -= s1_means
    s2_mean = s.mean(axis=0, keepdims=True)    # z-score across segments
    s2_std  = s.std( axis=0, keepdims=True)
    s = (s - s2_mean) / (s2_std + 1e-8)
    print(f'Standardised: global mean={s.mean():.5f}  std={s.std():.5f}')
    return s, s1_means, s2_mean, s2_std


In [ ]:
# Cell 44 -- Leg-Tip Site Index Guard
# ── Guard: ensure leg_tip_site_indices is valid (handles undefined or all-None) ──
# xpos_egocentric has shape (T, N_ego_sites, 3) indexed by site_names_egocentric.
try:
    _lti_ok = leg_tip_site_indices and not all(v is None for v in leg_tip_site_indices.values())
except NameError:
    _lti_ok = False

if not _lti_ok:
    _ego_names = list(bout_dict['info']['site_names_egocentric'])
    print(f'Recomputing leg_tip_site_indices from site_names_egocentric ({len(_ego_names)} entries)...')
    leg_tip_site_indices = find_leg_tip_site_indices(_ego_names, LEGS)
    print('leg_tip_site_indices:', leg_tip_site_indices)
    if all(v is None for v in leg_tip_site_indices.values()):
        print("\nAll egocentric site names (xpos_egocentric axis-1):")
        for _i, _n in enumerate(_ego_names):
            print(f"  [{_i:3d}] {_n}")

_missing = [l for l, v in leg_tip_site_indices.items() if v is None]
if _missing:
    _ego_names = list(bout_dict['info']['site_names_egocentric'])
    raise RuntimeError(
        f'Still no site found for: {_missing}.\n'
        f'All site_names_egocentric: {_ego_names}'
    )

# ── Extract egocentric segments ──────────────────────────────────────────
segments_raw, seg_meta = sample_segments(
    bout_dict, bout_keys, fly_ids,
    HALF_WIN_FRAMES, N_SEGMENTS_TARGET,
    leg_tip_site_indices, LEGS, SEGMENT_SEED
)
N_seg, T_seg, V_seg = segments_raw.shape

# ── Two-stage standardisation ────────────────────────────────────────────
segments_std, _s1, _s2_mu, _s2_sd = standardize_segments(segments_raw)

# ── Flatten → (N, T*V) and run cuML UMAP ────────────────────────────────
X_seg = segments_std.reshape(N_seg, T_seg * V_seg).astype(np.float32)
print(f'Feature matrix: {X_seg.shape}  (N × T×V)')

reducer_seg = cuml.manifold.UMAP(
    n_components=SEG_UMAP_COMPONENTS,
    n_neighbors=SEG_UMAP_N_NEIGHBORS,
    min_dist=SEG_UMAP_MIN_DIST,
    metric='euclidean',
    random_state=SEGMENT_SEED,
)
umap_seg = np.asarray(reducer_seg.fit_transform(X_seg))
print(f'Full-body segment UMAP: {umap_seg.shape}')


In [ ]:
# Cell 45 -- Full-Body UMAP: Helpers and Figures
# ── Lookup helpers ─────────────────────────────────────────────────────────────
if 'n_legs_stance' in df_valid.columns:
    _stance_lut = df_valid.set_index(['bout_id', 'frame'])['n_legs_stance'].to_dict()
    seg_n_stance = np.array([_stance_lut.get((bid, int(cf)), np.nan)
                              for bid, cf in zip(seg_meta['bout_id'],
                                                  seg_meta['center_frame'])],
                             dtype=np.float32)
else:
    seg_n_stance = np.full(N_seg, np.nan, dtype=np.float32)
stance_vals = np.where(np.isnan(seg_n_stance), 0, seg_n_stance).astype(int)

_valid_sites = [(leg, leg_tip_site_indices[leg])
                for leg in LEGS if leg_tip_site_indices.get(leg) is not None]
seg_mean_z = np.full(N_seg, np.nan, dtype=np.float32)
_ck2, _cx2 = None, None
for i, (b_idx, ctr) in enumerate(zip(seg_meta['bout_idx'], seg_meta['center_frame'])):
    key = bout_keys[b_idx]
    if key != _ck2:
        _ck2 = key
        _cx2 = np.asarray(bout_dict[key]['xpos_egocentric'], dtype=np.float32)
    seg_mean_z[i] = np.mean([_cx2[ctr, si, 2] for _, si in _valid_sites])

_bid_to_fly  = df_valid.groupby('bout_id')['fly_id'].first().to_dict()
fly_id_vals  = np.array([_bid_to_fly.get(bid, 'unknown') for bid in seg_meta['bout_id']])
fly_codes, fly_cats = pd.factorize(fly_id_vals)
n_flies_seg    = len(fly_cats)
seg_cmap_flies = plt.cm.get_cmap('tab10', max(n_flies_seg, 1))

# ── Center-frame value helper (also used by nb[59] Recompute Colors, Cell 46, 47) ───
_seg_lut_jt = {(bid, int(fr)): i
               for i, (bid, fr) in enumerate(
                   zip(df_valid['bout_id'], df_valid['frame']))}

def _seg_col(col_or_arr):
    arr = df_valid[col_or_arr].values if isinstance(col_or_arr, str) \
          else np.asarray(col_or_arr)
    idxs = np.array([_seg_lut_jt.get((bid, int(cf)), -1)
                     for bid, cf in zip(seg_meta['bout_id'],
                                        seg_meta['center_frame'])])
    out = np.full(len(idxs), np.nan)
    valid = idxs >= 0
    out[valid] = arr[idxs[valid]]
    return out

# ── Build color values from COLOR_BY_JT (set in Cell 41) ────────────────────────────
if COLOR_BY_JT == 'cycle_speed':
    _jt_cvals  = _seg_col('step_cycle_mean_speed')
    _jt_clabel = 'Step-cycle mean speed (mm/s)'
    _jt_cmap   = 'turbo'
    _jt_kw     = dict(vmin=float(np.nanmin(_jt_cvals)), vmax=float(np.nanmax(_jt_cvals)))
elif COLOR_BY_JT == 'n_legs_stance':
    _jt_cvals  = stance_vals.astype(float)
    _jt_clabel = 'N legs in stance'
    _jt_cmap   = plt.cm.get_cmap('RdYlGn', 7)
    _jt_kw     = dict(vmin=-0.5, vmax=6.5)
elif COLOR_BY_JT == 'phase_rel':
    _jt_cvals  = _seg_col(f'{PHASE_REL_LEG_JT}_phase_rel')
    _jt_clabel = f'{PHASE_REL_LEG_JT} phase rel. to L1 (rad)'
    _jt_cmap   = 'twilight'
    _jt_kw     = dict(vmin=-np.pi, vmax=np.pi)
elif COLOR_BY_JT == 'phase':
    _jt_cvals  = _seg_col(f'{PHASE_LEG_JT}_phase')
    _jt_clabel = f'{PHASE_LEG_JT} step-cycle phase (rad)'
    _jt_cmap   = 'twilight'
    _jt_kw     = dict(vmin=-np.pi, vmax=np.pi)
else:  # 'instant_vel'
    _jt_cvals  = _seg_col(vel_smooth)
    _jt_clabel = 'Mean |joint velocity| (smoothed)'
    _jt_cmap   = 'turbo'
    _jt_kw     = dict(vmin=float(np.nanmin(_jt_cvals)), vmax=float(np.nanmax(_jt_cvals)))

# ── Figure 1: 3-D scatter (toggle coloring) ──────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
ax  = fig.add_subplot(111, projection='3d')
sc  = ax.scatter(umap_seg[:,0], umap_seg[:,1], umap_seg[:,2],
                 c=_jt_cvals, cmap=_jt_cmap, s=4, alpha=0.5, **_jt_kw)
cb  = plt.colorbar(sc, ax=ax, pad=0.1, shrink=0.6)
cb.set_label(_jt_clabel)
if COLOR_BY_JT == 'n_legs_stance':
    cb.set_ticks(range(7))
    cb.set_ticklabels(range(7))
elif COLOR_BY_JT in ('phase_rel', 'phase'):
    cb.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    cb.set_ticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.set_zlabel('UMAP 3')
ax.set_title(f'Full-body segment UMAP — {_jt_clabel}  (DeAngelis et al.)')
ax.view_init(elev=0, azim=100)
plt.tight_layout()
_fname44_3d = f'seg_umap_3d_{COLOR_BY_JT}.pdf'
plt.savefig(OUTPUT_DIR / _fname44_3d)
plt.show()
print(f'Saved: {OUTPUT_DIR / _fname44_3d}')

# ── Figure 2: 2×2 overview (fixed informational views) ────────────────────────────
_2d_vars = [
    (seg_mean_z,                        'Mean limb Z',   'coolwarm', {}),
    (fly_codes,                         'Fly ID',        seg_cmap_flies,
     dict(vmin=-0.5, vmax=n_flies_seg-0.5)),
    (stance_vals,                       'N legs stance', 'viridis',  dict(vmin=0, vmax=6)),
    (seg_meta['bout_idx'].astype(float),'Bout index',    'tab20',    {}),
]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (c_vals, c_label, c_cmap, kw) in zip(axes.flatten(), _2d_vars):
    sc = ax.scatter(umap_seg[:,0], umap_seg[:,1],
                    c=c_vals, cmap=c_cmap, s=3, alpha=0.4, **kw)
    plt.colorbar(sc, ax=ax, label=c_label, fraction=0.046, pad=0.04)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title(f'Segment UMAP — {c_label}')
    sns.despine(ax=ax)
fig.suptitle('Segment UMAP 2D overview (DeAngelis et al.)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'seg_umap_2d_panels.pdf', bbox_inches='tight')
plt.show()

# ── Diagnostics ──────────────────────────────────────────────────────────────────────────
print(f'\n=== Segment UMAP diagnostics ===')
print(f'N segments : {N_seg:,}  |  window {T_seg} frames × {V_seg} vars = {T_seg*V_seg} dims')
for fid in sorted(np.unique(fly_id_vals)):
    n = int((fly_id_vals == fid).sum())
    print(f'  {fid}: {n:,}  ({100*n/N_seg:.1f}%)')
if not np.all(np.isnan(seg_n_stance)):
    for k in range(7):
        print(f'  {k} legs stance: {int((stance_vals==k).sum()):,}')


In [ ]:
# Cell 46 -- Sample Segments qpos Function
def sample_segments_qpos(bout_dict, bout_keys, seg_meta, joint_list):
    """Extract qpos segments for all joints, reusing existing centre frames.

    Extracts from bout_dict[key]['qpos'] using the same (bout_idx, center_frame)
    already sampled in seg_meta — ensures all UMAP analyses are comparable.

    Returns:
        qpos_segs : (N, WIN_LENGTH, N_joints)  float32
        col_names : list of '{leg}_{joint}' strings  (axis-2 labels)
    """
    N        = len(seg_meta['center_frame'])
    n_joints = len(joint_list)
    qpos_segs = np.empty((N, WIN_LENGTH, n_joints), dtype=np.float32)
    col_names = [f'{leg}_{joint}' for leg, joint, _ in joint_list]

    _key, _qpos = None, None
    for i, (b_idx, ctr) in enumerate(
            zip(seg_meta['bout_idx'], seg_meta['center_frame'])):
        key = bout_keys[b_idx]
        if key != _key:
            _key  = key
            _qpos = np.asarray(bout_dict[key]['qpos'], dtype=np.float32)
        start, end = ctr - HALF_WIN_FRAMES, ctr + HALF_WIN_FRAMES + 1
        for j, (leg, joint, qpos_idx) in enumerate(joint_list):
            qpos_segs[i, :, j] = _qpos[start:end, qpos_idx]

    print(f'qpos segments: {qpos_segs.shape}  '
          f'({N:,} × {WIN_LENGTH} × {n_joints} joints)')
    print(f'Memory: ~{qpos_segs.nbytes/1e9:.2f} GB (float32)')
    return qpos_segs, col_names


qpos_segs, qpos_col_names = sample_segments_qpos(
    bout_dict, bout_keys, seg_meta, joint_list
)


In [ ]:
# Cell 47 -- Recompute Colors
# ── Recompute Colors — change COLOR_BY_JT in Cell 41, then re-run this cell ──────
# Override here if you want a quick change without re-running Cell 41:
# COLOR_BY_JT = 'n_legs_stance'

_seg_lut_jt = {(bid, int(fr)): i
               for i, (bid, fr) in enumerate(
                   zip(df_valid['bout_id'], df_valid['frame']))}

def _seg_col(col_or_arr):
    arr = df_valid[col_or_arr].values if isinstance(col_or_arr, str) \
          else np.asarray(col_or_arr)
    idxs = np.array([_seg_lut_jt.get((bid, int(cf)), -1)
                     for bid, cf in zip(seg_meta['bout_id'],
                                        seg_meta['center_frame'])])
    out = np.full(len(idxs), np.nan)
    valid = idxs >= 0
    out[valid] = arr[idxs[valid]]
    return out

if COLOR_BY_JT == 'cycle_speed':
    _jt_cvals  = _seg_col('step_cycle_mean_speed')
    _jt_clabel = 'Step-cycle mean speed (mm/s)'
    _jt_cmap   = 'turbo'
    _jt_kw     = dict(vmin=float(np.nanmin(_jt_cvals)), vmax=float(np.nanmax(_jt_cvals)))
elif COLOR_BY_JT == 'n_legs_stance':
    _jt_cvals  = stance_vals.astype(float)
    _jt_clabel = 'N legs in stance'
    _jt_cmap   = 'RdYlGn'
    _jt_kw     = dict(vmin=0, vmax=6)
elif COLOR_BY_JT == 'phase_rel':
    _jt_cvals  = _seg_col(f'{PHASE_REL_LEG_JT}_phase_rel')
    _jt_clabel = f'{PHASE_REL_LEG_JT} phase rel. to L1 (rad)'
    _jt_cmap   = 'twilight'
    _jt_kw     = dict(vmin=-np.pi, vmax=np.pi)
elif COLOR_BY_JT == 'phase':
    _jt_cvals  = _seg_col(f'{PHASE_LEG_JT}_phase')
    _jt_clabel = f'{PHASE_LEG_JT} step-cycle phase (rad)'
    _jt_cmap   = 'twilight'
    _jt_kw     = dict(vmin=-np.pi, vmax=np.pi)
else:  # 'instant_vel'
    _jt_cvals  = _seg_col(vel_smooth)
    _jt_clabel = 'Mean |joint velocity| (smoothed)'
    _jt_cmap   = 'turbo'
    _jt_kw     = dict(vmin=float(np.nanmin(_jt_cvals)), vmax=float(np.nanmax(_jt_cvals)))

# ── Ordered unique joint types ─────────────────────────────────────────────────────────────


In [ ]:
# Cell 48 -- Per-(Leg × Joint) UMAP: Run + Plot
# COLOR_BY_JT set in Cell 41 (or overridden in Recompute Colors cell);
# _jt_cvals/_jt_clabel/_jt_cmap/_jt_kw from Cell 44 or Recompute Colors.

joint_types = list(dict.fromkeys(jt for _, jt, _ in joint_list))

# Run one 2-D UMAP per (leg, joint)
umap_joints = {}
for j_idx, (leg, joint, _) in enumerate(joint_list):
    raw_1d  = qpos_segs[:, :, j_idx]                # (N, T)
    raw_3d  = raw_1d[:, :, np.newaxis]              # (N, T, 1)
    std_3d, *_ = standardize_segments(raw_3d)
    X_j = std_3d.reshape(N_seg, WIN_LENGTH).astype(np.float32)
    reducer_j = cuml.manifold.UMAP(
        n_components=2,
        n_neighbors=SEG_UMAP_N_NEIGHBORS,
        min_dist=SEG_UMAP_MIN_DIST,
        metric='euclidean',
        random_state=SEGMENT_SEED,
    )
    umap_joints[(leg, joint)] = np.asarray(reducer_j.fit_transform(X_j))
    print(f'  {leg} / {joint}: done')

# ── Grid plot: rows=legs, cols=joint_types ─────────────────────────────────────────
fig, axes = plt.subplots(
    len(LEGS), len(joint_types),
    figsize=(3.2 * len(joint_types), 3.2 * len(LEGS)),
    squeeze=False
)
sc_last = None
for r, leg in enumerate(LEGS):
    for c, joint in enumerate(joint_types):
        ax = axes[r, c]
        if (leg, joint) not in umap_joints:
            ax.axis('off'); continue
        emb = umap_joints[(leg, joint)]
        sc_last = ax.scatter(emb[:, 0], emb[:, 1], c=_jt_cvals, cmap=_jt_cmap,
                             s=2, alpha=0.35, **_jt_kw)
        if r == 0:
            ax.set_title(JOINT_LABELS.get(joint, joint), fontsize=9,
                         fontweight='bold')
        if c == 0:
            ax.set_ylabel(LEG_LABELS.get(leg, leg), fontsize=9,
                          fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        sns.despine(ax=ax, left=True, bottom=True)
if sc_last is not None:
    cb = fig.colorbar(sc_last, ax=axes[:, -1], label=_jt_clabel,
                      fraction=0.03, pad=0.02)
    if COLOR_BY_JT == 'n_legs_stance':
        cb.set_ticks(range(7))
    elif COLOR_BY_JT in ('phase_rel', 'phase'):
        cb.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        cb.set_ticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
fig.suptitle(f'Per-(leg × joint) segment UMAP — {_jt_clabel}', fontsize=13)
plt.tight_layout()
_jt46_fname = f'umap_per_joint_grid_{COLOR_BY_JT}.pdf'
plt.savefig(OUTPUT_DIR / _jt46_fname, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR / _jt46_fname}')


In [ ]:
# Cell 49 -- Per-Joint UMAP (3-D)
# Uses COLOR_BY_JT, _jt_cvals, _jt_clabel, _jt_cmap, _jt_kw from Cell 46.
# Re-run Cell 46 first to change coloring, or override the toggle here.

# ── Run UMAP for each joint type ─────────────────────────────────────────────────────────
umap_joint_types = {}
for joint in joint_types:
    cols = [i for i, (_, jt, _) in enumerate(joint_list) if jt == joint]
    raw  = qpos_segs[:, :, cols]                    # (N, T, n_legs)
    std, *_ = standardize_segments(raw)
    X_jt = std.reshape(N_seg, WIN_LENGTH * len(cols)).astype(np.float32)
    reducer_jt = cuml.manifold.UMAP(
        n_components=6,
        n_neighbors=SEG_UMAP_N_NEIGHBORS,
        min_dist=SEG_UMAP_MIN_DIST,
        metric='euclidean',
        random_state=SEGMENT_SEED,
    )
    umap_joint_types[joint] = np.asarray(reducer_jt.fit_transform(X_jt))
    print(f'  {joint}: {umap_joint_types[joint].shape}')



In [ ]:
# Cell 50 -- Per-Joint-Type UMAP: Plot
# Re-run this cell after changing COLOR_BY_JT and re-running Recompute Colors.
import math

# ── Plot: one 3-D scatter per joint type ──────────────────────────────────────────
n_jt = len(joint_types)
n_cols = math.ceil(n_jt / 4)
fig = plt.figure(figsize=(8 * n_cols, 13))
for r, joint in enumerate(joint_types):
    emb = umap_joint_types[joint]
    ax  = fig.add_subplot(4, n_cols, r + 1, projection='3d')
    sc  = ax.scatter(emb[:, 0], emb[:, 1], emb[:, 2],
                     c=_jt_cvals, cmap=_jt_cmap, s=3, alpha=0.4, **_jt_kw)
    cb  = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.08, shrink=0.6)
    cb.set_label(_jt_clabel, fontsize=8)
    if COLOR_BY_JT == 'n_legs_stance':
        cb.set_ticks(range(7))
    elif COLOR_BY_JT in ('phase_rel', 'phase'):
        cb.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
        cb.set_ticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
    ax.set_title(JOINT_LABELS.get(joint, joint), fontsize=11, fontweight='bold')
    ax.set_xlabel('UMAP 1', fontsize=7)
    ax.set_ylabel('UMAP 2', fontsize=7)
    ax.set_zlabel('UMAP 3', fontsize=7)
    ax.view_init(elev=0, azim=100)
    ax.tick_params(labelsize=6)

fig.suptitle(f'Per-joint-type UMAP (3-D) — {_jt_clabel}', fontsize=13, fontweight='bold')
plt.tight_layout()
_jt47_fname = f'umap_per_joint_type_3d_{COLOR_BY_JT}.pdf'
plt.savefig(OUTPUT_DIR / _jt47_fname, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR / _jt47_fname}')


## Section 11 — Step-Cycle Joint-Angle UMAP

Each segment = one complete **T1_left step cycle** (biologically grounded, already computed in Cell 11).
Joint angles from `JOINT_SETS['main']` (6 legs × 7 joints) are resampled to `N_PHASE_BINS`
evenly-spaced phase-grid points, making fast and slow cycles directly comparable.

Complements the DeAngelis-style random-window UMAP (Section 10) above.

In [ ]:
# Cell 51 -- Step-Cycle UMAP: Configuration
# ── Segment definition ────────────────────────────────────────────────────────
N_PHASE_BINS     = 50      # resample each step cycle to this many evenly-spaced points
JOINT_SET_NAME   = 'main'  # 'core' | 'main' | 'full'  (see Cell 3 for contents)
MIN_CYCLE_FRAMES = 30      # skip cycles shorter than this (≈37 ms at 800 Hz)
N_CYCLES_MAX     = None    # set an int to cap cycles; None = use all

# ── UMAP parameters ───────────────────────────────────────────────────────────
SC_UMAP_COMPONENTS  = 6
SC_UMAP_N_NEIGHBORS = 50
SC_UMAP_MIN_DIST    = 0.2
SEGMENT_SEED_SC     = 42


In [ ]:
# Cell 52 -- Step-Cycle UMAP: Sampling Function
def sample_step_cycle_segments(df, joint_names, legs, n_phase_bins, min_cycle_frames):
    """Extract per-step-cycle feature vectors (Hilbert-phase-resampled joint angles).

    Each T1_left step cycle (mid-swing to mid-swing, from swing_onsets_from_hilbert)
    is resampled to n_phase_bins evenly-spaced Hilbert-phase points.
    All 6 legs' joint angles are extracted at those same phase-grid positions,
    preserving inter-leg coordination structure.

    Phase convention: phase_grid = [0, 2π) corresponds to Hilbert phase [-π, π):
      grid=0   → raw Hilbert phase -π  (mid-stance)
      grid=π/2 → raw Hilbert phase -π/2 (liftoff / swing onset)
      grid=π   → raw Hilbert phase  0  (mid-swing, peak speed)
      grid=3π/2→ raw Hilbert phase  π/2 (touchdown)
      grid=2π  → raw Hilbert phase  π  (mid-stance)

    Parameters
    ----------
    df               : DataFrame with {leg}_phase and step_cycle_mean_speed columns.
    joint_names      : list of joint suffix strings (e.g., from JOINT_SETS['main'])
    legs             : ordered list of leg names (LEGS)
    n_phase_bins     : int — number of phase-grid points per cycle
    min_cycle_frames : int — skip cycles shorter than this

    Returns
    -------
    X    : float32 array, shape (N_cycles, n_phase_bins * len(legs) * len(joint_names))
    meta : list of dicts with keys fly_id, bout_id, speed, n_legs_stance, duration
    """
    phase_grid    = np.linspace(0.0, 2 * np.pi, n_phase_bins, endpoint=False)
    X_list, meta  = [], []
    ref_phase_col = f"{REFERENCE_LEG}_phase"
    sc_spd_col    = 'step_cycle_mean_speed'
    n_stance_col  = 'n_legs_stance'

    for bid, grp in df.groupby('bout_id', sort=False):
        fid = grp['fly_id'].iloc[0]
        if ref_phase_col not in grp.columns:
            continue

        ref_phase = grp[ref_phase_col].values
        onsets    = swing_onsets_from_hilbert(ref_phase)
        if len(onsets) < 2:
            continue

        grp         = grp.reset_index(drop=True)
        ref_phase   = grp[ref_phase_col].values  # re-read after reset_index
        vals_spd    = grp[sc_spd_col].values   if sc_spd_col   in grp.columns else None
        vals_stance = grp[n_stance_col].values if n_stance_col in grp.columns else None

        for i in range(len(onsets) - 1):
            t0, t1 = int(onsets[i]), int(onsets[i + 1])
            dur    = t1 - t0
            if dur < min_cycle_frames:
                continue

            cyc = grp.iloc[t0:t1]

            # Shift Hilbert phase from [-π, π] to [0, 2π] for circular interpolation
            seg_phase = ref_phase[t0:t1] + np.pi

            seg = []
            for leg in legs:
                for jnt in joint_names:
                    col = f"{leg}_{jnt}"
                    if col in cyc.columns:
                        raw  = cyc[col].values.astype(float)
                        nans = np.isnan(raw)
                        if nans.all():
                            raw = np.zeros(dur)
                        elif nans.any():
                            raw[nans] = np.interp(
                                np.flatnonzero(nans),
                                np.flatnonzero(~nans),
                                raw[~nans],
                            )
                    else:
                        raw = np.zeros(dur)

                    valid = np.isfinite(seg_phase) & np.isfinite(raw)
                    if valid.sum() < 5:
                        seg.append(np.zeros(n_phase_bins))
                    else:
                        seg.append(np.interp(
                            phase_grid,
                            seg_phase[valid],
                            raw[valid],
                            period=2 * np.pi,
                        ))
            X_list.append(np.concatenate(seg))

            mid = (t0 + t1) // 2
            meta.append({
                'fly_id':        fid,
                'bout_id':       bid,
                'speed':         float(vals_spd[mid])    if vals_spd    is not None else np.nan,
                'n_legs_stance': float(vals_stance[mid]) if vals_stance is not None else np.nan,
                'duration':      dur,
            })

    if not X_list:
        raise RuntimeError(
            "No step cycles found. Ensure preprocessing cells have been run and "
            f"{ref_phase_col} is populated in df."
        )
    return np.array(X_list, dtype=np.float32), meta


In [ ]:
# Cell 53 -- Step-Cycle UMAP: Standardize + Run
JOINT_NAMES_SC = JOINT_SETS[JOINT_SET_NAME]
n_legs_sc      = len(LEGS)
n_joints_sc    = len(JOINT_NAMES_SC)
n_feat_sc      = N_PHASE_BINS * n_legs_sc * n_joints_sc

print(f"Joint set '{JOINT_SET_NAME}': {JOINT_NAMES_SC}")
print(f"Feature dims: {N_PHASE_BINS} bins × {n_legs_sc} legs × {n_joints_sc} joints = {n_feat_sc}")
print()

X_raw_sc, sc_meta = sample_step_cycle_segments(
    df, JOINT_NAMES_SC, LEGS,
    n_phase_bins=N_PHASE_BINS,
    min_cycle_frames=MIN_CYCLE_FRAMES,
)
print(f"Step cycles extracted: {len(X_raw_sc)}")

# Optional cap
if N_CYCLES_MAX is not None and len(X_raw_sc) > N_CYCLES_MAX:
    _rng_idx = np.random.default_rng(SEGMENT_SEED_SC).choice(
        len(X_raw_sc), N_CYCLES_MAX, replace=False)
    X_raw_sc = X_raw_sc[_rng_idx]
    sc_meta  = [sc_meta[i] for i in _rng_idx]
    print(f"  Capped to {N_CYCLES_MAX} cycles.")

# ── Stage 1: per-cycle DC removal ─────────────────────────────────────────────
# Reshape to (N, n_legs*n_joints, N_PHASE_BINS), subtract phase-axis mean, reshape back
_n_cyc_sc = len(X_raw_sc)
X_3d_sc   = X_raw_sc.reshape(_n_cyc_sc, n_legs_sc * n_joints_sc, N_PHASE_BINS)
X_dc_sc   = (X_3d_sc - X_3d_sc.mean(axis=2, keepdims=True)).reshape(_n_cyc_sc, n_feat_sc)

# ── Stage 2: cross-cycle z-score per feature ──────────────────────────────────
_mu_sc  = X_dc_sc.mean(axis=0)
_sd_sc  = X_dc_sc.std(axis=0) + 1e-8
X_z_sc  = ((X_dc_sc - _mu_sc) / _sd_sc).astype(np.float32)

# ── Metadata arrays ───────────────────────────────────────────────────────────
sc_speeds    = np.array([m['speed']          for m in sc_meta])
sc_stances   = np.array([m['n_legs_stance']  for m in sc_meta])
sc_durations = np.array([m['duration']       for m in sc_meta])
sc_fly_ids   = np.array([m['fly_id']         for m in sc_meta])

print(f"Speed range:    {np.nanmin(sc_speeds):.1f}–{np.nanmax(sc_speeds):.1f} mm/s")
print(f"Duration range: {sc_durations.min()}–{sc_durations.max()} frames "
      f"({sc_durations.min()/FPS*1000:.0f}–{sc_durations.max()/FPS*1000:.0f} ms)")
print()

# ── UMAP ─────────────────────────────────────────────────────────────────────
print("Running cuML UMAP …")
_sc_reducer = cuml.manifold.UMAP(
    n_components=SC_UMAP_COMPONENTS,
    n_neighbors=SC_UMAP_N_NEIGHBORS,
    min_dist=SC_UMAP_MIN_DIST,
    metric='euclidean',
    random_state=SEGMENT_SEED_SC,
)
umap_sc = np.asarray(_sc_reducer.fit_transform(X_z_sc))
print(f"Embedding shape: {umap_sc.shape}")


In [ ]:
# Cell 54 -- Step-Cycle UMAP: Visualisation
_sc_fly_ids_u  = sorted(set(sc_fly_ids))
_sc_n_flies    = len(_sc_fly_ids_u)
_sc_fly_codes  = np.array([_sc_fly_ids_u.index(f) for f in sc_fly_ids])
_sc_cmap_flies = plt.cm.get_cmap('tab20', _sc_n_flies)

_sc_color_specs = [
    (sc_speeds,      'Step-cycle speed (mm/s)', 'viridis', {}),
    (sc_stances,     'N legs in stance',         'RdYlGn',  dict(vmin=0, vmax=6)),
    (_sc_fly_codes,  'Fly ID',                   _sc_cmap_flies,
     dict(vmin=-0.5, vmax=_sc_n_flies - 0.5)),
    (sc_durations,   'Cycle duration (frames)',   'plasma',  {}),
]

# ── 3-D scatter: 1 row × 4 colourings ────────────────────────────────────────
if SC_UMAP_COMPONENTS >= 3:
    _fig_sc3 = plt.figure(figsize=(22, 5))
    for _ci, (_cvals, _clabel, _ccmap, _ckw) in enumerate(_sc_color_specs):
        _ax = _fig_sc3.add_subplot(1, 4, _ci + 1, projection='3d')
        _sc3 = _ax.scatter(
            umap_sc[:, 0], umap_sc[:, 1], umap_sc[:, 2],
            c=_cvals, cmap=_ccmap, s=8, alpha=0.4, **_ckw
        )
        _cb = plt.colorbar(_sc3, ax=_ax, fraction=0.04, pad=0.06, shrink=0.6)
        if 'stance' in _clabel:
            _cb.set_ticks(range(7))
        _cb.set_label(_clabel, fontsize=8)
        _ax.set_title(_clabel, fontsize=9)
        _ax.set_xlabel('UMAP 1', fontsize=7)
        _ax.set_ylabel('UMAP 2', fontsize=7)
        _ax.set_zlabel('UMAP 3', fontsize=7)
        _ax.tick_params(labelsize=6)
    _fig_sc3.suptitle(
        f"Step-cycle UMAP  |  '{JOINT_SET_NAME}' joints ({n_joints_sc})"
        f"  ×  {N_PHASE_BINS} phase bins  ×  6 legs"
        f"\n{len(umap_sc)} step cycles",
        fontsize=11, fontweight='bold',
    )
    plt.tight_layout()
    _pdf_sc3 = OUTPUT_DIR / f'umap_step_cycle_{JOINT_SET_NAME}.pdf'
    plt.savefig(_pdf_sc3, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_pdf_sc3}")

# ── 2-D projection pairs ──────────────────────────────────────────────────────
_pairs2d = [(0, 1, 'UMAP 1', 'UMAP 2'), (0, 2, 'UMAP 1', 'UMAP 3'), (1, 2, 'UMAP 2', 'UMAP 3')]
_fig_sc2, _axes_sc2 = plt.subplots(
    len(_sc_color_specs), len(_pairs2d),
    figsize=(5.5 * len(_pairs2d), 4.5 * len(_sc_color_specs)),
    squeeze=False,
)
for _ri, (_cvals, _clabel, _ccmap, _ckw) in enumerate(_sc_color_specs):
    for _ci, (_xi, _yi, _xl, _yl) in enumerate(_pairs2d):
        _ax2 = _axes_sc2[_ri, _ci]
        _sc2 = _ax2.scatter(
            umap_sc[:, _xi], umap_sc[:, _yi],
            c=_cvals, cmap=_ccmap, s=10, alpha=0.4, **_ckw
        )
        _cb2 = plt.colorbar(_sc2, ax=_ax2, fraction=0.04, pad=0.02)
        if 'stance' in _clabel:
            _cb2.set_ticks(range(7))
        if _ci == 0:
            _cb2.set_label(_clabel, fontsize=8)
        _ax2.set_xlabel(_xl, fontsize=8)
        _ax2.set_ylabel(_yl, fontsize=8)
        _ax2.set_xticks([])
        _ax2.set_yticks([])
        sns.despine(ax=_ax2, left=True, bottom=True)
        if _ri == 0:
            _ax2.set_title(f'{_xl} vs {_yl}', fontsize=9, fontweight='bold')
_fig_sc2.suptitle(
    f"Step-cycle UMAP — 2-D projections  |  '{JOINT_SET_NAME}' joints",
    fontsize=11, fontweight='bold',
)
plt.tight_layout()
_pdf_sc2 = OUTPUT_DIR / f'umap_step_cycle_{JOINT_SET_NAME}_2d.pdf'
plt.savefig(_pdf_sc2, bbox_inches='tight')
plt.show()
print(f"Saved: {_pdf_sc2}")


# FIG 2 

In [ ]:
# Cell 55 -- Paper Figure 2: Configuration
# ── Bout and window ───────────────────────────────────────────────────────────
FIG2_BOUT        = 'bout_249'
FIG2_JOINT_SET   = 'main'   # 'core' | 'main' | 'full'
FIG2_N_CYCLES    = 2      # number of T1_left step cycles to display
FIG2_CYCLE_START = 4       # 0-indexed: skip first couple of cycles (edge-artifact risk)

# ── Y-axis sharing ────────────────────────────────────────────────────────────
FIG2_SHARE_COL_Y = True     # share y-axis per joint column across T1/T2/T3 rows

# ── Colors ────────────────────────────────────────────────────────────────────
FIG2_TIER_COLORS = {'T1': '#d62728', 'T2': '#1f77b4', 'T3': '#2ca02c'}
FIG2_LEG_COLORS  = {
    'T1_left':  '#d62728', 'T1_right': '#ff7f0e',
    'T2_left':  '#1f77b4', 'T2_right': '#9467bd',
    'T3_left':  '#2ca02c', 'T3_right': '#8c564b',
}


# ── Angle panel ─────────────────────────────────────────────────────────────
FIG2_ANGLE_LEG   = 'T1_left'   # which single leg to show in the angle panel

In [ ]:
# Cell 56 -- Paper Figure 2: Walking Dynamics Overview
import math
import matplotlib.ticker as mticker
import matplotlib.transforms as mtransforms
import matplotlib.font_manager as _fm

# -- 1. Extract bout window ---------------------------------------------------
_grp = df[df['bout_id'] == FIG2_BOUT].reset_index(drop=True)
assert len(_grp) > 0, f"{FIG2_BOUT!r} not found in df"

_ref_phase_col = f"{REFERENCE_LEG}_phase"
_ref_onsets = liftoff_from_hilbert(_grp[_ref_phase_col].values)
assert len(_ref_onsets) >= FIG2_CYCLE_START + FIG2_N_CYCLES + 1, (
    f"Not enough step cycles in {FIG2_BOUT!r}: "
    f"found {len(_ref_onsets)-1}, need {FIG2_CYCLE_START + FIG2_N_CYCLES + 1}")

_t0   = int(_ref_onsets[FIG2_CYCLE_START])
_t1   = int(_ref_onsets[FIG2_CYCLE_START + FIG2_N_CYCLES])
_win  = _grp.iloc[_t0:_t1].reset_index(drop=True)
# Mean-center joint angles (same means as df_mc)
_win = _win.copy()
for _mc_col, _mc_mean in _jangle_means.items():
    if _mc_col in _win.columns:
        _win[_mc_col] = _win[_mc_col] - _mc_mean
_T    = len(_win)
_t_ms = np.arange(_T) / FPS * 1000

_cyc_ms = np.array([
    (int(_ref_onsets[FIG2_CYCLE_START + k]) - _t0) / FPS * 1000
    for k in range(FIG2_N_CYCLES + 1)
])

_spd = _win['step_cycle_mean_speed'].median() if 'step_cycle_mean_speed' in _win.columns else np.nan
print(f"Bout {FIG2_BOUT}  |  cycles {FIG2_CYCLE_START}--{FIG2_CYCLE_START+FIG2_N_CYCLES-1}"
      f"  |  {_T} frames = {_T/FPS*1000:.0f} ms  |  speed approx {_spd:.0f} mm/s")

# -- 2. Helpers ---------------------------------------------------------------
_joints = JOINT_SETS[FIG2_JOINT_SET]
_angle_tier = FIG2_ANGLE_LEG.split('_')[0]   # 'T1', 'T2', or 'T3'
_tiers      = [(_angle_tier, FIG2_ANGLE_LEG, None)]   # right leg hidden

def _swing_mask(col):
    if col not in _win.columns:
        return np.zeros(_T, dtype=bool)
    return _win[col].fillna(1).values == 0

def _add_cycle_nums(ax):
    for k in range(_cyc_ms.size - 1):
        mid = (_cyc_ms[k] + _cyc_ms[k + 1]) / 2
        ax.text(mid, 0.97, str(k + 1),
                transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=6,
                color='#555555', fontweight='bold')

# -- 3. Stacked offsets (coxa_abduction at top, tarsus at bottom) ---------------
_OFFSETS = {
    'coxa_abduct':  7.2,
    'coxa_twist':   6.2,
    'coxa':         5.2,
    'femur_twist':  3.8,
    'femur':        2.8,
    'tibia':        1.4,
    'tarsus':       0.0,
}
AMP_HALF = 0.42

def _normalise(arr_deg, combined):
    mu  = np.nanmean(combined)
    rng = np.nanmax(combined) - np.nanmin(combined)
    if rng < 1e-3:
        rng = 1.0
    return (arr_deg - mu) / rng * 2 * AMP_HALF, mu, rng

def _round5(x):
    return int(round(x / 5.0) * 5)

# -- 4. Build figure ----------------------------------------------------------
import matplotlib as _mpl
_fm.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 6,
    'axes.labelsize': 6, 'axes.titlesize': 6,
    'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'pdf.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2,    'ytick.major.size': 2,
})

fig2 = plt.figure(figsize=(7.87, 5.07))
gs_outer = fig2.add_gridspec(2, 1, height_ratios=[1.0, 3.5], hspace=0.08)

ax_z = fig2.add_subplot(gs_outer[0])

# Single angle panel for the selected leg
ax_ang    = fig2.add_subplot(gs_outer[1], sharex=ax_z)
_leg_axes = [ax_ang]

# -- 5. Z row -----------------------------------------------------------------
_Z_HIGH = {'T1_left', 'T2_right', 'T3_left'}
for _leg in LEGS:
    _z_col = f"{_leg}_tip_z_world"
    if _z_col not in _win.columns:
        continue
    _z_alpha = 0.68 if _leg in _Z_HIGH else 0.25
    ax_z.plot(_t_ms, _win[_z_col].values * 10,
              color='k', lw=1.1, alpha=_z_alpha,
              label=LEG_LABELS.get(_leg, _leg))

for _leg in ('T1_left', 'T1_right'):
    _sw = _swing_mask(f"{_leg}_swing_stance")
    ax_z.fill_between(_t_ms, 0, 1, where=_sw,
                      color=FIG2_LEG_COLORS[_leg], alpha=0.13, linewidth=0,
                      transform=ax_z.get_xaxis_transform(), zorder=0)

for _ms in _cyc_ms[1:-1]:
    ax_z.axvline(_ms, color='#bbbbbb', lw=0.7, linestyle=':', zorder=0)

_add_cycle_nums(ax_z)
ax_z.set_ylabel('Z (mm)')
ax_z.legend(loc='upper right', ncol=6, frameon=False,
            columnspacing=0.6, handlelength=1.2)
ax_z.tick_params(axis='x', labelbottom=False)
ax_z.xaxis.set_major_locator(mticker.MultipleLocator(100))
sns.despine(ax=ax_z)

# -- 6. Stacked-trace panels --------------------------------------------------
# Store (mu, rng, p5_d, p95_d, offset) per (panel_idx, jnt) for tick labelling
_norm_params = {}

for panel_idx, (ax, (tier, leg_L, leg_R)) in enumerate(zip(_leg_axes, _tiers)):
    _clr  = FIG2_TIER_COLORS[tier]
    _sw_L = _swing_mask(f"{leg_L}_swing_stance")

    for jnt in _joints:
        off   = _OFFSETS.get(jnt, 0.0)
        col_L = f"{leg_L}_{jnt}"

        deg_L = np.degrees(_win[col_L].values.astype(float)) \
                if col_L in _win.columns else np.full(_T, np.nan)

        if leg_R is not None:
            col_R = f"{leg_R}_{jnt}"
            deg_R = np.degrees(_win[col_R].values.astype(float)) \
                    if col_R in _win.columns else np.full(_T, np.nan)
            combined = np.concatenate([
                deg_L[np.isfinite(deg_L)],
                deg_R[np.isfinite(deg_R)],
            ])
        else:
            combined = deg_L[np.isfinite(deg_L)]

        if combined.size == 0:
            combined = np.array([0.0, 1.0])

        y_L, mu, rng = _normalise(deg_L, combined)
        _p5  = _round5(np.nanpercentile(combined, 5))
        _p95 = _round5(np.nanpercentile(combined, 95))
        _norm_params[(panel_idx, jnt)] = (mu, rng, _p5, _p95, off)

        ax.plot(_t_ms, y_L + off, color=_clr, lw=1.0)
        if leg_R is not None:
            y_R, _, _ = _normalise(deg_R, combined)
            ax.plot(_t_ms, y_R + off, color=_clr, lw=1.0, linestyle='--', alpha=0.7)

    ax.fill_between(_t_ms, 0, 1, where=_sw_L,
                    color=_clr, alpha=0.10, linewidth=0,
                    transform=ax.get_xaxis_transform(), zorder=0)
    for _ms in _cyc_ms[1:-1]:
        ax.axvline(_ms, color='#cccccc', lw=0.5, linestyle=':', zorder=0)

    _add_cycle_nums(ax)
    ax.set_title(LEG_LABELS.get(leg_L, leg_L), pad=3)
    ax.set_xlabel('Time (ms)')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(100))
    ax.tick_params(axis='x', rotation=30)
    sns.despine(ax=ax)

# -- 7. Y-axis: per-joint p5/p95 ticks + joint names on left margin ----------
_ylo = min(_OFFSETS.values()) - AMP_HALF - 0.35
_yhi = max(_OFFSETS.values()) + AMP_HALF + 0.35

for panel_idx, ax in enumerate(_leg_axes):
    ax.set_ylim(_ylo, _yhi)

    ytick_pos, ytick_lab = [], []
    for jnt in _joints:
        mu, rng, p5_d, p95_d, off = _norm_params[(panel_idx, jnt)]
        for d in [p5_d, p95_d]:
            y = (d - mu) / rng * 2 * AMP_HALF + off
            if off - AMP_HALF * 1.1 <= y <= off + AMP_HALF * 1.1:
                ytick_pos.append(y)
                ytick_lab.append(f"{d}°")

    ax.set_yticks(ytick_pos)
    ax.set_yticklabels(ytick_lab)
    ax.tick_params(axis='y', length=2, pad=1)

    if panel_idx == 0:
        ax.set_ylabel('Angle (°)')

# Joint name annotations on left margin of angle panel
_trans_ang = mtransforms.blended_transform_factory(ax_ang.transAxes, ax_ang.transData)
for jnt in _joints:
    off = _OFFSETS.get(jnt, 0.0)
    ax_ang.text(-0.02, off, jnt.replace('_', ' '),
                transform=_trans_ang,
                ha='right', va='center', color='#444444')

# -- 8. Title and save --------------------------------------------------------
fig2.suptitle(
    f"Walking dynamics  |  {FIG2_BOUT}  |  {FIG2_N_CYCLES} step cycles"
    f"  ({_T/FPS*1000:.0f} ms)  |  median speed approx {_spd:.0f} mm/s",
    fontweight='bold', y=1.01
)
plt.tight_layout()
_fig2_path = OUTPUT_DIR / f'fig2_walking_dynamics_{FIG2_JOINT_SET}.pdf'
plt.savefig(_fig2_path, bbox_inches='tight')
plt.show()
print(f"Saved: {_fig2_path}")

## Section 12 — Speed–ROM Correlation

Three complementary views of which joint ranges of motion scale with walking speed:

- **Approach A (Cell 57):** Per-step-cycle ROM × speed Pearson correlation heatmap
- **Approach B (Cell 58):** Speed-binned phase-averaged joint angle profiles
- **Approach C (Cell 59):** PCA circle loading decomposition (PC1²+PC2² contribution per joint)
- **Cell 60:** Combined paper figure

Requires Cells 50–52 (step-cycle UMAP) to have run before Cell 58.

In [ ]:
# Cell 57 -- Speed-ROM: Configuration
ROM_JOINT_SET        = 'main'   # 'core' | 'main' | 'full'
ROM_MIN_CYCLE_FRAMES = 10       # skip step cycles shorter than this (frames)
ROM_N_SPEED_BINS     = 3       # number of speed quantile bins for Approach B


In [ ]:
# Cell 58 -- Approach A: Per-cycle ROM x speed correlation
from scipy.stats import pearsonr, spearmanr, ttest_1samp

# ── Toggles ───────────────────────────────────────────────────────────────────
# MERGE_CONTRALATERAL:
#   True  → average L+R ROM per tier before correlating (3 columns: T1/T2/T3)
#   False → each leg separately (6 columns)
MERGE_CONTRALATERAL = True   # <- change here

# STAT_METHOD — choose one:
#   'pearson'     — Pearson r across all strides
#                   Fast; assumes linearity. p-values optimistic (pseudoreplication).
#   'spearman'    — Spearman ρ across all strides
#                   No linearity assumption, robust to outliers. Same caveat on p.
#   'per_fly'     — Pearson r on per-fly mean ROM vs mean speed (N = n flies)
#                   Conservative; removes pseudoreplication but discards
#                   within-fly variance.
#   'within_fly'  — Per-fly Pearson r, averaged via Fisher z-transform.
#                   Uses all stride-level variance but keeps flies as the
#                   unit of inference. p-value from one-sample t-test on z scores
#                   (no external libraries needed).
STAT_METHOD = 'within_fly'   # <- change here

_rom_joints = JOINT_SETS[ROM_JOINT_SET]
_n_rj       = len(_rom_joints)
_TIER_PAIRS = [('T1', 'T1_left', 'T1_right'),
               ('T2', 'T2_left', 'T2_right'),
               ('T3', 'T3_left', 'T3_right')]

# -- Build per-cycle ROM DataFrame -------------------------------------------
_sc_df = df.dropna(subset=['step_cycle_id']).copy()
_sc_df['step_cycle_id'] = _sc_df['step_cycle_id'].astype(int)

_rows = []
for (bid, scid), cyc in _sc_df.groupby(['bout_id', 'step_cycle_id'], sort=False):
    if len(cyc) < ROM_MIN_CYCLE_FRAMES:
        continue
    row = {
        'bout_id':  bid,
        'speed':    cyc['step_cycle_mean_speed'].iloc[0],
        'duration': len(cyc),
        'fly_id':   cyc['fly_id'].iloc[0],
    }
    for leg in LEGS:
        for jnt in _rom_joints:
            col = f'{leg}_{jnt}'
            v = np.degrees(cyc[col].dropna().values) if col in cyc.columns else np.array([])
            row[col] = float(v.max() - v.min()) if len(v) >= 2 else np.nan
    _rows.append(row)

rom_df = pd.DataFrame(_rows)
print(f'Step cycles: {len(rom_df)}  |  flies: {rom_df["fly_id"].nunique()}  |  '
      f'speed range: {rom_df["speed"].min():.1f}–{rom_df["speed"].max():.1f} mm/s')

# -- Add averaged contralateral columns to rom_df ----------------------------
for tier, leg_L, leg_R in _TIER_PAIRS:
    for jnt in _rom_joints:
        rom_df[f'{tier}_{jnt}'] = rom_df[[f'{leg_L}_{jnt}', f'{leg_R}_{jnt}']].mean(axis=1, skipna=True)

# -- Statistical test dispatcher ---------------------------------------------
def _compute_stat(speed_vals, rom_vals, fly_id_vals, method):
    """Return (effect_size, p_value, n_used) for the chosen method."""
    mask = np.isfinite(speed_vals) & np.isfinite(rom_vals)
    xs, ys, fids = speed_vals[mask], rom_vals[mask], fly_id_vals[mask]
    n = mask.sum()
    if n < 10:
        return np.nan, np.nan, n

    if method == 'pearson':
        r, p = pearsonr(xs, ys)
        return r, p, n

    elif method == 'spearman':
        r, p = spearmanr(xs, ys)
        return r, p, n

    elif method == 'per_fly':
        tmp = pd.DataFrame({'speed': xs, 'rom': ys, 'fly_id': fids})
        fly_means = tmp.groupby('fly_id')[['speed', 'rom']].mean()
        nf = len(fly_means)
        if nf < 4:
            return np.nan, np.nan, nf
        r, p = pearsonr(fly_means['speed'].values, fly_means['rom'].values)
        return r, p, nf

    elif method == 'within_fly':
        # Per-fly Pearson r, averaged via Fisher z-transform.
        # p-value: one-sample t-test on z scores (H0: mean z = 0).
        tmp = pd.DataFrame({'speed': xs, 'rom': ys, 'fly_id': fids})
        z_scores = []
        for fid, grp in tmp.groupby('fly_id'):
            if len(grp) < 5:
                continue
            r_fly, _ = pearsonr(grp['speed'].values, grp['rom'].values)
            # clip to avoid arctanh(±1) = ±inf
            r_fly = np.clip(r_fly, -0.9999, 0.9999)
            z_scores.append(np.arctanh(r_fly))
        nf = len(z_scores)
        if nf < 3:
            return np.nan, np.nan, nf
        z_arr = np.array(z_scores)
        r_avg = np.tanh(z_arr.mean())          # back-transform mean z → r
        _, p  = ttest_1samp(z_arr, 0)          # H0: no within-fly correlation
        return r_avg, p, nf

    else:
        raise ValueError(f'Unknown STAT_METHOD: {method!r}')

# -- Compute effect-size and p matrix ----------------------------------------
if MERGE_CONTRALATERAL:
    _col_labels = [tier for tier, *_ in _TIER_PAIRS]
    _n_cols     = len(_TIER_PAIRS)
    def _get_rom_col(jnt, ci): return f'{_col_labels[ci]}_{jnt}'
else:
    _col_labels = ['L1', 'R1', 'L2', 'R2', 'L3', 'R3']
    _n_cols     = len(LEGS)
    def _get_rom_col(jnt, ci): return f'{LEGS[ci]}_{jnt}'

r_matrix = np.full((_n_rj, _n_cols), np.nan)
p_matrix = np.full((_n_rj, _n_cols), np.nan)
n_matrix = np.zeros((_n_rj, _n_cols), dtype=int)

_spd  = rom_df['speed'].values.astype(float)
_fids = rom_df['fly_id'].values

for ji, jnt in enumerate(_rom_joints):
    for ci in range(_n_cols):
        col = _get_rom_col(jnt, ci)
        rom_vals = rom_df[col].values.astype(float)
        eff, p, n = _compute_stat(_spd, rom_vals, _fids, STAT_METHOD)
        r_matrix[ji, ci] = eff
        p_matrix[ji, ci] = p
        n_matrix[ji, ci] = n

# -- Heatmap ------------------------------------------------------------------
import matplotlib.font_manager as _fm2
import matplotlib.colors as _mcolors
from matplotlib.cm import ScalarMappable
import matplotlib as _mpl2

_fm2.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_mpl2.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                        'axes.labelsize': 6, 'axes.titlesize': 6,
                        'xtick.labelsize': 6, 'ytick.labelsize': 6,
                        'pdf.fonttype': 42})

_jnt_labels = [JOINT_LABELS.get(j, j.replace('_', ' ')) for j in _rom_joints]

_cbar_labels = {
    'pearson':    'Pearson r  (all strides)',
    'spearman':   'Spearman ρ  (all strides)',
    'per_fly':    'Pearson r  (fly means)',
    'within_fly': 'Fisher-z averaged r  (within-fly)',
}
_n_label = {
    'pearson':    'strides', 'spearman': 'strides',
    'per_fly':    'flies',   'within_fly': 'flies',
}
from matplotlib.colors import TwoSlopeNorm

# Slice RdYlGn from 0.25 (light orange) to 1.0 (bright green), skipping red
_rdylgn  = plt.get_cmap('RdYlGn')
_cmap_g  = _mcolors.LinearSegmentedColormap.from_list(
    'OrangeGreen', [_rdylgn(x) for x in np.linspace(0.25, 1.0, 256)])
_norm_div = TwoSlopeNorm(vmin=-0.8, vcenter=0.0, vmax=0.8)

fig_a, ax_a = plt.subplots(figsize=(80 / 25.4, 40 / 25.4))
# im = ax_a.imshow(r_matrix, cmap=_cmap_g, vmin=-0.2, vmax=0.8, aspect='auto')
im = ax_a.imshow(r_matrix, cmap='RdBu_r', norm=_norm_div, aspect='auto') ########### Modified by Elliott
cbar = plt.colorbar(im, ax=ax_a, label=_cbar_labels[STAT_METHOD], fraction=0.05, pad=0.02,norm=_norm_div)
cbar.set_ticks([-0.8, -0.4, 0.0, 0.4, 0.8])
cbar.set_ticklabels(['-0.8', '-0.4', '0.0', '0.4', '0.8'])


for ji in range(_n_rj):
    for ci in range(_n_cols):
        r = r_matrix[ji, ci];  p = p_matrix[ji, ci]
        if np.isnan(r): continue
        stars = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        ax_a.text(ci, ji, f'{r:.2f}{stars}', ha='center', va='center',
                  fontsize=6, color='white' if r > 0.5 else 'black')

if not MERGE_CONTRALATERAL:
    for x in [1.5, 3.5]:
        ax_a.axvline(x, color='white', lw=2)

ax_a.set_xticks(range(_n_cols))
ax_a.set_xticklabels(_col_labels)
ax_a.set_yticks(range(_n_rj))
ax_a.set_yticklabels(_jnt_labels)
ax_a.set_xlabel('Leg group' if MERGE_CONTRALATERAL else 'Leg')
plt.tight_layout()
_pdf_a = OUTPUT_DIR / f'speedrom_A_{ROM_JOINT_SET}_{STAT_METHOD}{"_merged" if MERGE_CONTRALATERAL else ""}.svg'
plt.savefig(_pdf_a, bbox_inches='tight')
plt.show()
print(f'Saved: {_pdf_a}')

# -- Top 10 pairs -------------------------------------------------------------
_pairs = []
for ji, jnt in enumerate(_rom_joints):
    for ci in range(_n_cols):
        r, p = r_matrix[ji, ci], p_matrix[ji, ci]
        if not np.isnan(r):
            _pairs.append((abs(r), r, p, n_matrix[ji, ci], f'{_col_labels[ci]}_{jnt}'))
_pairs.sort(reverse=True)
_effect_name = {'pearson': 'r', 'spearman': 'ρ', 'per_fly': 'r', 'within_fly': 'r̄'}[STAT_METHOD]
print(f'\nTop 10 by |{_effect_name}|  ({STAT_METHOD}, N in {_n_label[STAT_METHOD]}):')
print(f'  {"joint-leg":<28}  {_effect_name:>6}  {"p":>10}  {"N":>5}')
for _, r, p, n, name in _pairs[:10]:
    print(f'  {name:<28}  {r:>6.3f}  {p:>10.2e}  {n:>5}')

In [ ]:
# Cell 59 -- Approach B: Speed-binned phase profiles (degrees)
# Requires: X_raw_sc, sc_speeds, JOINT_SET_NAME, N_PHASE_BINS from Cell 52

import matplotlib.font_manager as _fmp59
import matplotlib as _mpl59
import matplotlib.colors as _mcolors59

_fmp59.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_mpl59.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                         'axes.labelsize': 6, 'axes.titlesize': 6,
                         'xtick.labelsize': 6, 'ytick.labelsize': 6,
                         'pdf.fonttype': 42})

try:
    _ = X_raw_sc
except NameError:
    raise RuntimeError('X_raw_sc not found. Run Cells 50-52 first.')

# -- Setup --------------------------------------------------------------------
_sc_joints = JOINT_SETS[JOINT_SET_NAME]
_n_sj = len(_sc_joints)
_n_sl = len(LEGS)
_n_pb = N_PHASE_BINS

_phase_x = np.linspace(0, 2 * np.pi, _n_pb, endpoint=False)

_bin_edges  = np.nanpercentile(sc_speeds, np.linspace(0, 100, ROM_N_SPEED_BINS + 1))
_bin_labels = [f'{_bin_edges[k]:.0f}–{_bin_edges[k+1]:.0f}'
               for k in range(ROM_N_SPEED_BINS)]
_bin_idx    = np.digitize(sc_speeds, _bin_edges[1:-1])

_X4_deg = np.degrees(X_raw_sc.reshape(len(X_raw_sc), _n_sl, _n_sj, _n_pb))
_X4_deg -= _X4_deg.mean(axis=(0, 3), keepdims=True)

_mean_profiles_deg = np.array([
    _X4_deg[_bin_idx == b].mean(axis=0) for b in range(ROM_N_SPEED_BINS)
])
_count_per_bin = [(_bin_idx == b).sum() for b in range(ROM_N_SPEED_BINS)]
print('Cycles per bin:', dict(zip(_bin_labels, _count_per_bin)))

# -- Mean liftoff phase per leg -----------------------------------------------
_lo_phases = {leg: [] for leg in LEGS}
for bid, grp in df.groupby('bout_id', sort=False):
    grp_r = grp.reset_index(drop=True)
    for leg in LEGS:
        ss_col = f'{leg}_swing_stance'
        ph_col = f'{leg}_phase'
        if ss_col not in grp_r.columns or ph_col not in grp_r.columns:
            continue
        ss  = grp_r[ss_col].fillna(1).values
        ph  = grp_r[ph_col].values
        trans = np.where(np.diff(ss) < 0)[0]
        for t in trans:
            lp = ph[t + 1]
            if np.isfinite(lp):
                _lo_phases[leg].append(lp + np.pi)
_mean_liftoff = {leg: float(np.mean(v)) if v else None
                 for leg, v in _lo_phases.items()}

# -- Color scheme: light blue (slow) → #00109f (fast) ------------------------
_FAST_RGB = (0x00 / 255, 0x10 / 255, 0x9f / 255)
_FAST_A   = 0x82 / 255   # alpha for fastest bin
_SLOW_RGB = (0xb3 / 255, 0xd1 / 255, 0xf5 / 255)
_SLOW_A   = 0.30

_cmap_blue = _mcolors59.LinearSegmentedColormap.from_list(
    'SpeedBlue',
    [(*_SLOW_RGB, _SLOW_A), (*_FAST_RGB, _FAST_A)],
    N=ROM_N_SPEED_BINS,
)

# -- Plot ---------------------------------------------------------------------
_tier_pairs = [
    ('L1/R1', [0, 1], ['T1_left', 'T1_right']),
    ('L2/R2', [2, 3], ['T2_left', 'T2_right']),
    ('L3/R3', [4, 5], ['T3_left', 'T3_right']),
]

fig_b, axes_b = plt.subplots(
    _n_sj, 3,
    figsize=(3 * 28 / 25.4, _n_sj * 14 / 25.4),
    sharex=True, sharey='row',
    squeeze=False,
)

for ji, jnt in enumerate(_sc_joints):
    for ci, (pair_lbl, leg_indices, leg_names) in enumerate(_tier_pairs):
        ax = axes_b[ji, ci]

        for bi in range(ROM_N_SPEED_BINS):
            clr = _cmap_blue(bi / max(ROM_N_SPEED_BINS - 1, 1))
            ax.plot(_phase_x, _mean_profiles_deg[bi, leg_indices[0], ji, :],
                    color=clr, lw=0.8)
            ax.plot(_phase_x, _mean_profiles_deg[bi, leg_indices[1], ji, :],
                    color=clr, lw=0.8, linestyle='--')

        # Liftoff markers
        for leg in leg_names:
            lo = _mean_liftoff.get(leg)
            if lo is not None:
                ax.axvline(lo, color='#444444', lw=0.5, linestyle=':')

        if ji == 0:
            ax.set_title(pair_lbl, fontsize=6, fontweight='bold')
        if ci == 0:
            jlabel = JOINT_LABELS.get(jnt, jnt.replace('_', ' '))
            ax.set_ylabel(jlabel + ' (°)', labelpad=2)
        else:
            ax.tick_params(axis='y', labelleft=False)
        if ji == _n_sj - 1:
            ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
            ax.set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
            if ci == 1:
                ax.set_xlabel('Hilbert phase')

        sns.despine(ax=ax)

# Colorbar as speed legend
_sm = plt.cm.ScalarMappable(
    cmap=_cmap_blue,
    norm=_mcolors59.Normalize(vmin=_bin_edges[0], vmax=_bin_edges[-1])
)
_sm.set_array([])
_cbar = fig_b.colorbar(_sm, ax=axes_b[:, -1], shrink=0.6, pad=0.02,
                        label='speed (mm/s)', fraction=0.04)
_cbar.ax.tick_params(labelsize=6)

# Solid/dashed legend
from matplotlib.lines import Line2D as _L2D
axes_b[0, 0].legend(
    [_L2D([0], [0], color='#555555', lw=0.8),
     _L2D([0], [0], color='#555555', lw=0.8, linestyle='--')],
    ['left', 'right'],
    fontsize=5, frameon=False, loc='upper right',
)

plt.tight_layout(pad=0.3)
_pdf_b = OUTPUT_DIR / f'speedrom_B_phase_{JOINT_SET_NAME}.pdf'
plt.savefig(_pdf_b, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved: {_pdf_b}')


In [ ]:
# Cell 62 -- ROM vs Speed: scatter + linear regression (7 joints x 6 legs)
# Requires: rom_df from Cell 57
from scipy.stats import linregress

try:
    _ = rom_df
except NameError:
    raise RuntimeError('rom_df not found. Run Cell 57 first.')

_leg_clr = globals().get('FIG2_LEG_COLORS', {
    'T1_left': '#d62728', 'T1_right': '#ff7f0e',
    'T2_left': '#1f77b4', 'T2_right': '#9467bd',
    'T3_left': '#2ca02c', 'T3_right': '#8c564b',
})

_joints   = JOINT_SETS[ROM_JOINT_SET]
_leg_abbr = ['L1', 'R1', 'L2', 'R2', 'L3', 'R3']

fig61, axes61 = plt.subplots(
    len(_joints), len(LEGS),
    figsize=(2.2 * len(LEGS), 2.2 * len(_joints)),
    sharex=False, sharey='row',
    squeeze=False,
)

for ji, jnt in enumerate(_joints):
    for li, (leg, abbr) in enumerate(zip(LEGS, _leg_abbr)):
        ax  = axes61[ji, li]
        col = f'{leg}_{jnt}'
        mask = rom_df[[col, 'speed']].notna().all(axis=1)
        x = rom_df.loc[mask, col].values
        y = rom_df.loc[mask, 'speed'].values

        ax.scatter(x, y, color=_leg_clr[leg], s=5, alpha=0.35, linewidths=0)

        if len(x) >= 4:
            slope, intercept, r_val, p_val, _ = linregress(x, y)
            _xl = np.linspace(x.min(), x.max(), 200)
            ax.plot(_xl, slope * _xl + intercept, color='black', lw=1.2, zorder=5)
            stars = '**' if p_val < 0.01 else ('*' if p_val < 0.05 else '')
            ax.text(0.97, 0.97, f'r={r_val:.2f}{stars}',
                    transform=ax.transAxes, fontsize=6,
                    ha='right', va='top', family='monospace')

        # Row label (leftmost column only)
        if li == 0:
            ax.set_ylabel(jnt.replace('_', ' ') + '\nspeed (mm/s)', fontsize=6)
        else:
            ax.tick_params(axis='y', labelleft=False)

        # Column header (top row only)
        if ji == 0:
            ax.set_title(abbr, fontsize=8, fontweight='bold',
                         color=_leg_clr[leg])

        # X label (bottom row only)
        if ji == len(_joints) - 1:
            ax.set_xlabel('ROM (°)', fontsize=6)

        ax.tick_params(labelsize=5)
        sns.despine(ax=ax)

fig61.suptitle(
    f'ROM vs speed — one panel per leg  |  {ROM_JOINT_SET} joints  |  '
    f'black line = linear regression',
    fontsize=10, fontweight='bold'
)
plt.tight_layout()
_pdf_61 = OUTPUT_DIR / f'speedrom_scatter_per_leg_{ROM_JOINT_SET}.pdf'
plt.savefig(_pdf_61, bbox_inches='tight')
plt.show()
print(f'Saved: {_pdf_61}')


In [ ]:
# Cell 63 -- ROM vs Speed: paired L+R scatter + regression (7 joints × 3 leg pairs)
# Scatter uses L+R averaged ROM (same as MERGE_CONTRALATERAL=True in Cell 58).
# r̄ is within-fly Fisher-z averaged — matches Cell 58 exactly.
# Requires: rom_df and _compute_stat from Cell 58
from scipy.stats import linregress

try:
    _ = rom_df; _ = _compute_stat
except NameError:
    raise RuntimeError('Run Cell 58 first (needs rom_df and _compute_stat).')

import matplotlib.font_manager as _fmp63
import matplotlib as _mpl63
_fmp63.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_mpl63.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                         'axes.labelsize': 6, 'axes.titlesize': 6,
                         'xtick.labelsize': 6, 'ytick.labelsize': 6,
                         'pdf.fonttype': 42})

_PAIRS63 = [
    ('T1', 'T1_left', 'T1_right'),
    ('T2', 'T2_left', 'T2_right'),
    ('T3', 'T3_left', 'T3_right'),
]
_joints63 = JOINT_SETS[ROM_JOINT_SET]
_nj = len(_joints63)
_nt = len(_PAIRS63)

fig63, axes63 = plt.subplots(
    _nj, _nt,
    figsize=(_nt * 22 / 25.4, _nj * 16 / 25.4),
    sharex='col', sharey=False,
    squeeze=False,
)

_GREY = '#999999'

for ji, jnt in enumerate(_joints63):
    for ti, (tier, leg_L, leg_R) in enumerate(_PAIRS63):
        ax = axes63[ji, ti]

        # Averaged L+R ROM — already added to rom_df by Cell 58
        col_avg = f'{tier}_{jnt}'
        mask = rom_df[[col_avg, 'speed']].notna().all(axis=1)
        x = rom_df.loc[mask, 'speed'].values
        y = rom_df.loc[mask, col_avg].values   # ROM (°)
        fids = rom_df.loc[mask, 'fly_id'].values

        ax.scatter(x, y, color=_GREY, s=1, alpha=0.15,
                   linewidths=0, rasterized=True)

        # Pooled regression line
        if len(x) >= 4:
            sl, ic, _, _, _ = linregress(x, y)
            _xl = np.linspace(x.min(), x.max(), 200)
            ax.plot(_xl, sl * _xl + ic, color='k', lw=0.8, zorder=5)

        # Within-fly Fisher-z averaged r — identical to Cell 58
        eff, p, _ = _compute_stat(x, y, fids, 'within_fly')
        if not np.isnan(eff):
            stars = '**' if p < 0.01 else ('*' if p < 0.05 else '')
            ax.text(0.97, 0.97, f'r̅={eff:.2f}{stars}',
                    transform=ax.transAxes, fontsize=5,
                    ha='right', va='top', color='k')

        jlabel = JOINT_LABELS.get(jnt, jnt.replace('_', ' '))
        ax.set_ylabel(jlabel + '\nROM (°)', labelpad=2)

        if ji == 0:
            ax.set_title(tier, fontsize=6, fontweight='bold')

        if ji == _nj - 1:
            ax.set_xlabel('speed (mm/s)')

        ax.tick_params(labelsize=5)
        sns.despine(ax=ax)

plt.tight_layout(pad=0.3)
_pdf_63 = OUTPUT_DIR / f'speedrom_paired_{ROM_JOINT_SET}.pdf'
plt.savefig(_pdf_63, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved: {_pdf_63}')


In [ ]:
# Cell 64 -- Distribution of legs in stance (mean per step cycle) — dot cloud
import matplotlib.font_manager as _fm64
import matplotlib as _mpl64
import matplotlib.colors as _mc64

_fm64.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf")
_mpl64.rcParams.update({'font.family': 'Arial', 'font.size': 6,
                         'axes.labelsize': 6, 'axes.titlesize': 6,
                         'xtick.labelsize': 6, 'ytick.labelsize': 6,
                         'pdf.fonttype': 42})

_stance_cols = [f'{leg}_swing_stance' for leg in LEGS
                if f'{leg}_swing_stance' in df.columns]

_n_stance = df[_stance_cols].sum(axis=1)
_n_stance[df[_stance_cols].isna().any(axis=1)] = np.nan
df['n_stance'] = _n_stance

_sc = (df.dropna(subset=['step_cycle_id', 'n_stance'])
         .groupby(['bout_id', 'step_cycle_id'])
         .agg(mean_stance=('n_stance', 'mean'),
              speed=('step_cycle_mean_speed', 'first'))
         .reset_index())

print(f"Step cycles: {len(_sc)}")
print(f"Min mean legs in stance: {_sc['mean_stance'].min():.2f}")
print(f"Mean ± std: {_sc['mean_stance'].mean():.2f} ± {_sc['mean_stance'].std():.2f}")

# Color by speed
_spd = _sc['speed'].values
_norm = _mc64.Normalize(vmin=np.nanpercentile(_spd, 5), vmax=np.nanpercentile(_spd, 95))
_cmap = _mpl64.colormaps.get_cmap('Blues')
_colors = _cmap(_norm(_spd))

rng = np.random.default_rng(42)
_jitter = rng.uniform(-0.3, 0.3, len(_sc))

fig64, ax64 = plt.subplots(figsize=(40 / 25.4, 60 / 25.4))

ax64.scatter(_jitter, _sc['mean_stance'].values,
             c=_colors, s=1.5, alpha=0.4, linewidths=0, rasterized=True)

ax64.axhline(0, color='k', lw=0.8, linestyle='--')
ax64.axhline(_sc['mean_stance'].mean(), color='k', lw=0.8)

ax64.set_xlim(-1, 1)
ax64.set_xticks([])
ax64.set_ylabel('mean legs in stance per cycle')
ax64.set_ylim(-0.3, 6.3)
ax64.set_yticks(range(7))

# Colorbar for speed
_sm = _mpl64.cm.ScalarMappable(cmap=_cmap, norm=_norm)
_sm.set_array([])
_cb = fig64.colorbar(_sm, ax=ax64, shrink=0.5, pad=0.04, fraction=0.05)
_cb.set_label('speed (mm/s)')
_cb.ax.tick_params(labelsize=6)

sns.despine(ax=ax64, bottom=True)
plt.tight_layout(pad=0.3)
_pdf64 = OUTPUT_DIR / 'stance_distribution.pdf'
plt.savefig(_pdf64, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved: {_pdf64}')
